# AetherTTS — Phase 1 流式主干对比实验（Colab + 自建 RTX 4090 双环境兼容版）

本 Notebook 自动适配 **Google Colab（T4 / L4 / A100）** 与 **自建 Linux GPU 服务器（如 RTX 4090 / AutoDL，全部文件纯本地保存，零云盘依赖）**，端到端完成：

1. **智能环境识别与存储策略**：
   - **Google Colab**：自动挂载 Google Drive（`/content/drive/MyDrive/AetherTTS_Exp`），数据与高频断点自动同步到云盘，本地 SSD 负责高速训练。
   - **自建 RTX 4090 / 本地 Linux**：自动识别为 `local` 模式，**不挂载云盘、不做多余拷贝**，全部数据、权重、音频和报表直接保存在本地目录（默认 `./aethertts_exp`，可通过环境变量 `AETHER_ROOT` 指定），并自动启用多核并行特征提取与 TF32/BF16 加速。
2. **高频断点备份与真正的中途续训（防 T4/L4 断连丢进度）**：
   - 声码器与 5 个声学模型均支持 **每 `N` 步或每 `10` 分钟自动保存 `latest.pt`（含模型 + 优化器 + 调度器 + Scaler + step + history）**、`best.pt`（最优验证集权重）、阶段里程碑快照与训练中试听预览 WAV。
   - 只有完整跑完所有步数才会写入 `.done` 标记；若中途断连，重跑会自动从上次 `latest.pt` 的精确 step 无缝续训！
3. **聚焦「流式输入」的 6 槽控制变量实验矩阵**（~8.6–9.5M 含声码器）：
   - `E0_offline_bitransformer`（非流式全双向 Transformer 音质天花板）
   - `E1_conv_unigru_L0`（0 前瞻严格因果基线：量化“无右前瞻”的韵律下界）
   - `E1_conv_unigru_L8`（8 音素≈4 汉字浅前瞻 + ConvNeXt + Uni-GRU：端侧极简基线 + 前瞻拐点对照）
   - `E2_swa_L8`（1 层 8 音素浅前瞻 + 滑窗 SWA `W=48`：与 `E3` 同前瞻预算的浅前瞻对照）
   - `E3_chunk_attn_C4_L8`（**主力冲冠候选**：5 层深双向分块前瞻注意力，提交块 `C=4` + 前瞻 `L=8` + 有界 KV `W=48`，丢弃前瞻 KV 零前瞻膨胀，原生适配流式前端暂定纠错）
   - `E4_gated_deltanet_L8`（$O(1)$ 状态族独苗：8 音素前瞻 + 标准门控 DeltaNet，验证长段落无限流式收益）
4. **流式↔离线数学等价 + 零前瞻膨胀扰动单测 + 强遗忘梯度硬门禁 + 四维自动打分 + 盲听表**。

> **a4 流式输入架构升级（详见各 cell 内注释）**
> - **重构实验矩阵聚焦流式输入**：砍掉短句冗余的 SSM / 累加 WKV 槽位，新增 **5 层深双向分块前瞻 `ChunkLookaheadBackbone`** 与 **`E1` 的 `L=0` vs `L=8` 前瞻消融**；前瞻预算扩展至 8 音素（~4 汉字），与 Phase 2 流式前端消歧滑窗共享同一延迟预算。
> - **帧级解码器 2 帧集中式右前瞻**：`frame_decoder[0]` 采用 `CentralizedLookaheadBlock(k_left=4, lookahead=2)`（21 ms，环境变量 `AETHER_FRAME_LOOKAHEAD` 可配），其余 3 层严格因果，补齐元音→辅音协同发音过渡；训练期 `h_dec` 按 `mel_mask` 置零保证与推理逐帧等价。
> - **延续 a3 全部基础修复**：音素内 24 维多尺度相位特征 `(τ, d-1-τ, (τ+0.5)/d)`、声码器集中式 2 帧右前瞻与 `n_fft // 2` 精确 OLA 对齐、log 空间无溢出分块 DeltaNet、批向量化 Beta-Binomial/Viterbi 对齐。
> - 产物目录名带架构版本后缀 `_a4` 及 `_fL{frame_lookahead}` / `_L{vocoder_lookahead}`，消融实验互不冲突。

> **自建 4090 无界面命令行挂后台**：也可直接在项目根目录运行 `python scripts/run_phase1_local.py --profile standard_phase1 --root ./aethertts_exp`（配合 `tmux` 离线跑完）。


## 1. 环境自动识别（Colab / 自建 4090 本地 / Kaggle）与全局配置


In [ ]:
import os
import sys
import json
import math
import time
import signal
import shutil
import random
import tarfile
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import torch

# ==============================================================================
# 1) 环境自动检测（支持环境变量覆盖，方便自建 4090 CLI 脚本调用）
# ==============================================================================
ENV_MODE = os.environ.get("AETHER_ENV_MODE", "auto").lower()  # "auto" | "colab" | "local" | "kaggle"
if ENV_MODE == "auto":
    if "google.colab" in sys.modules:
        ENV_MODE = "colab"
    elif Path("/kaggle/working").exists():
        ENV_MODE = "kaggle"
    else:
        ENV_MODE = "local"

IS_COLAB = (ENV_MODE == "colab")
IS_LOCAL = (ENV_MODE == "local")
IS_KAGGLE = (ENV_MODE == "kaggle")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    STORAGE_ROOT = Path(os.environ.get("AETHER_ROOT", "/content/drive/MyDrive/AetherTTS_Exp"))
    LOCAL_WORK = Path("/content/aethertts_work")
elif IS_KAGGLE:
    STORAGE_ROOT = Path(os.environ.get("AETHER_ROOT", "/kaggle/working/AetherTTS_Exp"))
    LOCAL_WORK = Path("/kaggle/temp/aethertts_work")
else:
    # 自建 4090 / 本地 Linux：所有文件统一留在本地，不做任何云盘挂载或重复拷贝！
    STORAGE_ROOT = Path(os.environ.get("AETHER_ROOT", "./aethertts_exp")).expanduser().resolve()
    LOCAL_WORK = STORAGE_ROOT

# 兼容别名（方便后续统一引用）
DRIVE_ROOT = STORAGE_ROOT

for p in [
    STORAGE_ROOT / "datasets" / "raw",
    STORAGE_ROOT / "vocoder",
    STORAGE_ROOT / "runs",
    STORAGE_ROOT / "eval_samples",
    STORAGE_ROOT / "reports",
    LOCAL_WORK / "data",
    LOCAL_WORK / "cache",
    LOCAL_WORK / "vocoder",
    LOCAL_WORK / "runs",
    LOCAL_WORK / "eval_samples",
    LOCAL_WORK / "reports",
]:
    p.mkdir(parents=True, exist_ok=True)

# 针对 4090 / Ampere+ GPU 启用 TF32 加速
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

# ==============================================================================
# 2) 用户核心配置区（支持环境变量覆写；国内自建 4090 可提前 export HF_ENDPOINT=https://hf-mirror.com）
# ==============================================================================
# 实验档位："quick_smoke" | "standard_phase1" | "full_quality"
EXPERIMENT_PROFILE = os.environ.get("AETHER_PROFILE", "standard_phase1")

# 数据集选择："bznsyp" | "ljspeech" | "custom_bilingual"
DATASET_SOURCE = os.environ.get("AETHER_DATASET", "bznsyp")

# 声码器模式："both" | "pretrained_vocos" | "causal_vocos_lite"
VOCODER_MODE = os.environ.get("AETHER_VOCODER", "both")

# 要训练和评测的模型列表：(实验名, 主干类型, 文本右前瞻音素数 L)
ALL_EXPERIMENTS = [
    ("E0_offline_bitransformer", "offline_transformer", 0),  # 离线全双向 Transformer（理论音质天花板）
    ("E1_conv_unigru_L0",        "conv_gru",            0),  # 0 前瞻严格因果基线（测无右前瞻下界）
    ("E1_conv_unigru_L8",        "conv_gru",            8),  # 极简端侧基线 + 8 音素（~4 汉字）浅前瞻（对比 L=0 测前瞻净收益）
    ("E2_swa_L8",                "swa",                 8),  # 1 层 8 音素浅前瞻 + 滑窗 SWA (W=48)（与 E3 同预算浅前瞻对照）
    ("E3_chunk_attn_C4_L8",      "chunk_attn",          8),  # 主力冲冠：5 层深双向分块前瞻 (C=4, L=8, W=48，丢弃前瞻 KV 零膨胀)
    ("E4_gated_deltanet_L8",     "gdn",                 8),  # O(1) 状态族独苗：8 音素浅前瞻 + 标准门控 DeltaNet
]
_filter_models = os.environ.get("AETHER_MODELS", "").strip()
if _filter_models:
    _wanted = {m.strip() for m in _filter_models.split(",") if m.strip()}
    EXPERIMENTS = [e for e in ALL_EXPERIMENTS if e[0] in _wanted]
else:
    EXPERIMENTS = ALL_EXPERIMENTS

# 架构版本号：模型结构一旦变更就递增，并写入声码器 / 声学模型 / 评测音频 / 报告 / INT8 的全部产物目录名，
# 防止新代码静默续训或加载旧架构的 checkpoint（旧目录原样保留，可做修复前后对比）
ARCH_TAG = "a4"

# 断点续训与高频备份策略（支持环境变量覆写）
SKIP_IF_DONE = True                                                        # 仅当存在 .done 完结标记时才跳过；未完结则自动从 latest.pt 精确续训
BACKUP_EVERY_STEPS = int(os.environ.get("AETHER_BACKUP_STEPS", "1000"))    # 声学模型每 1000 步保存并同步 latest.pt（防 Colab T4/L4 断连）
VOCODER_BACKUP_EVERY_STEPS = int(os.environ.get("AETHER_VOC_BACKUP_STEPS", "1500"))  # 声码器每 1500 步保存并同步 latest.pt
BACKUP_EVERY_MINUTES = float(os.environ.get("AETHER_BACKUP_MINUTES", "8.0"))         # 时间兜底：每满 8 分钟强制触发一次断点备份
SAVE_MILESTONE_EVERY = 5000                                                # 每 5000 步保留一份不可覆盖的里程碑权重快照
SYNC_FEATURES_TAR_TO_DRIVE = IS_COLAB                                      # Colab 下把预处理好的特征分片打包备份到云盘，重连秒恢复；自建本地就地缓存
CACHE_RAW_TO_DRIVE = (os.environ.get("AETHER_CACHE_RAW_DRIVE", "0") == "1") if not IS_LOCAL else True  # Colab 下默认不上传/回读 3GB+ 原始包到 DriveFS，防 FUSE 卡死断连
AUTO_UNASSIGN_COLAB = False                                                # Colab Pro 后台跑完后是否自动释放运行时（默认 False）

# 根据显存自动适配 batch_size（T4 15.8GB -> 16 防 OOM；L4/4090 24GB -> 32 提速）
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
CPU_CORES = os.cpu_count() or 2
NUM_WORKERS = min(6, max(2, CPU_CORES // 2)) if IS_LOCAL else 2

PROFILE_PRESETS = {
    "quick_smoke": {
        "max_train_hours": 2.0,
        "val_utts": 40,
        "vocoder_stage1_steps": 1000,
        "vocoder_stage2_steps": 2000,
        "acoustic_steps": 3000,
        "batch_size": 16,
        "eval_max_utts": 12,
    },
    "standard_phase1": {
        "max_train_hours": 8.0,
        "val_utts": 80,
        "vocoder_stage1_steps": 4000,
        "vocoder_stage2_steps": 14000,
        "acoustic_steps": 15000,
        "batch_size": 32 if GPU_MEM_GB >= 22 else (24 if GPU_MEM_GB >= 18 else 16),
        "eval_max_utts": 25,
    },
    "full_quality": {
        "max_train_hours": 20.0,
        "val_utts": 100,
        "vocoder_stage1_steps": 8000,
        "vocoder_stage2_steps": 42000,
        "acoustic_steps": 40000,
        "batch_size": 32 if GPU_MEM_GB >= 22 else 16,
        "eval_max_utts": 50,
    },
}

CFG = {
    "seed": 42,
    "sample_rate": 24000,
    "n_fft": 1024,
    "hop_length": 256,
    "win_length": 1024,
    "n_mels": 100,
    "min_audio_sec": 0.8,
    "max_audio_sec": 12.0,
    "d_model": 256,
    "k_left": 3,
    "swa_window": 48,
    "chunk_attn_size": 4,       # E3 分块深前瞻注意力每次提交的音素数 C（≈ 2 汉字）
    "stream_text_chunk": 4,     # 流式评测/测速默认每次喂入 4 音素（≈ 2 汉字，与 C=4 对齐）
    "stream_mel_chunk": 16,
    "lr_acoustic": 4e-4,
    "lr_vocoder": 2e-4,
    "env_mode": ENV_MODE,
    "profile": EXPERIMENT_PROFILE,
    "dataset_source": DATASET_SOURCE,
    "vocoder_mode": VOCODER_MODE,
    "num_workers": NUM_WORKERS,
    # 帧级解码器首层集中式右前瞻帧数（默认 2 帧 ≈ 21 ms，补齐元音->辅音协同发音过渡；0 表示关闭）
    "frame_lookahead": int(os.environ.get("AETHER_FRAME_LOOKAHEAD", "2")),
    # 声码器集中式右前瞻帧数（仅 embed 层看右边，其余层严格因果）。iSTFT 每帧合成窗以当前帧为中心，
    # 右半窗 512 采样点正好是 2 帧；零前瞻等于让网络凭过去的 mel 预测未来半个窗
    "vocoder_lookahead": int(os.environ.get("AETHER_VOCODER_LOOKAHEAD", "2")),
    **PROFILE_PRESETS[EXPERIMENT_PROFILE],
}
_bs_override = os.environ.get("AETHER_BATCH_SIZE", "").strip()
if _bs_override:
    CFG["batch_size"] = int(_bs_override)

# 所有产物目录共用的后缀（数据集_档位_架构版本；声学与评测额外区分 frame/vocoder 前瞻消融）
CFG["run_suffix"] = f"{CFG['dataset_source']}_{CFG['profile']}_{ARCH_TAG}"
CFG["acoustic_suffix"] = f"{CFG['run_suffix']}_fL{CFG['frame_lookahead']}"
CFG["eval_suffix"] = f"{CFG['acoustic_suffix']}_vL{CFG['vocoder_lookahead']}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

print("=" * 72)
print(f"Environment   : {ENV_MODE.upper()} ({'纯本地模式，无云盘拷贝' if IS_LOCAL else '云盘同步备份模式'})")
print(f"Device        : {DEVICE}" + (f" | {torch.cuda.get_device_name(0)} ({GPU_MEM_GB:.1f} GB)" if torch.cuda.is_available() else ""))
print(f"CPU Cores     : {CPU_CORES} (DataLoader workers={NUM_WORKERS})")
print(f"Storage Root  : {STORAGE_ROOT}")
print(f"Local Work    : {LOCAL_WORK}")
if os.environ.get("HF_ENDPOINT"):
    print(f"HF Endpoint   : {os.environ['HF_ENDPOINT']}")
print(f"Profile       : {EXPERIMENT_PROFILE} | max_train_hours={CFG['max_train_hours']}h | batch={CFG['batch_size']} | steps={CFG['acoustic_steps']}")
print(f"Backup Cadence: every {BACKUP_EVERY_STEPS} steps or {BACKUP_EVERY_MINUTES:.1f} min (+ milestone every {SAVE_MILESTONE_EVERY} steps + SIGTERM/Ctrl+C emergency save)")
print(f"Arch / LA     : {ARCH_TAG} | chunk_C={CFG['chunk_attn_size']} | frame_LA={CFG['frame_lookahead']}f | vocoder_LA={CFG['vocoder_lookahead']}f")
print(f"Models        : {[x[0] for x in EXPERIMENTS]}")
print("=" * 72)


## 2. 智能依赖安装（锁定系统基础包防冲突，已装包自动跳过）


In [ ]:
import importlib.metadata
import importlib.util
import subprocess

AUTO_INSTALL_DEPS = os.environ.get("AETHER_SKIP_PIP", "0") != "1"

# 保护 Colab / 4090 环境中已预装的基础包版本，避免 pip 升级破坏 CUDA PyTorch / pandas / huggingface-hub
PROTECTED_BASE_PKGS = [
    "torch", "torchaudio", "torchvision", "numpy", "pandas",
    "scipy", "matplotlib", "huggingface-hub", "decorator",
]

CORE_PKGS = [
    ("pypinyin", "pypinyin"),
    ("g2p_en", "g2p-en"),
    ("soundfile", "soundfile"),
    ("librosa", "librosa"),
    ("scipy", "scipy"),
    ("einops", "einops"),
    ("tqdm", "tqdm"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("huggingface_hub", "huggingface-hub"),
    ("jiwer", "jiwer"),
]

OPTIONAL_EVAL_PKGS = [
    ("vocos", "vocos"),
    ("faster_whisper", "faster-whisper"),
]


def _pip_install(pkgs, constraints_file: Path, required: bool = True):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints_file), *pkgs]
    ret = subprocess.call(cmd)
    if ret != 0:
        # 若约束过严导致解析失败，回退为普通安装（保持 --upgrade-strategy only-if-needed）
        fallback_cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *pkgs]
        if required:
            subprocess.check_call(fallback_cmd)
        else:
            ret2 = subprocess.call(fallback_cmd)
            if ret2 != 0:
                print(f"[Pip Warn] 可选包安装失败（不影响主训练，评测时自动降级跳过）: {pkgs}")


if AUTO_INSTALL_DEPS:
    constraints_lines = []
    for pkg in PROTECTED_BASE_PKGS:
        try:
            ver = importlib.metadata.version(pkg)
            constraints_lines.append(f"{pkg}=={ver}")
        except importlib.metadata.PackageNotFoundError:
            pass
    constraints_file = LOCAL_WORK / "pip_constraints.txt"
    constraints_file.write_text("\n".join(constraints_lines) + "\n", encoding="utf-8")

    missing_core = [pip_name for mod_name, pip_name in CORE_PKGS if importlib.util.find_spec(mod_name) is None]
    if missing_core:
        print(f"[Pip] 正在安装缺失核心依赖: {missing_core}")
        _pip_install(missing_core, constraints_file, required=True)
    else:
        print("[Pip] 核心训练依赖均已就绪，跳过安装。")

    missing_opt = [pip_name for mod_name, pip_name in OPTIONAL_EVAL_PKGS if importlib.util.find_spec(mod_name) is None]
    if missing_opt:
        print(f"[Pip] 正在安装可选评测/对比依赖: {missing_opt}")
        _pip_install(missing_opt, constraints_file, required=False)

    # resemblyzer 依赖老版 webrtcvad（Python 3.12 源码编译慢/易报错），改用预编译 webrtcvad-wheels + --no-deps 秒装
    if importlib.util.find_spec("resemblyzer") is None:
        print("[Pip] 正在安装声纹评测依赖: webrtcvad-wheels + resemblyzer（免源码编译）...")
        r1 = subprocess.call([sys.executable, "-m", "pip", "install", "-q", "webrtcvad-wheels"])
        r2 = subprocess.call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "resemblyzer"])
        if r1 != 0 or r2 != 0:
            print("[Pip Warn] resemblyzer 安装未成功，Cell 11 将自动跳过 SIM 声纹指标，不影响训练与其余评测。")

    print("[Pip] 依赖检查与安装已完成！（提示：Colab 下若上方出现 cudf/moviepy/google-colab 等镜像自带的红色冲突提示，属于正常现象，可直接运行后续 Cell。）")
else:
    print("[Pip] 已设置 AETHER_SKIP_PIP=1，跳过自动安装。")


## 3. 基础设施：零冗余同步工具、24 kHz 100-Mel 特征提取与统一中英混读前端


In [ ]:
import gc
import re
import ctypes
import subprocess
import unicodedata
import numpy as np
import soundfile as sf
import librosa
from scipy import signal as scipy_signal
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from pypinyin import pinyin, Style

try:
    from g2p_en import G2p
    import nltk
    for res in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger", "cmudict"]:
        try:
            nltk.download(res, quiet=True)
        except Exception:
            pass
    _EN_G2P = G2p()
except Exception as e:
    print(f"[Warn] g2p_en unavailable ({e}), falling back to character-level EN G2P.")
    _EN_G2P = None


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG["seed"])


def _same_path(a: Path, b: Path) -> bool:
    try:
        return a.resolve() == b.resolve()
    except Exception:
        return str(a) == str(b)


def _is_subpath(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except Exception:
        return False


def trim_process_memory():
    gc.collect()
    if sys.platform.startswith("linux"):
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass


def decode_wav_tensor(w: torch.Tensor) -> torch.Tensor:
    return (w.float() / 32767.0) if w.dtype == torch.int16 else w.float()


def atomic_torch_save(obj, path: Path):
    """原子写盘，防止断电/断连写坏 checkpoint"""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def sync_file_to_storage(local_path: Path, storage_path: Path):
    """Colab 下原子同步到云盘；自建 4090 本地模式下若源与目标相同则零开销直接跳过"""
    if _same_path(local_path, storage_path):
        return
    storage_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = storage_path.with_suffix(storage_path.suffix + ".tmp")
    shutil.copy2(local_path, tmp_path)
    os.replace(tmp_path, storage_path)


def copy_from_storage_to_local(storage_path: Path, local_path: Path, chunk_mb: int = 8) -> Path:
    """将云盘大文件按 8MB 顺序分块复制到本地 SSD，避免在 DriveFS FUSE 上直接小块流式读取导致断连"""
    if _same_path(storage_path, local_path):
        return local_path
    local_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = local_path.with_suffix(local_path.suffix + ".part")
    total = storage_path.stat().st_size
    copied = 0
    last_log = 0
    buf_sz = chunk_mb * 1024 * 1024
    with open(storage_path, "rb") as fsrc, open(tmp_path, "wb") as fdst:
        while True:
            buf = fsrc.read(buf_sz)
            if not buf:
                break
            fdst.write(buf)
            copied += len(buf)
            if total >= 200 * 1024 * 1024 and (copied - last_log) >= 256 * 1024 * 1024:
                last_log = copied
                print(f"  [Stage] {storage_path.name}: {copied / 1e6:.0f} / {total / 1e6:.0f} MB", flush=True)
    os.replace(tmp_path, local_path)
    return local_path


def sync_dir_to_storage(local_dir: Path, storage_dir: Path):
    if _same_path(local_dir, storage_dir):
        return
    storage_dir.mkdir(parents=True, exist_ok=True)
    for src in local_dir.rglob("*"):
        if src.is_file():
            rel = src.relative_to(local_dir)
            sync_file_to_storage(src, storage_dir / rel)


def atomic_write_text(text: str, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)


def save_progress_status(stage: str, detail: dict):
    """在 STORAGE_ROOT/progress.json 实时记录运行进度，云盘/本地随时可一眼查看"""
    prog_local = LOCAL_WORK / "progress.json"
    prog_storage = STORAGE_ROOT / "progress.json"
    try:
        cur = json.loads(prog_storage.read_text(encoding="utf-8")) if prog_storage.exists() else {}
    except Exception:
        cur = {}
    cur["updated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
    cur["env_mode"] = ENV_MODE
    cur["profile"] = CFG["profile"]
    cur["dataset"] = CFG["dataset_source"]
    cur.setdefault("stages", {})[stage] = detail
    txt = json.dumps(cur, ensure_ascii=False, indent=2)
    atomic_write_text(txt, prog_local)
    sync_file_to_storage(prog_local, prog_storage)


_STOP_REQUESTED = False


def _handle_sigterm(signum, frame):
    global _STOP_REQUESTED
    _STOP_REQUESTED = True
    print("\n[Signal] 收到 SIGTERM 停止信号，当前 step 完成后将立即执行紧急断点备份并安全退出！", flush=True)


try:
    signal.signal(signal.SIGTERM, _handle_sigterm)
except Exception:
    pass



# ------------------------------------------------------------------------------
# 24 kHz 100-bin Log-Mel 提取器（与 charactr/vocos-mel-24khz 100% 参数对齐）
# ------------------------------------------------------------------------------
class VocosCompatibleMel(nn.Module):
    def __init__(self, sample_rate=24000, n_fft=1024, hop_length=256, n_mels=100):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            center=True,
            power=1,
        )

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        squeeze = False
        if wav.dim() == 1:
            wav = wav.unsqueeze(0)
            squeeze = True
        mel = self.mel_spec(wav)
        log_mel = torch.log(torch.clamp(mel, min=1e-7))
        return log_mel.squeeze(0) if squeeze else log_mel


MEL_EXTRACTOR = VocosCompatibleMel(
    sample_rate=CFG["sample_rate"],
    n_fft=CFG["n_fft"],
    hop_length=CFG["hop_length"],
    n_mels=CFG["n_mels"],
)


def load_and_normalize_wav(path: Path, target_sr: int = 24000) -> np.ndarray:
    wav, sr = sf.read(str(path), dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=-1)
    if sr != target_sr:
        if sr > target_sr and sr % target_sr == 0:
            wav = scipy_signal.resample_poly(wav, 1, sr // target_sr).astype(np.float32, copy=False)
        elif target_sr > sr and target_sr % sr == 0:
            wav = scipy_signal.resample_poly(wav, target_sr // sr, 1).astype(np.float32, copy=False)
        else:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    wav_trimmed, idx = librosa.effects.trim(wav, top_db=35, frame_length=1024, hop_length=256)
    pad = int(0.06 * target_sr)
    start = max(0, idx[0] - pad)
    end = min(len(wav), idx[1] + pad)
    wav = wav[start:end]
    rms = np.sqrt(np.mean(wav ** 2) + 1e-8)
    wav = wav * (0.10 / max(rms, 1e-4))
    peak = np.max(np.abs(wav)) + 1e-8
    if peak > 0.92:
        wav = wav * (0.92 / peak)
    return wav.astype(np.float32)


def extract_f0_and_energy(wav: np.ndarray, sr: int = 24000, hop_length: int = 256, n_frames: int = None):
    f0 = librosa.yin(
        wav, fmin=65.0, fmax=600.0, sr=sr, frame_length=1024, hop_length=hop_length, center=True
    ).astype(np.float32)
    rms = librosa.feature.rms(y=wav, frame_length=1024, hop_length=hop_length, center=True)[0].astype(np.float32)
    if n_frames is not None:
        if len(f0) < n_frames:
            f0 = np.pad(f0, (0, n_frames - len(f0)), mode="edge")
            rms = np.pad(rms, (0, n_frames - len(rms)), mode="edge")
        else:
            f0 = f0[:n_frames]
            rms = rms[:n_frames]
    uv = (rms > 0.008) & (f0 > 70.0) & (f0 < 580.0)
    log_f0 = np.zeros_like(f0, dtype=np.float32)
    if uv.any():
        voiced_idx = np.where(uv)[0]
        voiced_vals = np.log(f0[voiced_idx])
        log_f0 = np.interp(np.arange(len(f0)), voiced_idx, voiced_vals).astype(np.float32)
    log_energy = np.log(np.clip(rms, 1e-5, None)).astype(np.float32)
    return log_f0, log_energy, uv.astype(np.float32)


# ------------------------------------------------------------------------------
# 统一中英混读音素前端 (ZH 声母/带调韵母 + EN ARPAbet + 韵律/标点 + LangID)
# ------------------------------------------------------------------------------
ZH_INITIALS = [
    "b", "p", "m", "f", "d", "t", "n", "l", "g", "k", "h",
    "j", "q", "x", "zh", "ch", "sh", "r", "z", "c", "s", "y", "w",
]
ZH_FINALS_BASE = [
    "a", "o", "e", "i", "u", "v", "ai", "ei", "ui", "ao", "ou", "iu", "ie", "ve", "er",
    "an", "en", "in", "un", "vn", "ang", "eng", "ing", "ong",
    "ia", "iao", "ian", "iang", "iong", "ua", "uo", "uai", "uan", "uang", "ue", "van",
]
EN_ARPABET = [
    "AA0", "AA1", "AA2", "AE0", "AE1", "AE2", "AH0", "AH1", "AH2", "AO0", "AO1", "AO2",
    "AW0", "AW1", "AW2", "AY0", "AY1", "AY2", "B", "CH", "D", "DH",
    "EH0", "EH1", "EH2", "ER0", "ER1", "ER2", "EY0", "EY1", "EY2", "F", "G", "HH",
    "IH0", "IH1", "IH2", "IY0", "IY1", "IY2", "JH", "K", "L", "M", "N", "NG",
    "OW0", "OW1", "OW2", "OY0", "OY1", "OY2", "P", "R", "S", "SH", "T", "TH",
    "UH0", "UH1", "UH2", "UW0", "UW1", "UW2", "V", "W", "Y", "Z", "ZH",
]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>", "<sp>", ",", ".", "?", "!", ";", ":", "#1", "#2", "#3"]


class BilingualPhonemeFrontend:
    def __init__(self):
        vocab = list(SPECIAL_TOKENS)
        vocab += [f"zh_{ini}" for ini in ZH_INITIALS]
        for fin in ZH_FINALS_BASE:
            for tone in ["1", "2", "3", "4", "5"]:
                vocab.append(f"zh_{fin}{tone}")
        vocab += [f"en_{ph}" for ph in EN_ARPABET]
        self.vocab = vocab
        self.tok2id = {t: i for i, t in enumerate(vocab)}
        self.id2tok = {i: t for i, t in enumerate(vocab)}
        self.pad_id = self.tok2id["<pad>"]
        self.bos_id = self.tok2id["<bos>"]
        self.eos_id = self.tok2id["<eos>"]
        self.unk_id = self.tok2id["<unk>"]
        self.sp_id = self.tok2id["<sp>"]

    @property
    def vocab_size(self) -> int:
        return len(self.vocab)

    @staticmethod
    def _num_to_zh(num_str: str) -> str:
        digits = "零一二三四五六七八九"
        if "." in num_str:
            int_p, dec_p = num_str.split(".", 1)
            return BilingualPhonemeFrontend._num_to_zh(int_p) + "点" + "".join(digits[int(c)] for c in dec_p if c.isdigit())
        n = int(num_str)
        if len(num_str) == 4 and 1900 <= n <= 2099:
            return "".join(digits[int(c)] for c in num_str)
        if n == 0:
            return "零"
        if n < 10:
            return digits[n]
        if n < 20:
            return "十" + ("" if n == 10 else digits[n % 10])
        if n < 100:
            return digits[n // 10] + "十" + ("" if n % 10 == 0 else digits[n % 10])
        return "".join(digits[int(c)] for c in num_str)

    def normalize_text(self, text: str) -> str:
        text = unicodedata.normalize("NFKC", text).strip()
        punc_map = {"，": ",", "。": ".", "？": "?", "！": "!", "；": ";", "：": ":", "、": ",", "—": ",", "…": "."}
        for k, v in punc_map.items():
            text = text.replace(k, v)
        has_zh = bool(re.search(r"[\u4e00-\u9fff]", text))
        if has_zh:
            text = re.sub(r"(\d+(?:\.\d+)?)%", lambda m: "百分之" + self._num_to_zh(m.group(1)), text)
            text = re.sub(r"\d+(?:\.\d+)?", lambda m: self._num_to_zh(m.group(0)), text)
        return text

    @staticmethod
    def _split_zh_syllable(syl: str):
        syl = syl.lower().replace("ü", "v").replace("u:", "v")
        tone = syl[-1] if syl and syl[-1] in "12345" else "5"
        body = syl[:-1] if syl and syl[-1] in "12345" else syl
        if not body:
            return []
        ini = ""
        for cand in sorted(ZH_INITIALS, key=len, reverse=True):
            if body.startswith(cand) and len(body) > len(cand):
                ini = cand
                body = body[len(cand):]
                break
        if len(body) > 1 and body.endswith("r") and body != "er":
            body = body[:-1]
        if body == "u" and ini in ("j", "q", "x", "y"):
            body = "v"
        elif body == "ue" and ini in ("j", "q", "x", "y"):
            body = "ve"
        elif body == "uan" and ini in ("j", "q", "x", "y"):
            body = "van"
        elif body == "un" and ini in ("j", "q", "x", "y"):
            body = "vn"
        toks = []
        if ini:
            toks.append(f"zh_{ini}")
        toks.append(f"zh_{body}{tone}")
        return toks

    def _zh_span_to_phonemes(self, span: str):
        pys = [p[0] for p in pinyin(span, style=Style.TONE3, neutral_tone_with_five=True, errors="ignore") if p]
        for i in range(len(pys) - 1):
            if pys[i].endswith("3") and pys[i + 1].endswith("3"):
                pys[i] = pys[i][:-1] + "2"
        out = []
        for s in pys:
            out.extend(self._split_zh_syllable(s))
        return out

    def _en_span_to_phonemes(self, span: str):
        if _EN_G2P is not None:
            raw = _EN_G2P(span)
            out = []
            for p in raw:
                if p == " ":
                    out.append("<sp>")
                elif f"en_{p}" in self.tok2id:
                    out.append(f"en_{p}")
            return out
        return ["<sp>"]

    def text_to_ids(self, text: str, prosody_pinyin: list = None):
        tokens = ["<bos>"]
        langs = [0]
        if prosody_pinyin is not None:
            for item in prosody_pinyin:
                if item in self.tok2id:
                    tokens.append(item); langs.append(0)
                else:
                    for ph in self._split_zh_syllable(item):
                        tokens.append(ph if ph in self.tok2id else "<unk>")
                        langs.append(1)
        else:
            norm = self.normalize_text(text)
            pattern = re.compile(r"([\u4e00-\u9fff]+|[A-Za-z][A-Za-z'\-]*|#1|#2|#3|[,\.\?!;:])")
            prev_lang = 0
            for m in pattern.finditer(norm):
                seg = m.group(0)
                if re.match(r"^[\u4e00-\u9fff]+$", seg):
                    if prev_lang == 2:
                        tokens.append("<sp>"); langs.append(0)
                    for ph in self._zh_span_to_phonemes(seg):
                        tokens.append(ph if ph in self.tok2id else "<unk>")
                        langs.append(1)
                    prev_lang = 1
                elif re.match(r"^[A-Za-z]", seg):
                    if prev_lang in (1, 2):
                        tokens.append("<sp>"); langs.append(0)
                    for ph in self._en_span_to_phonemes(seg):
                        tokens.append(ph if ph in self.tok2id else "<unk>")
                        langs.append(2 if ph.startswith("en_") else 0)
                    prev_lang = 2
                else:
                    if seg in self.tok2id:
                        tokens.append(seg); langs.append(0)
                    prev_lang = 0
        tokens.append("<eos>")
        langs.append(0)
        return [self.tok2id.get(t, self.unk_id) for t in tokens], langs


FRONTEND = BilingualPhonemeFrontend()
print(f"Unified Bilingual Vocab Size: {FRONTEND.vocab_size}")
for demo in ["你好，请问今天北京的天气怎么样？", "Hello world, how is your API latency today?", "帮我把 Wi-Fi 切换到 5G，谢谢！"]:
    pids, lids = FRONTEND.text_to_ids(demo)
    print(f"  {demo} -> {len(pids)} phonemes")


## 4. 数据下载与 24 kHz 预处理（Colab 自动备份打包到云盘；自建 4090 纯本地直存+多核加速）


In [ ]:
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download


def safe_extract_tar(archive_path: Path, dest_dir: Path, desc: str = "Extract"):
    """安全高速解压 tar / tar.bz2 / tar.gz：优先走本地 GNU tar（带断点进度、低内存），回退流式 tarfile(..., filter='data') 彻底消除 Python 3.14 DeprecationWarning"""
    dest_dir.mkdir(parents=True, exist_ok=True)
    name_l = archive_path.name.lower()
    is_bz2 = name_l.endswith((".tar.bz2", ".tbz2", ".bz2"))
    is_gz = name_l.endswith((".tar.gz", ".tgz", ".gz"))

    use_gnu_tar = False
    if sys.platform != "win32" and shutil.which("tar"):
        try:
            ver_out = subprocess.check_output(["tar", "--version"], text=True, stderr=subprocess.STDOUT)
            use_gnu_tar = ("GNU tar" in ver_out)
        except Exception:
            use_gnu_tar = False

    if use_gnu_tar:
        cmd = ["tar", "--no-same-owner"]
        if is_bz2:
            bzip_prog = shutil.which("lbzip2") or shutil.which("pbzip2")
            if bzip_prog:
                cmd.append(f"--use-compress-program={bzip_prog}")
            else:
                cmd.append("-j")
        elif is_gz:
            cmd.append("-z")
        cmd.extend([
            "--checkpoint=2500",
            "--checkpoint-action=echo=tar_checkpoint:%u",
            "-xf", str(archive_path),
            "-C", str(dest_dir),
        ])
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        last_err_lines = []
        for raw_line in proc.stdout:
            line = raw_line.strip()
            if "tar_checkpoint:" in line:
                cnt = line.rsplit(":", 1)[-1]
                print(f"  [{desc}] 已解压 {cnt} 个归档块...", flush=True)
            elif line:
                last_err_lines.append(line)
                if len(last_err_lines) > 10:
                    last_err_lines.pop(0)
        rc = proc.wait()
        if rc != 0:
            raise RuntimeError(f"tar 解压失败 (exit={rc}): {' | '.join(last_err_lines)}")
        return

    # 便携回退：使用流式模式 ("r|...") + filter="data"，O(1) 内存且无 Python 3.14 DeprecationWarning
    rmode = "r|bz2" if is_bz2 else ("r|gz" if is_gz else ("r|" if name_l.endswith(".tar") else "r|*"))
    extract_kwargs = {"filter": "data"} if hasattr(tarfile, "data_filter") else {}
    with tarfile.open(archive_path, rmode) as tf:
        tf.extractall(dest_dir, **extract_kwargs)


def _extract_possibly_drive_archive(storage_archive: Path, local_tmp_dir: Path, dest_dir: Path, desc: str):
    """若归档位于云盘挂载路径（Colab DriveFS），先按大块复制到本地 SSD 再解压，严禁直接在 DriveFS 上解压 GB 级包"""
    if IS_LOCAL or not _is_subpath(storage_archive, STORAGE_ROOT):
        safe_extract_tar(storage_archive, dest_dir, desc=desc)
        return
    staged = local_tmp_dir / storage_archive.name
    try:
        print(f"[Stage] 将云盘归档分块复制到本地 SSD 再解压: {storage_archive.name}", flush=True)
        copy_from_storage_to_local(storage_archive, staged)
        safe_extract_tar(staged, dest_dir, desc=desc)
    finally:
        staged.unlink(missing_ok=True)


def _extract_single_item(args):
    it, pids, lids, feat_file_str, cfg_sub = args
    feat_file = Path(feat_file_str)
    if feat_file.exists():
        try:
            d = torch.load(feat_file, map_location="cpu", weights_only=False)
            wav_t = decode_wav_tensor(d["wav"])
            dur_sec = float(wav_t.shape[0]) / cfg_sub["sample_rate"]
            return (
                it, feat_file_str, dur_sec,
                d["mel"].float().numpy(),
                d["log_f0"].float().numpy(),
                d["log_energy"].float().numpy(),
                d["uv"].float().numpy(),
                len(d["phoneme_ids"]), int(d["mel"].shape[-1]),
            )
        except Exception:
            feat_file.unlink(missing_ok=True)
    try:
        wav = load_and_normalize_wav(Path(it["wav_path"]), cfg_sub["sample_rate"])
    except Exception:
        return None
    dur_sec = len(wav) / cfg_sub["sample_rate"]
    if dur_sec < cfg_sub["min_audio_sec"] or dur_sec > cfg_sub["max_audio_sec"]:
        return None
    if len(pids) < 4 or len(pids) > 220:
        return None
    wav_t = torch.from_numpy(wav)
    with torch.no_grad():
        mel_t = MEL_EXTRACTOR(wav_t)
    n_frames = mel_t.shape[-1]
    if n_frames < len(pids) * 2:
        return None
    f0_np, e_np, uv_np = extract_f0_and_energy(wav, cfg_sub["sample_rate"], cfg_sub["hop_length"], n_frames)
    wav_i16 = torch.clamp(torch.round(wav_t * 32767.0), -32768, 32767).to(torch.int16)
    d = {
        "utt_id": it["utt_id"],
        "text": it["text"],
        "phoneme_ids": torch.tensor(pids, dtype=torch.long),
        "lang_ids": torch.tensor(lids, dtype=torch.long),
        "mel": mel_t.to(torch.float16),
        "log_f0": torch.from_numpy(f0_np),
        "log_energy": torch.from_numpy(e_np),
        "uv": torch.from_numpy(uv_np),
        "wav": wav_i16,
    }
    atomic_torch_save(d, feat_file)
    return (it, feat_file_str, dur_sec, mel_t.to(torch.float32).numpy(), f0_np, e_np, uv_np, len(pids), int(n_frames))


def download_and_prepare_dataset(cfg):
    src = cfg["dataset_source"]
    tag = f"{src}_v2_max{cfg['max_train_hours']}h"
    storage_raw_dir = STORAGE_ROOT / "datasets" / "raw"
    local_raw_dir = LOCAL_WORK / "data" / src
    local_feat_dir = LOCAL_WORK / "cache" / tag
    local_manifest = local_feat_dir / "manifest.json"
    storage_feat_tar = STORAGE_ROOT / "datasets" / f"{tag}_features.tar"
    storage_shards_dir = STORAGE_ROOT / "datasets" / f"{tag}_shards"
    storage_shards_meta = storage_shards_dir / "shards_meta.json"

    local_raw_dir.mkdir(parents=True, exist_ok=True)
    local_feat_dir.mkdir(parents=True, exist_ok=True)

    # 快速通道 1：本地已预处理完，秒加载！
    if local_manifest.exists():
        meta = json.loads(local_manifest.read_text(encoding="utf-8"))
        if meta.get("train") and Path(meta["train"][0]["feat_path"]).exists():
            print(f"[Cache Hit] 直接加载本地预处理特征集: {local_manifest} ({meta['total_hours']}h, train={meta['train_count']}, val={meta['val_count']})")
            save_progress_status("dataset", {"status": "cached_local", "hours": meta["total_hours"], "train": meta["train_count"], "val": meta["val_count"]})
            return meta

    # 快速通道 2（Colab 断连重连）：优先从云盘分片缓存 (~250MB/片) 或兼容旧版单文件缓存快速恢复
    if SYNC_FEATURES_TAR_TO_DRIVE and (storage_shards_meta.exists() or storage_feat_tar.exists()):
        restored_ok = False
        if storage_shards_meta.exists():
            print(f"[Drive Feature Cache Hit] 发现云盘分片特征缓存，正在快速解压恢复: {storage_shards_dir}", flush=True)
            try:
                smeta = json.loads(storage_shards_meta.read_text(encoding="utf-8"))
                for s_info in tqdm(smeta.get("shards", []), desc="Restore Drive Shards", mininterval=1.0):
                    shard_src = storage_shards_dir / s_info["name"]
                    assert shard_src.exists(), f"Missing shard: {shard_src}"
                    _extract_possibly_drive_archive(shard_src, LOCAL_WORK / "tmp_dl", local_feat_dir.parent, desc=s_info["name"])
                restored_ok = True
            except Exception as e:
                print(f"[Warn] 云盘分片缓存恢复失败 ({e})，将改用原始数据集重新预处理。", flush=True)
        elif storage_feat_tar.exists():
            print(f"[Drive Feature Cache Hit] 发现云盘特征缓存，正在快速解压恢复: {storage_feat_tar}", flush=True)
            try:
                _extract_possibly_drive_archive(storage_feat_tar, LOCAL_WORK / "tmp_dl", local_feat_dir.parent, desc=storage_feat_tar.name)
                restored_ok = True
            except Exception as e:
                print(f"[Warn] 云盘单文件缓存恢复失败 ({e})，将改用原始数据集重新预处理。", flush=True)

        if restored_ok and local_manifest.exists():
            meta = json.loads(local_manifest.read_text(encoding="utf-8"))
            for split in ("train", "val"):
                for e in meta[split]:
                    e["feat_path"] = str(local_feat_dir / Path(e["feat_path"]).name)
            if meta.get("train") and Path(meta["train"][0]["feat_path"]).exists():
                atomic_write_text(json.dumps(meta, ensure_ascii=False, indent=2), local_manifest)
                print(f"[Restored] 秒恢复 {meta['total_hours']}h 预处理特征 (train={meta['train_count']}, val={meta['val_count']})")
                save_progress_status("dataset", {"status": "restored_from_drive", "hours": meta["total_hours"], "train": meta["train_count"], "val": meta["val_count"]})
                return meta

    items = []
    if src == "bznsyp":
        archive_path = storage_raw_dir / "BZNSYP.tar.bz2"
        marker = local_raw_dir / ".extracted"
        if marker.exists():
            has_txt = bool(list(local_raw_dir.rglob("*000001-010000.txt")))
            has_wav = any(p.is_dir() for p in local_raw_dir.rglob("Wave"))
            if not (has_txt and has_wav):
                marker.unlink(missing_ok=True)

        if not marker.exists():
            dl_dir = storage_raw_dir if IS_LOCAL else (LOCAL_WORK / "tmp_dl")
            dl_dir.mkdir(parents=True, exist_ok=True)
            extracted_ok = False

            if archive_path.exists() and archive_path.stat().st_size > 100_000_000:
                print(f"[Cache Hit] 复用原始压缩包: {archive_path}", flush=True)
                try:
                    _extract_possibly_drive_archive(archive_path, dl_dir, local_raw_dir, desc="BZNSYP")
                    extracted_ok = True
                except Exception as e:
                    print(f"[Warn] 已有压缩包解压失败 ({e})，将直接从 HuggingFace 下载到本地高速盘...", flush=True)
                    shutil.rmtree(local_raw_dir, ignore_errors=True)
                    local_raw_dir.mkdir(parents=True, exist_ok=True)

            if not extracted_ok:
                print(f"[Download] 从 HuggingFace 下载标贝 BZNSYP (12h) 到本地目录 {dl_dir} ...", flush=True)
                dl_path = Path(hf_hub_download(
                    repo_id="openspeech/BZNSYP",
                    filename="BZNSYP.tar.bz2",
                    repo_type="model",
                    local_dir=str(dl_dir),
                ))
                print(f"[Extract] 解压 BZNSYP 到 {local_raw_dir} ...", flush=True)
                safe_extract_tar(dl_path, local_raw_dir, desc="BZNSYP")
                if CACHE_RAW_TO_DRIVE and not _same_path(dl_path, archive_path):
                    print(f"[Cache] 同步原始压缩包到 {archive_path} ...", flush=True)
                    sync_file_to_storage(dl_path, archive_path)
                if not IS_LOCAL:
                    shutil.rmtree(dl_dir, ignore_errors=True)
            marker.touch()

        txt_candidates = list(local_raw_dir.rglob("*000001-010000.txt"))
        assert txt_candidates, f"未找到 000001-010000.txt，请检查目录 {local_raw_dir}"
        label_file = txt_candidates[0]
        wav_dirs = [p for p in local_raw_dir.rglob("Wave") if p.is_dir()]
        assert wav_dirs, f"未找到 Wave 目录，请检查目录 {local_raw_dir}"
        wav_dir = wav_dirs[0]

        raw_bytes = label_file.read_bytes()
        try:
            lines = raw_bytes.decode("utf-8").splitlines()
        except UnicodeDecodeError:
            lines = raw_bytes.decode("gb18030", errors="ignore").splitlines()

        def _merge_bznsyp_punc(clean_txt: str, pys: list):
            norm_t = FRONTEND.normalize_text(clean_txt)
            hanzi_cnt = len(re.findall(r"[\u4e00-\u9fff]", norm_t))
            if hanzi_cnt == len(pys):
                merged, pi = [], 0
                for ch in norm_t:
                    if re.match(r"[\u4e00-\u9fff]", ch):
                        merged.append(pys[pi]); pi += 1
                    elif ch in (",", ".", "?", "!", ";", ":"):
                        merged.append(ch)
                return merged
            # 含儿化音字数不等时保留原始拼音并补齐句末标点
            out_py = list(pys)
            if norm_t and norm_t[-1] in (".", "?", "!"):
                out_py.append(norm_t[-1])
            return out_py

        for i in range(0, len(lines) - 1, 2):
            line_txt, line_py = lines[i].strip(), lines[i + 1].strip()
            if not line_txt or "\t" not in line_txt:
                continue
            utt_id, text_with_prosody = line_txt.split("\t", 1)
            wav_path = wav_dir / f"{utt_id.strip()}.wav"
            if wav_path.exists():
                clean_text = re.sub(r"#[1-4]", "", text_with_prosody).strip()
                items.append({
                    "utt_id": utt_id.strip(),
                    "wav_path": str(wav_path),
                    "text": clean_text,
                    "prosody_pinyin": _merge_bznsyp_punc(clean_text, line_py.split()),
                })

    elif src == "ljspeech":
        archive_path = storage_raw_dir / "LJSpeech-1.1.tar.bz2"
        marker = local_raw_dir / ".extracted"
        if marker.exists() and not (local_raw_dir / "LJSpeech-1.1" / "metadata.csv").exists():
            marker.unlink(missing_ok=True)

        if not marker.exists():
            dl_dir = storage_raw_dir if IS_LOCAL else (LOCAL_WORK / "tmp_dl")
            dl_dir.mkdir(parents=True, exist_ok=True)
            extracted_ok = False

            if archive_path.exists() and archive_path.stat().st_size > 100_000_000:
                print(f"[Cache Hit] 复用原始压缩包: {archive_path}", flush=True)
                try:
                    _extract_possibly_drive_archive(archive_path, dl_dir, local_raw_dir, desc="LJSpeech")
                    extracted_ok = True
                except Exception as e:
                    print(f"[Warn] 已有压缩包解压失败 ({e})，将重新下载到本地高速盘...", flush=True)
                    shutil.rmtree(local_raw_dir, ignore_errors=True)
                    local_raw_dir.mkdir(parents=True, exist_ok=True)

            if not extracted_ok:
                print(f"[Download] 下载 LJSpeech-1.1 (Public Domain) 到 {dl_dir} ...", flush=True)
                dl_path = dl_dir / "LJSpeech-1.1.tar.bz2"
                tmp_part = dl_path.with_suffix(".bz2.part")
                url = "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"
                if shutil.which("wget"):
                    ret = subprocess.call(["wget", "-q", "--show-progress", "-O", str(tmp_part), url])
                    assert ret == 0 and tmp_part.exists(), "LJSpeech 下载失败，请检查网络连接"
                else:
                    import urllib.request
                    urllib.request.urlretrieve(url, str(tmp_part))
                os.replace(tmp_part, dl_path)
                print(f"[Extract] 解压 LJSpeech-1.1 到 {local_raw_dir} ...", flush=True)
                safe_extract_tar(dl_path, local_raw_dir, desc="LJSpeech")
                if CACHE_RAW_TO_DRIVE and not _same_path(dl_path, archive_path):
                    sync_file_to_storage(dl_path, archive_path)
                if not IS_LOCAL:
                    shutil.rmtree(dl_dir, ignore_errors=True)
            marker.touch()

        lj_root = local_raw_dir / "LJSpeech-1.1"
        for line in (lj_root / "metadata.csv").read_text(encoding="utf-8").splitlines():
            parts = line.strip().split("|")
            if len(parts) < 2:
                continue
            utt_id = parts[0]
            text = parts[2] if len(parts) >= 3 else parts[1]
            wav_path = lj_root / "wavs" / f"{utt_id}.wav"
            if wav_path.exists():
                items.append({"utt_id": utt_id, "wav_path": str(wav_path), "text": text, "prosody_pinyin": None})

    elif src == "custom_bilingual":
        custom_dir = STORAGE_ROOT / "datasets" / "custom_bilingual"
        meta_path = custom_dir / "metadata.csv"
        assert meta_path.exists(), f"请先将单音色双语数据放到 {custom_dir} (包含 metadata.csv 与 wavs/)"
        for line in meta_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split("|")
            if len(parts) < 2:
                continue
            utt_id, text = parts[0].strip(), parts[1].strip()
            wav_path = custom_dir / "wavs" / f"{utt_id}.wav"
            if wav_path.exists():
                items.append({"utt_id": utt_id, "wav_path": str(wav_path), "text": text, "prosody_pinyin": None})
    else:
        raise ValueError(f"Unsupported DATASET_SOURCE: {src}")

    rng = random.Random(cfg["seed"])
    rng.shuffle(items)
    # 估算需要抽多少条以覆盖 max_train_hours + 0.5h，减少多余特征计算
    max_total_sec = (cfg["max_train_hours"] + 0.5) * 3600.0
    est_needed = min(len(items), int(max_total_sec / 2.6) + 400)
    cand_items = items[:est_needed]

    cfg_sub = {k: cfg[k] for k in ("sample_rate", "hop_length", "min_audio_sec", "max_audio_sec")}
    # 先在主线程完成极速文本 G2P，确保 pypinyin / g2p_en 100% 线程安全且无多进程 pickle/fork 风险
    tasks = []
    for it in cand_items:
        feat_file_str = str(local_feat_dir / f"{it['utt_id']}.pt")
        if Path(feat_file_str).exists():
            tasks.append((it, [], [], feat_file_str, cfg_sub))
        else:
            pids, lids = FRONTEND.text_to_ids(it["text"], prosody_pinyin=it["prosody_pinyin"])
            tasks.append((it, pids, lids, feat_file_str, cfg_sub))

    print(f"[Preprocess] 开始提取 24 kHz Mel+F0+Energy（候选 {len(tasks)} 条，目标上限 {cfg['max_train_hours']}h）...", flush=True)
    processed = []
    total_sec = 0.0
    mel_sum = np.zeros(cfg["n_mels"], dtype=np.float64)
    mel_sq = np.zeros(cfg["n_mels"], dtype=np.float64)
    mel_cnt = 0
    f0_vals, e_vals = [], []

    def _consume(res):
        nonlocal total_sec, mel_cnt
        if res is None:
            return
        it, fpath, dur_sec, mel_np, f0_np, e_np, uv_np, n_ph, n_fr = res
        if total_sec >= max_total_sec:
            return
        total_sec += dur_sec
        mel_f64 = np.asarray(mel_np, dtype=np.float64)
        mel_sum[:] += mel_f64.sum(axis=-1)
        mel_sq[:] += (mel_f64 ** 2).sum(axis=-1)
        mel_cnt += mel_f64.shape[-1]
        if uv_np.sum() > 0:
            f0_vals.append(f0_np[uv_np > 0.5])
        e_vals.append(e_np)
        processed.append({
            "utt_id": it["utt_id"],
            "text": it["text"],
            "feat_path": fpath,
            "dur_sec": dur_sec,
            "n_phonemes": int(n_ph),
            "n_frames": int(n_fr),
        })

    # 在主线程预热一次 DSP/Torch STFT，确保所有懒加载与 JIT 编译完成后再开多线程池
    _dummy_wav = np.zeros(int(0.2 * cfg["sample_rate"]), dtype=np.float32)
    with torch.no_grad():
        _ = MEL_EXTRACTOR(torch.from_numpy(_dummy_wav))
    _ = extract_f0_and_energy(_dummy_wav, cfg["sample_rate"], cfg["hop_length"], 10)
    del _dummy_wav

    prev_threads = torch.get_num_threads()
    prep_workers = min(4 if IS_COLAB else 8, max(2, CPU_CORES // 2 if CPU_CORES >= 4 else CPU_CORES))
    try:
        if prep_workers >= 2 and len(tasks) > 0:
            torch.set_num_threads(1)
            print(f"[Preprocess] 启用 {prep_workers} 线程并行提取 24 kHz 音频特征（单线程 Torch 防内存膨胀）...", flush=True)
            pbar = tqdm(total=len(tasks), desc="Parallel Extract", mininterval=2.0)
            done_cnt = 0
            with ThreadPoolExecutor(max_workers=prep_workers) as ex:
                batch_sz = 200
                for st in range(0, len(tasks), batch_sz):
                    if total_sec >= max_total_sec:
                        break
                    sub = tasks[st : st + batch_sz]
                    for res in ex.map(_extract_single_item, sub):
                        _consume(res)
                        pbar.update(1)
                        done_cnt += 1
                    if done_cnt % 600 == 0:
                        pbar.set_postfix(hours=f"{total_sec/3600:.2f}/{cfg['max_train_hours']+0.5:.1f}")
                        trim_process_memory()
                    if done_cnt % 1200 == 0:
                        print(f"  [Preprocess] 已处理 {done_cnt}/{len(tasks)} 条，有效累计 {total_sec/3600:.2f}h", flush=True)
            pbar.close()
        else:
            pbar = tqdm(tasks, desc="Extract 24kHz Mel+F0+Energy", mininterval=2.0)
            for idx_t, t in enumerate(pbar, 1):
                if total_sec >= max_total_sec:
                    break
                _consume(_extract_single_item(t))
                if idx_t % 500 == 0:
                    pbar.set_postfix(hours=f"{total_sec/3600:.2f}/{cfg['max_train_hours']+0.5:.1f}")
                    trim_process_memory()
    finally:
        torch.set_num_threads(prev_threads)

    del tasks, cand_items, items
    trim_process_memory()

    mel_mean = (mel_sum / max(mel_cnt, 1)).astype(np.float32)
    mel_std = np.sqrt(np.maximum(mel_sq / max(mel_cnt, 1) - mel_mean ** 2, 1e-4)).astype(np.float32)
    all_f0 = np.concatenate(f0_vals) if f0_vals else np.array([5.0], dtype=np.float32)
    all_e = np.concatenate(e_vals) if e_vals else np.array([-3.0], dtype=np.float32)

    stats = {
        "mel_mean": mel_mean.tolist(),
        "mel_std": mel_std.tolist(),
        "f0_mean": float(np.mean(all_f0)),
        "f0_std": float(max(np.std(all_f0), 1e-3)),
        "e_mean": float(np.mean(all_e)),
        "e_std": float(max(np.std(all_e), 1e-3)),
    }

    n_val = min(cfg["val_utts"], max(10, len(processed) // 10))
    val_list = processed[:n_val]
    train_list = processed[n_val:]

    meta = {
        "source": src,
        "total_hours": round(total_sec / 3600.0, 3),
        "train_count": len(train_list),
        "val_count": len(val_list),
        "stats": stats,
        "train": train_list,
        "val": val_list,
    }
    atomic_write_text(json.dumps(meta, ensure_ascii=False, indent=2), local_manifest)
    if not IS_LOCAL:
        sync_file_to_storage(local_manifest, STORAGE_ROOT / "datasets" / f"{tag}_manifest.json")
        # 特征已提取完毕，Colab/Kaggle 下清理临时解压的 48kHz 原始 WAV 释放约 4.5GB 本地空间
        if not _is_subpath(local_raw_dir, STORAGE_ROOT):
            shutil.rmtree(local_raw_dir, ignore_errors=True)

    # Colab 下把特征目录按 ~250MB 分片打包备份到 Google Drive（避免单写 4GB+ 大文件撑爆 DriveFS 内存/磁盘）
    if SYNC_FEATURES_TAR_TO_DRIVE and not storage_shards_meta.exists():
        print(f"[Backup] 将预处理特征分片 (~250MB/片) 备份至 Google Drive ({storage_shards_dir})...", flush=True)
        try:
            storage_shards_dir.mkdir(parents=True, exist_ok=True)
            all_files = sorted([p for p in local_feat_dir.iterdir() if p.is_file()])
            max_shard_bytes = 250 * 1024 * 1024
            shards, cur_group, cur_bytes = [], [], 0
            for fp in all_files:
                fsz = fp.stat().st_size
                if cur_group and cur_bytes + fsz > max_shard_bytes:
                    shards.append(cur_group)
                    cur_group, cur_bytes = [], 0
                cur_group.append(fp)
                cur_bytes += fsz
            if cur_group:
                shards.append(cur_group)

            shard_records = []
            for idx, grp in enumerate(tqdm(shards, desc="Backup Shards to Drive", mininterval=1.0)):
                sname = f"part_{idx:03d}.tar"
                dst_shard = storage_shards_dir / sname
                if dst_shard.exists() and dst_shard.stat().st_size > 0:
                    shard_records.append({"name": sname, "files": len(grp), "bytes": dst_shard.stat().st_size})
                    continue
                tmp_shard = LOCAL_WORK / f"{tag}_{sname}"
                with tarfile.open(tmp_shard, "w") as tf:
                    for fp in grp:
                        tf.add(fp, arcname=f"{local_feat_dir.name}/{fp.name}")
                sync_file_to_storage(tmp_shard, dst_shard)
                shard_records.append({"name": sname, "files": len(grp), "bytes": tmp_shard.stat().st_size})
                tmp_shard.unlink(missing_ok=True)
                trim_process_memory()

            meta_local = LOCAL_WORK / f"{tag}_shards_meta.json"
            atomic_write_text(json.dumps({"shards": shard_records, "total_files": len(all_files)}, indent=2), meta_local)
            sync_file_to_storage(meta_local, storage_shards_meta)
            meta_local.unlink(missing_ok=True)
        except Exception as e:
            print(f"[Backup Warn] 云盘特征分片备份未完成 ({e})，不影响当前本地高速盘继续训练。", flush=True)

    save_progress_status("dataset", {"status": "ready", "hours": meta["total_hours"], "train": len(train_list), "val": len(val_list)})
    print(f"[Ready] 有效数据 {len(processed)} 句（共 {total_sec/3600:.2f} 小时）| Train={len(train_list)} | Val={len(val_list)}")
    return meta


DATASET_META = download_and_prepare_dataset(CFG)


## 5. 核心模型实现：聚焦流式输入的 5 类主干（`E0`–`E4`）+ 单调对齐器 + 有限前瞻帧解码器与声码器


In [ ]:
# ==============================================================================
# 5.1 集中式有限右前瞻层 & 严格因果 ConvNeXt 块（100% 数值等价流式缓存）
# ==============================================================================
class CentralizedLookaheadBlock(nn.Module):
    def __init__(self, d_model: int, k_left: int = 3, lookahead: int = 4, mult: int = 2):
        super().__init__()
        self.d_model = d_model
        self.k_left = k_left
        self.lookahead = lookahead
        self.kernel_size = k_left + 1 + lookahead
        self.dwconv = nn.Conv1d(d_model, d_model, kernel_size=self.kernel_size, groups=d_model)
        self.norm = nn.LayerNorm(d_model)
        self.pw1 = nn.Linear(d_model, d_model * mult)
        self.pw2 = nn.Linear(d_model * mult, d_model)

    def _apply_body(self, conv_out: torch.Tensor, residual: torch.Tensor) -> torch.Tensor:
        h = self.norm(conv_out.transpose(1, 2))
        h = self.pw2(F.gelu(self.pw1(h)))
        return residual + h

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_pad = F.pad(x.transpose(1, 2), (self.k_left, self.lookahead))
        return self._apply_body(self.dwconv(x_pad), x)

    def init_state(self, batch_size: int, device, dtype):
        return torch.zeros(batch_size, self.k_left, self.d_model, device=device, dtype=dtype)

    def forward_step(self, x_new: torch.Tensor, state: torch.Tensor, is_final: bool = False):
        buf = torch.cat([state, x_new], dim=1) if x_new.shape[1] > 0 else state
        if is_final and self.lookahead > 0:
            zeros_r = torch.zeros(buf.shape[0], self.lookahead, self.d_model, device=buf.device, dtype=buf.dtype)
            buf = torch.cat([buf, zeros_r], dim=1)
        if buf.shape[1] < self.kernel_size:
            return torch.zeros(buf.shape[0], 0, self.d_model, device=buf.device, dtype=buf.dtype), buf
        n_emit = buf.shape[1] - self.kernel_size + 1
        conv_out = self.dwconv(buf.transpose(1, 2))
        residual = buf[:, self.k_left : self.k_left + n_emit, :]
        return self._apply_body(conv_out, residual), buf[:, n_emit:, :]


class CausalConv1d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, groups: int = 1):
        super().__init__()
        self.kernel_size = kernel_size
        self.pad_left = kernel_size - 1
        self.in_ch = in_ch
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, groups=groups)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(F.pad(x, (self.pad_left, 0)))

    def init_state(self, batch_size: int, device, dtype):
        return torch.zeros(batch_size, self.in_ch, self.pad_left, device=device, dtype=dtype)

    def forward_step(self, x: torch.Tensor, state: torch.Tensor):
        if x.shape[-1] == 0:
            return torch.zeros(x.shape[0], self.conv.out_channels, 0, device=x.device, dtype=x.dtype), state
        buf = torch.cat([state, x], dim=-1)
        out = self.conv(buf)
        new_state = buf[:, :, -self.pad_left:] if self.pad_left > 0 else state
        return out, new_state


class LookaheadConv1d(nn.Module):
    """k_left 步左上下文 + lookahead 步有限右前瞻的 Conv1d（集中式前瞻：只允许这一层看右边）。
    流式时需攒够 lookahead 帧未来输入才吐出当前帧；is_final 时右侧补零，与离线 F.pad 逐样本等价。"""

    def __init__(self, in_ch: int, out_ch: int, k_left: int, lookahead: int, groups: int = 1):
        super().__init__()
        self.k_left = k_left
        self.lookahead = lookahead
        self.kernel_size = k_left + 1 + lookahead
        self.in_ch = in_ch
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=self.kernel_size, groups=groups)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(F.pad(x, (self.k_left, self.lookahead)))

    def init_state(self, batch_size: int, device, dtype):
        # 初始只有 k_left 帧左上下文； lookahead 帧待处理输入是流式开始后才攒出来的，不能预先占位（否则开头多吐 lookahead 帧）
        return torch.zeros(batch_size, self.in_ch, self.k_left, device=device, dtype=dtype)

    def forward_step(self, x: torch.Tensor, state: torch.Tensor, is_final: bool = False):
        buf = torch.cat([state, x], dim=-1) if x.shape[-1] > 0 else state
        if is_final and self.lookahead > 0:
            buf = F.pad(buf, (0, self.lookahead))
        n_emit = buf.shape[-1] - self.kernel_size + 1
        if n_emit <= 0:
            return torch.zeros(buf.shape[0], self.conv.out_channels, 0, device=buf.device, dtype=buf.dtype), buf
        keep_len = self.k_left + self.lookahead
        new_state = buf[:, :, -keep_len:] if keep_len > 0 else buf[:, :, :0]
        return self.conv(buf), new_state


class CausalConvNeXtBlock(nn.Module):
    def __init__(self, d_model: int, kernel_size: int = 5, mult: int = 3):
        super().__init__()
        self.dwconv = CausalConv1d(d_model, d_model, kernel_size=kernel_size, groups=d_model)
        self.norm = nn.LayerNorm(d_model)
        self.pw1 = nn.Linear(d_model, d_model * mult)
        self.pw2 = nn.Linear(d_model * mult, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.dwconv(x.transpose(1, 2)).transpose(1, 2)
        h = self.pw2(F.gelu(self.pw1(self.norm(h))))
        return x + h

    def init_state(self, batch_size: int, device, dtype):
        return self.dwconv.init_state(batch_size, device, dtype)

    def forward_step(self, x: torch.Tensor, state: torch.Tensor):
        if x.shape[1] == 0:
            return x, state
        h, new_state = self.dwconv.forward_step(x.transpose(1, 2), state)
        h = self.pw2(F.gelu(self.pw1(self.norm(h.transpose(1, 2)))))
        return x + h, new_state


# ==============================================================================
# 5.1b 门控 DeltaNet（E4）训练期 log 空间分块并行（与逐步递推数学等价；推理期走逐步递推）
# ==============================================================================
SCAN_CHUNK = 32  # 分块块长（音素步数）：块内全并行、块间只递推状态


def pad_to_chunk(x: torch.Tensor, chunk: int = SCAN_CHUNK, value: float = 0.0) -> torch.Tensor:
    """沿 dim=1（时间）右填充到 chunk 的整数倍；填充值须取对应递推的结合律单位元"""
    pad = (-x.shape[1]) % chunk
    return F.pad(x, (0, 0) * (x.dim() - 2) + (0, pad), value=value) if pad else x


def unit_lower_tri_inverse(N: torch.Tensor) -> torch.Tensor:
    """求 (I - N)^{-1}，N 为严格下三角（幂零）。S_m = Σ_{j<m} N^j，倍增 S_{2m} = S_m + N^m·S_m，共 ceil(log2(L)) 步全并行。"""
    L = N.shape[-1]
    P, A = N, torch.eye(L, device=N.device, dtype=N.dtype).expand_as(N)
    for _ in range(max(1, math.ceil(math.log2(L)))):
        A = A + P @ A
        P = P @ P
    return A


def gated_delta_rule_chunkwise(q, k, v, beta, log_alpha, S_init, chunk: int = SCAN_CHUNK):
    """标准门控 DeltaNet（Yang et al., ICLR 2025）的 log 空间分块并行形式，与 gated_delta_rule_recurrent 数学等价：
        S_t = α_t (I - β_t k_t k_t^T) S_{t-1} + β_t k_t v_t^T = α_t S_{t-1} + β_t k_t (v_t - α_t S_{t-1}^T k_t)^T
    记 log_G_t = Σ_{i<=t} log α_i <= 0，块内：
        (I + T) W = R，  T_{n,i} = β_n exp(log_G_n - log_G_i)(k_n·k_i)  (i < n)，  R_n = β_n (v_n - G_n S_0^T k_n)
        O = G ⊙ (Q S_0) + [tril(Q K^T ⊙ exp(log_G_n - log_G_i))] W，  S_L = G_L S_0 + K^T diag(exp(log_G_L - log_G)) W
    全程在 log 空间先 mask 后 exp，彻底杜绝强遗忘下 G/G 的 0/0 与 inf*0 NaN。
    入参布局：q,k (B,T,H,K)，v (B,T,H,V)，beta/log_alpha (B,T,H)；S (B,H,K,V)。"""
    T = q.shape[1]
    q, k, v, beta, log_alpha = (pad_to_chunk(t, chunk) for t in (q, k, v, beta, log_alpha))
    S, ys = S_init, []
    for s0 in range(0, q.shape[1], chunk):
        qc, kc, vc = (t[:, s0:s0 + chunk].transpose(1, 2) for t in (q, k, v))                          # (B,H,L,K) / (B,H,L,V)
        bc, la = beta[:, s0:s0 + chunk].transpose(1, 2), log_alpha[:, s0:s0 + chunk].transpose(1, 2)   # (B,H,L)
        L = qc.shape[2]
        tri = torch.ones(L, L, device=q.device, dtype=torch.bool)
        strict_mask = torch.tril(tri, diagonal=-1)
        causal_mask = torch.tril(tri)
        log_G = torch.cumsum(la, dim=-1)                                                # (B,H,L) <= 0
        G = log_G.exp()                                                                 # (B,H,L) ∈ (0, 1]
        log_ratio = log_G[..., :, None] - log_G[..., None, :]                           # 下三角 <= 0
        ratio_strict = torch.where(strict_mask, log_ratio, torch.zeros_like(log_ratio)).exp() * strict_mask
        ratio_causal = torch.where(causal_mask, log_ratio, torch.zeros_like(log_ratio)).exp() * causal_mask
        tail_ratio = (log_G[..., -1:] - log_G).exp()                                    # (B,H,L) <= 0 -> ∈ (0, 1]
        T_mat = bc[..., None] * torch.einsum("bhik,bhjk->bhij", kc, kc) * ratio_strict
        R = bc[..., None] * (vc - G[..., None] * torch.einsum("bhik,bhkv->bhiv", kc, S))
        W = unit_lower_tri_inverse(-T_mat) @ R
        y = G[..., None] * torch.einsum("bhik,bhkv->bhiv", qc, S) \
            + (torch.einsum("bhik,bhjk->bhij", qc, kc) * ratio_causal) @ W
        S = G[..., -1][..., None, None] * S + torch.einsum("bhik,bhi,bhiv->bhkv", kc, tail_ratio, W)
        ys.append(y)
    return torch.cat(ys, dim=2).transpose(1, 2)[:, :T], S


def gated_delta_rule_recurrent(q, k, v, beta, log_alpha, S):
    """标准门控 DeltaNet 逐步递推（流式推理 + 等价性参考）：
        S_t = α_t S_{t-1} + β_t k_t (v_t - α_t S_{t-1}^T k_t)^T，  o_t = S_t^T q_t"""
    ys = []
    for t in range(q.shape[1]):
        a, b = torch.exp(log_alpha[:, t])[..., None, None], beta[:, t][..., None, None]
        S_dec = a * S
        S = S_dec + b * torch.einsum("bhk,bhv->bhkv", k[:, t], v[:, t] - torch.einsum("bhk,bhkv->bhv", k[:, t], S_dec))
        ys.append(torch.einsum("bhkv,bhk->bhv", S, q[:, t]))
    return torch.stack(ys, dim=1), S


# ==============================================================================
# 5.2 五类序列主干（E0 离线双向 / E1 Conv+UniGRU / E2 SWA 浅前瞻 / E3 分块深前瞻注意力 / E4 门控 DeltaNet）
# ==============================================================================
def apply_rope(x: torch.Tensor, pos: torch.Tensor) -> torch.Tensor:
    dh = x.shape[-1]
    inv_freq = 1.0 / (10000.0 ** (torch.arange(0, dh, 2, device=x.device, dtype=torch.float32) / dh))
    freqs = torch.outer(pos.to(torch.float32), inv_freq)
    cos = freqs.cos()[None, None, :, :].to(x.dtype)
    sin = freqs.sin()[None, None, :, :].to(x.dtype)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


def _apply_rope_with_cache(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


class OfflineBiTransformerBackbone(nn.Module):
    def __init__(self, d_model: int = 256, n_layers: int = 4, n_heads: int = 4, d_ff: int = 896):
        super().__init__()
        self.is_streaming = False
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=0.1, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)
        self.pos_emb = nn.Embedding(512, d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        T = x.shape[1]
        pos = torch.arange(T, device=x.device).clamp(max=511)
        h = x + self.pos_emb(pos)[None, :, :]
        pad_mask = (~mask) if mask is not None else None
        return self.encoder(h, src_key_padding_mask=pad_mask)


class ConvUniGRUBackbone(nn.Module):
    def __init__(self, d_model: int = 256, k_left: int = 3, lookahead: int = 4, n_conv: int = 4):
        super().__init__()
        self.is_streaming = True
        self.lookahead_layer = CentralizedLookaheadBlock(d_model, k_left=k_left, lookahead=lookahead, mult=2)
        self.convs = nn.ModuleList([CausalConvNeXtBlock(d_model, kernel_size=5, mult=3) for _ in range(n_conv)])
        self.gru_norm = nn.LayerNorm(d_model)
        self.gru = nn.GRU(d_model, d_model, num_layers=2, batch_first=True, bidirectional=False)
        self.out_norm = nn.LayerNorm(d_model)
        self.out_ffn = nn.Sequential(nn.Linear(d_model, d_model * 3), nn.GELU(), nn.Linear(d_model * 3, d_model))

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        h = self.lookahead_layer(x)
        for blk in self.convs:
            h = blk(h)
        gru_out, _ = self.gru(self.gru_norm(h))
        h = h + gru_out
        return h + self.out_ffn(self.out_norm(h))

    def init_state(self, batch_size: int, device, dtype):
        return {
            "la": self.lookahead_layer.init_state(batch_size, device, dtype),
            "convs": [blk.init_state(batch_size, device, dtype) for blk in self.convs],
            "gru": torch.zeros(2, batch_size, self.gru.hidden_size, device=device, dtype=dtype),
        }

    def forward_step(self, x_new: torch.Tensor, state: dict, is_final: bool = False):
        h, la_st = self.lookahead_layer.forward_step(x_new, state["la"], is_final=is_final)
        if h.shape[1] == 0:
            return h, {"la": la_st, "convs": state["convs"], "gru": state["gru"]}
        new_convs = []
        for blk, cst in zip(self.convs, state["convs"]):
            h, nc = blk.forward_step(h, cst)
            new_convs.append(nc)
        gru_out, new_gru = self.gru(self.gru_norm(h), state["gru"])
        h = h + gru_out
        h = h + self.out_ffn(self.out_norm(h))
        return h, {"la": la_st, "convs": new_convs, "gru": new_gru}


class SWAConvLayer(nn.Module):
    def __init__(self, d_model: int = 256, n_heads: int = 4, window_size: int = 48):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size
        self.norm1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.proj = nn.Linear(d_model, d_model)
        self.conv_ffn = CausalConvNeXtBlock(d_model, kernel_size=5, mult=2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        qkv = self.qkv(self.norm1(x)).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = [t.transpose(1, 2) for t in qkv.unbind(dim=2)]
        pos = torch.arange(T, device=x.device)
        q, k = apply_rope(q, pos), apply_rope(k, pos)
        diff = pos[:, None] - pos[None, :]
        allow = (diff >= 0) & (diff < self.window_size)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~allow[None, None, :, :], float("-inf"))
        out = torch.matmul(torch.softmax(scores, dim=-1), v).transpose(1, 2).contiguous().view(B, T, D)
        return self.conv_ffn(x + self.proj(out))

    def init_state(self, batch_size: int, device, dtype):
        return {
            "k": torch.zeros(batch_size, self.n_heads, 0, self.head_dim, device=device, dtype=dtype),
            "v": torch.zeros(batch_size, self.n_heads, 0, self.head_dim, device=device, dtype=dtype),
            "pos": 0,
            "conv": self.conv_ffn.init_state(batch_size, device, dtype),
        }

    def forward_step(self, x: torch.Tensor, state: dict):
        B, C, D = x.shape
        if C == 0:
            return x, state
        qkv = self.qkv(self.norm1(x)).view(B, C, 3, self.n_heads, self.head_dim)
        q, k, v = [t.transpose(1, 2) for t in qkv.unbind(dim=2)]
        pos_q = torch.arange(state["pos"], state["pos"] + C, device=x.device)
        q, k = apply_rope(q, pos_q), apply_rope(k, pos_q)
        k_all = torch.cat([state["k"], k], dim=2)
        v_all = torch.cat([state["v"], v], dim=2)
        k_len = k_all.shape[2]
        pos_k = torch.arange(state["pos"] + C - k_len, state["pos"] + C, device=x.device)
        diff = pos_q[:, None] - pos_k[None, :]
        allow = (diff >= 0) & (diff < self.window_size)
        scores = torch.matmul(q, k_all.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~allow[None, None, :, :], float("-inf"))
        out = torch.matmul(torch.softmax(scores, dim=-1), v_all).transpose(1, 2).contiguous().view(B, C, D)
        x, new_conv = self.conv_ffn.forward_step(x + self.proj(out), state["conv"])
        keep = self.window_size - 1
        return x, {
            "k": k_all[:, :, -keep:, :] if keep > 0 else k_all[:, :, :0, :],
            "v": v_all[:, :, -keep:, :] if keep > 0 else v_all[:, :, :0, :],
            "pos": state["pos"] + C,
            "conv": new_conv,
        }


class SWABackbone(nn.Module):
    def __init__(self, d_model: int = 256, k_left: int = 3, lookahead: int = 4, n_layers: int = 5, n_heads: int = 4, window_size: int = 48):
        super().__init__()
        self.is_streaming = True
        self.lookahead_layer = CentralizedLookaheadBlock(d_model, k_left=k_left, lookahead=lookahead, mult=2)
        self.layers = nn.ModuleList([SWAConvLayer(d_model, n_heads=n_heads, window_size=window_size) for _ in range(n_layers)])

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        h = self.lookahead_layer(x)
        for l in self.layers:
            h = l(h)
        return h

    def init_state(self, batch_size: int, device, dtype):
        return {"la": self.lookahead_layer.init_state(batch_size, device, dtype), "layers": [l.init_state(batch_size, device, dtype) for l in self.layers]}

    def forward_step(self, x_new: torch.Tensor, state: dict, is_final: bool = False):
        h, la_st = self.lookahead_layer.forward_step(x_new, state["la"], is_final=is_final)
        if h.shape[1] == 0:
            return h, {"la": la_st, "layers": state["layers"]}
        new_layers = []
        for l, st in zip(self.layers, state["layers"]):
            h, nst = l.forward_step(h, st)
            new_layers.append(nst)
        return h, {"la": la_st, "layers": new_layers}


class ChunkLookaheadLayer(nn.Module):
    """单层分块深前瞻注意力 + 因果 ConvNeXt：
    在当前 [commit_chunk C + lookahead L] 窗口内做完整双向自注意力，同时因果读取长度 <= W 的历史 KV Cache；
    算完仅提交前 commit_len 个 token 的 KV 与卷积缓存，彻底丢弃 lookahead token 的状态，保证多层堆叠零前瞻膨胀。
    训练期通过 forward_windows 实现层主序零 chunk 循环全并行（Attention + ConvNeXt 同步向量化），与流式 forward_chunk 100% 数学等价。"""

    def __init__(self, d_model: int = 256, n_heads: int = 4, window_size: int = 48):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size
        self.norm1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, d_model * 3, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.conv = CausalConvNeXtBlock(d_model, kernel_size=5, mult=2)
        # 非持久化 RoPE 三角查表缓存（预建 512 步，零 state_dict 改动，兼容旧 a4 checkpoint 且对 torch.compile 友好）
        _pos = torch.arange(512, dtype=torch.float32)
        _inv = 1.0 / (10000.0 ** (torch.arange(0, self.head_dim, 2, dtype=torch.float32) / self.head_dim))
        _freqs = torch.outer(_pos, _inv)
        self.register_buffer("_rope_cos", _freqs.cos(), persistent=False)
        self.register_buffer("_rope_sin", _freqs.sin(), persistent=False)

    def _get_rope(self, needed_len: int, device: torch.device, dtype: torch.dtype):
        if self._rope_cos.shape[0] < needed_len or self._rope_cos.device != device:
            cap = max(needed_len, self._rope_cos.shape[0], 512)
            with torch.inference_mode(False), torch.no_grad():
                pos = torch.arange(cap, device=device, dtype=torch.float32)
                inv_freq = 1.0 / (10000.0 ** (torch.arange(0, self.head_dim, 2, device=device, dtype=torch.float32) / self.head_dim))
                freqs = torch.outer(pos, inv_freq)
                self._rope_cos = freqs.cos()
                self._rope_sin = freqs.sin()
        return self._rope_cos[:needed_len].to(dtype), self._rope_sin[:needed_len].to(dtype)

    def init_state(self, batch_size: int, device, dtype):
        return {
            "k": torch.zeros(batch_size, self.n_heads, 0, self.head_dim, device=device, dtype=dtype),
            "v": torch.zeros(batch_size, self.n_heads, 0, self.head_dim, device=device, dtype=dtype),
            "pos": 0,
            "conv": self.conv.init_state(batch_size, device, dtype),
        }

    def forward_chunk(self, x_win: torch.Tensor, state: dict, commit_len: int, win_mask: torch.Tensor = None, output_commit_only: bool = False):
        B, M, D = x_win.shape
        xn = self.norm1(x_win)
        # 末层剪枝：前瞻位置的输出与卷积状态都会被丢弃，只需计算完整 M 步 K/V，而 Q/Attn/out_proj/ConvNeXt 仅算前 commit_len 步
        if output_commit_only and commit_len < M and type(self.qkv) is nn.Linear:
            w_q, w_kv = self.qkv.weight.split([D, 2 * D], dim=0)
            q = F.linear(xn[:, :commit_len, :], w_q).reshape(B, commit_len, self.n_heads, self.head_dim).transpose(1, 2)
            kv = F.linear(xn, w_kv).reshape(B, M, 2, self.n_heads, self.head_dim)
            k, v = kv.permute(2, 0, 3, 1, 4)
        else:
            qkv = self.qkv(xn).reshape(B, M, 3, self.n_heads, self.head_dim)
            q, k, v = qkv.permute(2, 0, 3, 1, 4)
            if output_commit_only and commit_len < M:
                q = q[:, :, :commit_len, :]
        q_len = q.shape[2]
        cos, sin = self._get_rope(state["pos"] + M, x_win.device, q.dtype)
        cos_m = cos[state["pos"] : state["pos"] + M][None, None, :, :]
        sin_m = sin[state["pos"] : state["pos"] + M][None, None, :, :]
        q = _apply_rope_with_cache(q, cos_m[:, :, :q_len, :], sin_m[:, :, :q_len, :])
        k = _apply_rope_with_cache(k, cos_m, sin_m)
        k_all = torch.cat([state["k"], k], dim=2)
        v_all = torch.cat([state["v"], v], dim=2)

        scores = torch.matmul(q, k_all.transpose(-2, -1)) * (self.head_dim ** -0.5)
        if win_mask is not None:
            past_len = state["k"].shape[2]
            key_valid = torch.cat(
                [torch.ones(B, past_len, device=x_win.device, dtype=torch.bool), win_mask],
                dim=1,
            )
            scores = scores.masked_fill(~key_valid[:, None, None, :], torch.finfo(scores.dtype).min)
        out = torch.matmul(torch.softmax(scores, dim=-1), v_all).transpose(1, 2).reshape(B, q_len, D)
        h = x_win[:, :q_len, :] + self.out_proj(out)

        # 局部因果 ConvNeXt：仅将 commit_len 段写入下一块卷积状态（因果卷积不依赖右侧被裁掉的 lookahead）
        pad_left = self.conv.dwconv.pad_left
        buf = torch.cat([state["conv"], h.transpose(1, 2)], dim=-1)
        conv_out = self.conv.dwconv.conv(buf).transpose(1, 2)
        h_out = h + self.conv.pw2(F.gelu(self.conv.pw1(self.conv.norm(conv_out))))
        new_conv = buf[:, :, commit_len : commit_len + pad_left] if pad_left > 0 else state["conv"]

        new_k = torch.cat([state["k"], k[:, :, :commit_len, :]], dim=2)[:, :, -self.window_size:, :]
        new_v = torch.cat([state["v"], v[:, :, :commit_len, :]], dim=2)[:, :, -self.window_size:, :]
        return h_out, {"k": new_k, "v": new_v, "pos": state["pos"] + commit_len, "conv": new_conv}

    def forward_windows(
        self,
        x_win: torch.Tensor,
        win_mask: torch.Tensor,
        past_valid: torch.Tensor,
        cos_win: torch.Tensor,
        sin_win: torch.Tensor,
        chunk_size: int,
        output_commit_only: bool = False,
    ) -> torch.Tensor:
        """层主序全窗口零循环并行前向（Attention + ConvNeXt 同步向量化，与逐块 forward_chunk 数学等价）。"""
        B, N, M, D = x_win.shape
        C, W = chunk_size, self.window_size
        H, dh = self.n_heads, self.head_dim
        q_len = C if (output_commit_only and C < M) else M

        xn = self.norm1(x_win)
        if output_commit_only and C < M and type(self.qkv) is nn.Linear:
            w_q, w_kv = self.qkv.weight.split([D, 2 * D], dim=0)
            q = F.linear(xn[:, :, :C, :], w_q).reshape(B, N, C, H, dh).permute(0, 1, 3, 2, 4)
            kv = F.linear(xn, w_kv).reshape(B, N, M, 2, H, dh)
            k = kv[:, :, :, 0].permute(0, 1, 3, 2, 4)
            v = kv[:, :, :, 1].permute(0, 1, 3, 2, 4)
        else:
            qkv = self.qkv(xn).reshape(B, N, M, 3, H, dh)
            q = qkv[:, :, :q_len, 0].permute(0, 1, 3, 2, 4)
            k = qkv[:, :, :, 1].permute(0, 1, 3, 2, 4)
            v = qkv[:, :, :, 2].permute(0, 1, 3, 2, 4)

        cos_b = cos_win[None, :, None, :, :].to(q.dtype)
        sin_b = sin_win[None, :, None, :, :].to(q.dtype)
        q = _apply_rope_with_cache(q, cos_b[:, :, :, :q_len, :], sin_b[:, :, :, :q_len, :])
        k = _apply_rope_with_cache(k, cos_b, sin_b)

        # 用 F.pad + unfold 零循环组装全部 N 个窗口的历史 KV Cache（仅含前序窗口已提交的前 C 步）
        if W > 0:
            k_commit = k[:, :, :, :C, :].permute(0, 2, 1, 3, 4).reshape(B, H, N * C, dh)
            v_commit = v[:, :, :, :C, :].permute(0, 2, 1, 3, 4).reshape(B, H, N * C, dh)
            end_p = W + (N - 1) * C
            k_past = F.pad(k_commit, (0, 0, W, 0))[:, :, :end_p, :].unfold(2, W, C).permute(0, 2, 1, 4, 3)
            v_past = F.pad(v_commit, (0, 0, W, 0))[:, :, :end_p, :].unfold(2, W, C).permute(0, 2, 1, 4, 3)
            k_all = torch.cat([k_past, k], dim=3)
            v_all = torch.cat([v_past, v], dim=3)
            key_valid = torch.cat([past_valid[None, :, :].expand(B, -1, -1), win_mask], dim=2)
        else:
            k_all, v_all, key_valid = k, v, win_mask

        # B*N 个窗口合并为单次 SDPA（每个窗口至少有 1 个 True key，前反向严格无 NaN）
        out = F.scaled_dot_product_attention(
            q.reshape(B * N, H, q_len, dh),
            k_all.reshape(B * N, H, W + M, dh),
            v_all.reshape(B * N, H, W + M, dh),
            attn_mask=key_valid.reshape(B * N, 1, 1, W + M),
        )
        out = out.transpose(1, 2).reshape(B, N, q_len, D)
        h = x_win[:, :, :q_len, :] + self.out_proj(out)

        # Conv state 仅依赖注意力后的已提交 h[:, :, :C, :]，同样 F.pad + unfold 零循环全并行
        pad_left = self.conv.dwconv.pad_left
        if pad_left > 0:
            h_commit = h[:, :, :C, :].reshape(B, N * C, D)
            end_c = pad_left + (N - 1) * C
            conv_state = F.pad(h_commit, (0, 0, pad_left, 0))[:, :end_c, :].unfold(1, pad_left, C)
            buf = torch.cat([conv_state, h.transpose(2, 3)], dim=3).reshape(B * N, D, pad_left + q_len)
        else:
            buf = h.transpose(2, 3).reshape(B * N, D, q_len)
        conv_out = self.conv.dwconv.conv(buf).transpose(1, 2).reshape(B, N, q_len, D)
        return h + self.conv.pw2(F.gelu(self.conv.pw1(self.conv.norm(conv_out))))


class ChunkLookaheadBackbone(nn.Module):
    """E3 主力流式输入主干：5 层深双向分块前瞻注意力（C=4, L=8, W=48）。
    - 相比 E2（1 层浅层 Conv 前瞻 + 上层严格因果），全部 5 层均可对 [C + L] 未来窗口做深层双向交互；
    - 每次仅提交 C 步状态、丢弃 L 步前瞻 KV，总前瞻严格锁死且天然适配流式前端的“暂定前瞻纠错”；
    - 训练期走层主序全窗口零循环并行 forward（+ 末层 commit_len 剪枝），流式推理走 _run_window，二者 100% 数学等价。"""

    def __init__(self, d_model: int = 256, chunk_size: int = 4, lookahead: int = 8, n_layers: int = 5, n_heads: int = 4, window_size: int = 48):
        super().__init__()
        self.is_streaming = True
        self.d_model = d_model
        self.chunk_size = chunk_size
        self.lookahead = lookahead
        self.layers = nn.ModuleList([ChunkLookaheadLayer(d_model, n_heads=n_heads, window_size=window_size) for _ in range(n_layers)])

    def _run_window(self, win: torch.Tensor, layer_states: list, commit_len: int, win_mask: torch.Tensor = None):
        h = win
        new_layers = []
        last_idx = len(self.layers) - 1
        for idx, (l, st) in enumerate(zip(self.layers, layer_states)):
            h, nst = l.forward_chunk(h, st, commit_len=commit_len, win_mask=win_mask, output_commit_only=(idx == last_idx))
            new_layers.append(nst)
        return h[:, :commit_len, :], new_layers

    def _forward_sequential(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """逐块串行参考路径（供 --selftest 数学等价门禁与性能基准对照）。"""
        B, T, _ = x.shape
        layer_states = [l.init_state(B, x.device, x.dtype) for l in self.layers]
        emits = []
        for s in range(0, T, self.chunk_size):
            c_len = min(self.chunk_size, T - s)
            e = min(T, s + self.chunk_size + self.lookahead)
            win_mask = mask[:, s:e] if mask is not None else None
            h_commit, layer_states = self._run_window(x[:, s:e, :], layer_states, commit_len=c_len, win_mask=win_mask)
            emits.append(h_commit)
        return torch.cat(emits, dim=1) if emits else x[:, :0, :]

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        B, T, D = x.shape
        if T == 0:
            return x[:, :0, :]
        C, L = self.chunk_size, self.lookahead
        M = C + L
        N = (T + C - 1) // C
        T_pad = (N - 1) * C + M
        pad_r = T_pad - T
        x_pad = F.pad(x, (0, 0, 0, pad_r)) if pad_r > 0 else x
        if mask is None:
            mask_pad = (torch.arange(T_pad, device=x.device)[None, :] < T).expand(B, -1)
        else:
            mask_pad = F.pad(mask, (0, pad_r), value=False) if pad_r > 0 else mask

        x_win = x_pad.unfold(1, M, C).transpose(2, 3).contiguous()
        win_mask = mask_pad.unfold(1, M, C).contiguous()

        W = self.layers[0].window_size
        if W > 0:
            chunk_starts = torch.arange(N, device=x.device)[:, None] * C
            past_valid = (chunk_starts - W + torch.arange(W, device=x.device)[None, :]) >= 0
        else:
            past_valid = torch.zeros(N, 0, device=x.device, dtype=torch.bool)

        cos_all, sin_all = self.layers[0]._get_rope(T_pad, x.device, x.dtype)
        cos_win = cos_all[:T_pad].unfold(0, M, C).transpose(1, 2).contiguous()
        sin_win = sin_all[:T_pad].unfold(0, M, C).transpose(1, 2).contiguous()

        h = x_win
        last_idx = len(self.layers) - 1
        for idx, layer in enumerate(self.layers):
            h = layer.forward_windows(
                h, win_mask, past_valid, cos_win, sin_win,
                chunk_size=C, output_commit_only=(idx == last_idx),
            )
        return h[:, :, :C, :].reshape(B, N * C, D)[:, :T, :]

    def init_state(self, batch_size: int, device, dtype):
        return {
            "buf": torch.zeros(batch_size, 0, self.d_model, device=device, dtype=dtype),
            "layers": [l.init_state(batch_size, device, dtype) for l in self.layers],
        }

    def forward_committed_with_lookahead(self, x_commit: torch.Tensor, x_lookahead: torch.Tensor, state: dict):
        """Phase 2 流式前端原生接口：显式区分已确认提交音素与暂定前瞻音素（前瞻永不写入缓存，下一拍可被前端无损修正）。"""
        c_len = x_commit.shape[1]
        if c_len == 0:
            return x_commit, state
        win = torch.cat([x_commit, x_lookahead], dim=1) if x_lookahead.shape[1] > 0 else x_commit
        h_commit, new_layers = self._run_window(win, state["layers"], commit_len=c_len)
        return h_commit, {"buf": state["buf"], "layers": new_layers}

    def forward_step(self, x_new: torch.Tensor, state: dict, is_final: bool = False):
        buf = torch.cat([state["buf"], x_new], dim=1) if x_new.shape[1] > 0 else state["buf"]
        layer_states = state["layers"]
        emits = []
        while buf.shape[1] >= self.chunk_size + self.lookahead:
            win = buf[:, :self.chunk_size + self.lookahead, :]
            h_commit, layer_states = self._run_window(win, layer_states, commit_len=self.chunk_size)
            emits.append(h_commit)
            buf = buf[:, self.chunk_size:, :]
        if is_final:
            while buf.shape[1] > 0:
                c_len = min(self.chunk_size, buf.shape[1])
                win = buf[:, :c_len + self.lookahead, :]
                h_commit, layer_states = self._run_window(win, layer_states, commit_len=c_len)
                emits.append(h_commit)
                buf = buf[:, c_len:, :]
        out = torch.cat(emits, dim=1) if emits else torch.zeros(buf.shape[0], 0, self.d_model, device=buf.device, dtype=buf.dtype)
        return out, {"buf": buf, "layers": layer_states}


class GatedDeltaNetBlock(nn.Module):
    """标准门控 DeltaNet（Yang et al., ICLR 2025）：4-tap 因果短卷积 + 线性注意力 + Delta Rule 写入 + 标量状态门控。
    逐头状态 S ∈ R^{K×V}：
        S_t = α_t (I - β_t k_t k_t^T) S_{t-1} + β_t k_t v_t^T
            = α_t S_{t-1} + β_t k_t (v_t - α_t S_{t-1}^T k_t)^T，  o_t = S_t^T q_t   （q、k 经短卷积+SiLU 后 L2 归一化）
    与 E4 的逐通道衰减累加线性注意力相比，写入前先按 α_t 遗忘旧状态并沿 k_t 方向擦除冗余分量；
    训练期 log 空间分块并行、推理期逐步递推，二者数学等价且强遗忘下数值稳定。"""

    def __init__(self, d_model: int = 256, n_heads: int = 4, ffn_hidden: int = 384, d_conv: int = 4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, d_model * 3, bias=False)
        self.qkv_conv = CausalConv1d(d_model * 3, d_model * 3, kernel_size=d_conv, groups=d_model * 3)
        self.ba_proj = nn.Linear(d_model, n_heads * 2, bias=True)   # 每头两个标量：β（写入强度）与 α（状态门控）
        with torch.no_grad():
            self.ba_proj.bias[:n_heads].zero_()                     # β 初值 0.5
            self.ba_proj.bias[n_heads:].fill_(2.0)                  # α 初值 ≈ 0.88
        self.z = nn.Linear(d_model, d_model, bias=False)
        self.o_norm = nn.GroupNorm(n_heads, d_model)                # 逐时间步、逐头归一化（与 E4 一致，防跨时间泄漏）
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn_gate = nn.Linear(d_model, ffn_hidden, bias=False)
        self.ffn_up = nn.Linear(d_model, ffn_hidden, bias=False)
        self.ffn_down = nn.Linear(ffn_hidden, d_model, bias=False)

    def _run(self, x: torch.Tensor, attn, conv_state, S_init):
        B, T, D = x.shape
        xn = self.norm1(x)
        qkv_in = self.qkv(xn).transpose(1, 2)
        if conv_state is None:
            qkv_c, new_conv = F.silu(self.qkv_conv(qkv_in).transpose(1, 2)), None
        else:
            qkv_raw, new_conv = self.qkv_conv.forward_step(qkv_in, conv_state)
            qkv_c = F.silu(qkv_raw.transpose(1, 2))
        q, k, v = qkv_c.reshape(B, T, 3, self.n_heads, self.head_dim).float().unbind(2)
        q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
        b_logit, a_logit = self.ba_proj(xn).float().reshape(B, T, 2, self.n_heads).unbind(2)
        beta, log_alpha = torch.sigmoid(b_logit), F.logsigmoid(a_logit)
        if S_init is None:
            S_init = torch.zeros(B, self.n_heads, self.head_dim, self.head_dim, device=x.device, dtype=torch.float32)
        o, S = attn(q, k, v, beta, log_alpha, S_init)
        # 先转回 x.dtype 再经逐头 GroupNorm，兼容纯 fp16/bf16 推理且与 E4 一致
        y = self.o_norm(o.reshape(B * T, D).to(x.dtype)).reshape(B, T, D) * F.silu(self.z(xn))
        h = x + self.out_proj(y)
        xc = self.norm2(h)
        return h + self.ffn_down(F.silu(self.ffn_gate(xc)) * self.ffn_up(xc)), new_conv, S

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self._run(x, gated_delta_rule_chunkwise, None, None)[0]

    def init_state(self, batch_size: int, device, dtype):
        return {
            "conv": self.qkv_conv.init_state(batch_size, device, dtype),
            "S": torch.zeros(batch_size, self.n_heads, self.head_dim, self.head_dim, device=device, dtype=torch.float32),
        }

    def forward_step(self, x: torch.Tensor, state: dict):
        if x.shape[1] == 0:
            return x, state
        out, new_conv, S = self._run(x, gated_delta_rule_recurrent, state["conv"], state["S"])
        return out, {"conv": new_conv, "S": S}


class GatedDeltaNetBackbone(nn.Module):
    def __init__(self, d_model: int = 256, k_left: int = 3, lookahead: int = 4, n_layers: int = 5, n_heads: int = 4):
        super().__init__()
        self.is_streaming = True
        self.lookahead_layer = CentralizedLookaheadBlock(d_model, k_left=k_left, lookahead=lookahead, mult=2)
        self.layers = nn.ModuleList([GatedDeltaNetBlock(d_model, n_heads=n_heads, ffn_hidden=384, d_conv=4) for _ in range(n_layers)])

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        h = self.lookahead_layer(x)
        for l in self.layers:
            h = l(h)
        return h

    def init_state(self, batch_size: int, device, dtype):
        return {"la": self.lookahead_layer.init_state(batch_size, device, dtype), "layers": [l.init_state(batch_size, device, dtype) for l in self.layers]}

    def forward_step(self, x_new: torch.Tensor, state: dict, is_final: bool = False):
        h, la_st = self.lookahead_layer.forward_step(x_new, state["la"], is_final=is_final)
        if h.shape[1] == 0:
            return h, {"la": la_st, "layers": state["layers"]}
        new_layers = []
        for l, st in zip(self.layers, state["layers"]):
            h, nst = l.forward_step(h, st)
            new_layers.append(nst)
        return h, {"la": la_st, "layers": new_layers}


# ==============================================================================
# 5.3 内置单调对齐器 (Beta-Binomial 对角先验 + Soft->Hard 直通二值化，批向量化热路径) & 完整声学模型
# ==============================================================================
@torch.no_grad()
def build_beta_binomial_log_prior_ref(text_lens: torch.Tensor, mel_lens: torch.Tensor, max_t: int, max_f: int, scaling: float = 1.0) -> torch.Tensor:
    B = text_lens.shape[0]
    device = text_lens.device
    log_prior = torch.full((B, max_f, max_t), -1e4, device=device, dtype=torch.float32)
    for b in range(B):
        F_b = int(mel_lens[b].item())
        T_b = int(text_lens[b].item())
        if T_b <= 1:
            log_prior[b, :F_b, :T_b] = 0.0
            continue
        n = float(T_b - 1)
        f = torch.arange(F_b, device=device, dtype=torch.float32)[:, None]
        t = torch.arange(T_b, device=device, dtype=torch.float32)[None, :]
        alpha = scaling * (f + 1.0)
        beta = scaling * (float(F_b) - f)
        log_comb = torch.lgamma(torch.tensor(n + 1.0, device=device)) - torch.lgamma(t + 1.0) - torch.lgamma(n - t + 1.0)
        log_beta_num = torch.lgamma(t + alpha) + torch.lgamma(n - t + beta) - torch.lgamma(n + alpha + beta)
        log_beta_den = torch.lgamma(alpha) + torch.lgamma(beta) - torch.lgamma(alpha + beta)
        log_prior[b, :F_b, :T_b] = (log_comb + log_beta_num - log_beta_den).clamp(min=-50.0)
    return log_prior


@torch.no_grad()
def build_beta_binomial_log_prior(text_lens: torch.Tensor, mel_lens: torch.Tensor, max_t: int, max_f: int, scaling: float = 1.0) -> torch.Tensor:
    """批向量化 Beta-Binomial 对角先验：一次广播计算整个 (B, F, T)，与 *_ref 逐位等价。"""
    B = text_lens.shape[0]
    device = text_lens.device
    f_col = torch.arange(max_f, device=device, dtype=torch.float32).view(1, max_f, 1)
    t_row = torch.arange(max_t, device=device, dtype=torch.float32).view(1, 1, max_t)
    n = (text_lens.to(torch.float32) - 1.0).view(B, 1, 1)
    f_b = mel_lens.to(torch.float32).view(B, 1, 1)
    alpha = scaling * (f_col + 1.0)
    beta = scaling * (f_b - f_col)
    log_comb = torch.lgamma(n + 1.0) - torch.lgamma(t_row + 1.0) - torch.lgamma(n - t_row + 1.0)
    log_beta_num = torch.lgamma(t_row + alpha) + torch.lgamma(n - t_row + beta) - torch.lgamma(n + alpha + beta)
    log_beta_den = torch.lgamma(alpha) + torch.lgamma(beta) - torch.lgamma(alpha + beta)
    dense = (log_comb + log_beta_num - log_beta_den).clamp_(min=-50.0)
    valid = (f_col < f_b) & (t_row < text_lens.to(torch.float32).view(B, 1, 1))
    single_tok = (text_lens <= 1).view(B, 1, 1)
    out = torch.full((B, max_f, max_t), -1e4, device=device, dtype=torch.float32)
    out = torch.where(valid & ~single_tok, dense, out)
    out = torch.where(valid & single_tok, torch.zeros_like(out), out)
    return out


class AlignmentEncoder(nn.Module):
    def __init__(self, n_mels: int = 100, d_text: int = 256, d_attn: int = 80, scale: float = 8.0):
        super().__init__()
        self.key_proj = nn.Sequential(nn.Conv1d(d_text, d_text, 3, padding=1), nn.ReLU(), nn.Conv1d(d_text, d_attn, 1))
        self.query_proj = nn.Sequential(nn.Conv1d(n_mels, n_mels * 2, 3, padding=1), nn.ReLU(), nn.Conv1d(n_mels * 2, d_attn, 1))
        self.scale = scale

    def forward(self, mel: torch.Tensor, text_emb: torch.Tensor, text_mask: torch.Tensor, attn_log_prior: torch.Tensor = None):
        # L2 归一化余弦相似度将声学 logits 严格约束在 [-scale, +scale]，永不发散或淹没 Beta-Binomial 对角先验
        keys = F.normalize(self.key_proj(text_emb.transpose(1, 2)).float(), dim=1)
        queries = F.normalize(self.query_proj(mel).float(), dim=1)
        logits = torch.bmm(queries.transpose(1, 2), keys) * self.scale
        if attn_log_prior is not None:
            logits = logits + attn_log_prior.float()
        logits = logits.masked_fill(~text_mask[:, None, :], -1e4)
        return F.log_softmax(logits, dim=-1), torch.softmax(logits, dim=-1)


def forward_sum_loss(attn_logprob: torch.Tensor, text_lens: torch.Tensor, mel_lens: torch.Tensor) -> torch.Tensor:
    B, F_max, _ = attn_logprob.shape
    blank = torch.full((B, F_max, 1), -1e4, device=attn_logprob.device, dtype=torch.float32)
    log_probs = torch.cat([blank, attn_logprob.float()], dim=-1).transpose(0, 1)
    targets = torch.cat([torch.arange(1, tl + 1, device=attn_logprob.device, dtype=torch.long) for tl in text_lens])
    return F.ctc_loss(log_probs, targets, mel_lens, text_lens, blank=0, zero_infinity=True)


@torch.no_grad()
def viterbi_monotonic_alignment_ref(attn_logprob: torch.Tensor, text_lens: torch.Tensor, mel_lens: torch.Tensor):
    B, F_max, T_max = attn_logprob.shape
    lp_np = attn_logprob.detach().cpu().float().numpy()
    durations = np.zeros((B, T_max), dtype=np.int64)
    attn_hard = np.zeros((B, F_max, T_max), dtype=np.float32)
    for b in range(B):
        F_b, T_b = int(mel_lens[b].item()), int(text_lens[b].item())
        val = lp_np[b, :F_b, :T_b]
        dp = np.full((F_b, T_b), -1e9, dtype=np.float32)
        dp[0, 0] = val[0, 0]
        for f in range(1, F_b):
            dp[f, 0] = dp[f - 1, 0] + val[f, 0]
            t_max = min(f + 1, T_b)
            dp[f, 1:t_max] = np.maximum(dp[f - 1, 1:t_max], dp[f - 1, 0:t_max - 1]) + val[f, 1:t_max]
        t = T_b - 1
        for f in range(F_b - 1, -1, -1):
            durations[b, t] += 1
            attn_hard[b, f, t] = 1.0
            if f > 0 and t > 0 and (dp[f - 1, t - 1] > dp[f - 1, t] or f <= t):
                t -= 1
    return torch.from_numpy(attn_hard).to(attn_logprob.device), torch.from_numpy(durations).to(attn_logprob.device)


@torch.no_grad()
def viterbi_monotonic_alignment(attn_logprob: torch.Tensor, text_lens: torch.Tensor, mel_lens: torch.Tensor):
    """批向量化单调 Viterbi 回溯：批与 token 维全程向量化，与 *_ref 逐位等价。"""
    B, F_max, T_max = attn_logprob.shape
    lp = attn_logprob.detach().cpu().float().numpy()
    mel_np = mel_lens.detach().cpu().numpy().astype(np.int64)
    txt_np = text_lens.detach().cpu().numpy().astype(np.int64)

    dp = np.full((B, F_max, T_max), -1e9, dtype=np.float32)
    if F_max > 0:
        dp[:, 0, 0] = lp[:, 0, 0]
    for f in range(1, F_max):
        np.add(dp[:, f - 1, 0], lp[:, f, 0], out=dp[:, f, 0])
        t_hi = min(f + 1, T_max)
        if t_hi > 1:
            np.maximum(dp[:, f - 1, 1:t_hi], dp[:, f - 1, 0:t_hi - 1], out=dp[:, f, 1:t_hi])
            dp[:, f, 1:t_hi] += lp[:, f, 1:t_hi]

    durations = np.zeros((B, T_max), dtype=np.int64)
    attn_hard = np.zeros((B, F_max, T_max), dtype=np.float32)
    t_cur = np.maximum(txt_np - 1, 0)
    for f in range(F_max - 1, -1, -1):
        active = np.nonzero(f < mel_np)[0]
        if active.size == 0:
            continue
        tb = t_cur[active]
        durations[active, tb] += 1
        attn_hard[active, f, tb] = 1.0
        if f > 0:
            d_left = dp[active, f - 1, np.maximum(tb - 1, 0)]
            d_stay = dp[active, f - 1, tb]
            move = (tb > 0) & ((d_left > d_stay) | (f <= tb))
            t_cur[active] = tb - move
    return torch.from_numpy(attn_hard).to(attn_logprob.device), torch.from_numpy(durations).to(attn_logprob.device)


def expand_by_duration(h: torch.Tensor, durations: torch.Tensor, max_frames: int = None):
    B, _, D = h.shape
    outs, out_lens = [], []
    for b in range(B):
        rep = torch.repeat_interleave(h[b], durations[b].clamp(min=0), dim=0)
        if max_frames is not None:
            rep = F.pad(rep, (0, 0, 0, max_frames - rep.shape[0])) if rep.shape[0] < max_frames else rep[:max_frames]
        outs.append(rep); out_lens.append(rep.shape[0])
    F_max = max(out_lens) if max_frames is None else max_frames
    padded = torch.zeros(B, F_max, D, device=h.device, dtype=h.dtype)
    for b, rep in enumerate(outs):
        padded[b, :rep.shape[0]] = rep
    return padded, torch.tensor(out_lens, device=h.device, dtype=torch.long)


FRAME_PHASE_DIM = 24


def frame_phase_and_dur(durations: torch.Tensor, max_frames: int):
    """用 searchsorted 以 O(B·F·log T) 零同步计算每帧在其所属音素内的序号 τ（0 起）与该音素时长 d，
    自动跳过 0 时长 token 并在有效帧之外返回 (0, 1)，与 expand_by_duration 网格严格对齐。"""
    B, T = durations.shape
    device = durations.device
    if T == 0 or max_frames == 0:
        return (
            torch.zeros(B, max_frames, device=device, dtype=torch.long),
            torch.ones(B, max_frames, device=device, dtype=torch.long),
        )
    d = durations.long().clamp(min=0)
    ends = torch.cumsum(d, dim=1)                                                     # (B, T) 单调非降
    starts = ends - d                                                                 # (B, T)
    idx = torch.arange(max_frames, device=device, dtype=torch.long).unsqueeze(0).expand(B, max_frames).contiguous()
    tok_idx = torch.searchsorted(ends, idx, right=True)                               # (B, F) 有效帧 < T，填充帧 == T
    valid = tok_idx < T
    tok_safe = tok_idx.clamp(max=T - 1)
    s_f = starts.gather(1, tok_safe)
    d_f = d.gather(1, tok_safe)
    tau = torch.where(valid, idx - s_f, torch.zeros_like(idx))
    dur_f = torch.where(valid, d_f.clamp(min=1), torch.ones_like(idx))
    return tau, dur_f


def build_frame_phase_features(durations: torch.Tensor, max_frames: int) -> torch.Tensor:
    """构造 24 维音素内相位特征（仅依赖当前音素时长，零额外前瞻，天然适配变速推理 speed != 1.0）：
      - 相对进度 prog = (τ + 0.5) / d 的 4 阶 Fourier（8 维）：表征声调轮廓与双元音滑动进度
      - 已过帧数 τ 与剩余帧数 d-1-τ 的短尺度 Fourier（各 6 维，尺度 2/6/18 帧）：表征辅音起始与尾音释放
      - 归一化标量 [τ/32, rem/32, prog, log1p(d)/4]（4 维）"""
    tau, dur_f = frame_phase_and_dur(durations, max_frames)
    tau_f = tau.to(torch.float32)
    dur_ff = dur_f.to(torch.float32)
    rem_f = (dur_ff - 1.0 - tau_f).clamp(min=0.0)
    prog = (tau_f + 0.5) / dur_ff

    prog_angles = prog.unsqueeze(-1) * torch.tensor([2.0 * math.pi * k for k in (1, 2, 4, 8)], device=durations.device, dtype=torch.float32)
    inv_scales = torch.tensor([1.0 / 2.0, 1.0 / 6.0, 1.0 / 18.0], device=durations.device, dtype=torch.float32)
    tau_angles = tau_f.unsqueeze(-1) * inv_scales
    rem_angles = rem_f.unsqueeze(-1) * inv_scales
    scalars = torch.stack([
        (tau_f / 32.0).clamp(max=2.0),
        (rem_f / 32.0).clamp(max=2.0),
        prog,
        torch.log1p(dur_ff) / 4.0,
    ], dim=-1)
    return torch.cat([
        torch.sin(prog_angles), torch.cos(prog_angles),
        torch.sin(tau_angles), torch.cos(tau_angles),
        torch.sin(rem_angles), torch.cos(rem_angles),
        scalars,
    ], dim=-1)


class CausalPredictorHead(nn.Module):
    def __init__(self, d_model: int = 256, kernel_size: int = 3):
        super().__init__()
        self.c1 = CausalConv1d(d_model, d_model, kernel_size)
        self.n1 = nn.LayerNorm(d_model)
        self.c2 = CausalConv1d(d_model, d_model, kernel_size)
        self.n2 = nn.LayerNorm(d_model)
        self.proj = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.silu(self.n1(self.c1(x.transpose(1, 2)).transpose(1, 2)))
        h = F.silu(self.n2(self.c2(h.transpose(1, 2)).transpose(1, 2)))
        return self.proj(h).squeeze(-1)

    def init_state(self, batch_size: int, device, dtype):
        return (self.c1.init_state(batch_size, device, dtype), self.c2.init_state(batch_size, device, dtype))

    def forward_step(self, x: torch.Tensor, state):
        if x.shape[1] == 0:
            return torch.zeros(x.shape[0], 0, device=x.device, dtype=x.dtype), state
        h, s1 = self.c1.forward_step(x.transpose(1, 2), state[0])
        h = F.silu(self.n1(h.transpose(1, 2)))
        h, s2 = self.c2.forward_step(h.transpose(1, 2), state[1])
        h = F.silu(self.n2(h.transpose(1, 2)))
        return self.proj(h).squeeze(-1), (s1, s2)


class AetherAcousticModel(nn.Module):
    def __init__(self, vocab_size: int, backbone_type: str = "chunk_attn", d_model: int = 256, n_mels: int = 100, k_left: int = 3, lookahead: int = 8, swa_window: int = 48, chunk_size: int = 4, frame_lookahead: int = 2):
        super().__init__()
        self.backbone_type = backbone_type
        self.d_model = d_model
        self.n_mels = n_mels
        self.lookahead = lookahead
        self.frame_lookahead = frame_lookahead
        self.phoneme_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.lang_emb = nn.Embedding(4, d_model)
        # 音素内多尺度相位投影（24 维 -> d_model，小初始化避免初态淹没 h_txt），作用见 expand_frames
        self.frame_phase_proj = nn.Linear(FRAME_PHASE_DIM, d_model)
        nn.init.normal_(self.frame_phase_proj.weight, std=0.02)
        nn.init.zeros_(self.frame_phase_proj.bias)

        if backbone_type == "offline_transformer":
            self.backbone = OfflineBiTransformerBackbone(d_model=d_model, n_layers=4, n_heads=4, d_ff=896)
        elif backbone_type == "conv_gru":
            self.backbone = ConvUniGRUBackbone(d_model=d_model, k_left=k_left, lookahead=lookahead, n_conv=4)
        elif backbone_type == "swa":
            self.backbone = SWABackbone(d_model=d_model, k_left=k_left, lookahead=lookahead, n_layers=5, n_heads=4, window_size=swa_window)
        elif backbone_type == "chunk_attn":
            self.backbone = ChunkLookaheadBackbone(d_model=d_model, chunk_size=chunk_size, lookahead=lookahead, n_layers=5, n_heads=4, window_size=swa_window)
        elif backbone_type == "gdn":
            self.backbone = GatedDeltaNetBackbone(d_model=d_model, k_left=k_left, lookahead=lookahead, n_layers=5, n_heads=4)
        else:
            raise ValueError(f"Unknown backbone_type: {backbone_type}")

        self.is_streaming = self.backbone.is_streaming
        self.aligner = AlignmentEncoder(n_mels=n_mels, d_text=d_model, d_attn=80)
        self.dur_predictor = CausalPredictorHead(d_model, kernel_size=3)
        nn.init.constant_(self.dur_predictor.proj.bias, math.log(9.0))
        self.f0_predictor = CausalPredictorHead(d_model, kernel_size=3)
        self.energy_predictor = CausalPredictorHead(d_model, kernel_size=3)
        self.f0_emb = nn.Linear(1, d_model)
        self.energy_emb = nn.Linear(1, d_model)
        # 集中式帧级右前瞻：仅 frame_decoder[0] 允许看右边 frame_lookahead 帧（默认 2 帧 ≈ 21 ms）改善元音->辅音协同发音，其余 3 层严格因果
        self.frame_decoder = nn.ModuleList(
            [CentralizedLookaheadBlock(d_model, k_left=4, lookahead=frame_lookahead, mult=2)]
            + [CausalConvNeXtBlock(d_model, kernel_size=5, mult=2) for _ in range(3)]
        )
        self.mel_out = nn.Linear(d_model, n_mels)

    def encode_text(self, phoneme_ids: torch.Tensor, lang_ids: torch.Tensor, text_mask: torch.Tensor = None):
        return self.backbone(self.embed_text(phoneme_ids, lang_ids, text_mask), mask=text_mask)

    def embed_text(self, phoneme_ids: torch.Tensor, lang_ids: torch.Tensor, text_mask: torch.Tensor = None):
        """音素 + 语种嵌入，右填充位置置零。lang_id=0 的嵌入非零，若不置零，填充位会经集中右前瞻层泄漏进句末 L 个位置的隐状态；
        而推理期（离线 F.pad / 流式 is_final 补零）看到的都是零，训练与推理不一致"""
        x_emb = self.phoneme_emb(phoneme_ids) + self.lang_emb(lang_ids)
        return x_emb if text_mask is None else x_emb * text_mask[:, :, None].to(x_emb.dtype)

    def _frame_phase_embed(self, durations: torch.Tensor, max_frames: int, dtype: torch.dtype) -> torch.Tensor:
        feats = build_frame_phase_features(durations, max_frames).to(dtype=self.frame_phase_proj.weight.dtype)
        return self.frame_phase_proj(feats).to(dtype)

    def expand_frames(self, h_txt: torch.Tensor, durations: torch.Tensor, max_frames: int = None):
        """帧级展开 + 注入音素内多尺度相位特征（已过帧数 τ、剩余帧数 d-1-τ、相对进度 (τ+0.5)/d）。
        F0/能量预测器与帧级解码器全是因果卷积，而按时长展开后的输入是分段常量：音素内部过了感受野输出必然冻结
        （实测 F0/能量 4 帧≈43 ms 后为常数，mel 20 帧≈213 ms 后冻结），无法表达声调轮廓、双元音滑动等音素内部动态。
        所有相位特征仅依赖当前音素自身时长 d，流式推理可逐音素实时计算、零额外前瞻，且适配变速推理。"""
        h_frame, out_lens = expand_by_duration(h_txt, durations, max_frames=max_frames)
        valid = torch.arange(h_frame.shape[1], device=h_frame.device)[None, :, None] < out_lens[:, None, None]
        h_frame = h_frame + self._frame_phase_embed(durations, h_frame.shape[1], h_frame.dtype)
        return h_frame * valid.to(h_frame.dtype), out_lens

    def forward_train(self, phoneme_ids, lang_ids, text_lens, mel_gt, f0_gt, energy_gt, mel_lens, step_ratio: float = 1.0):
        B, T_max = phoneme_ids.shape
        F_max = mel_gt.shape[-1]
        text_mask = torch.arange(T_max, device=phoneme_ids.device)[None, :] < text_lens[:, None]
        mel_mask = torch.arange(F_max, device=phoneme_ids.device)[None, :] < mel_lens[:, None]

        # Aligner 基于纯音素嵌入 + Beta-Binomial 对角先验，避免受流式右前瞻/因果偏移影响且严防 <bos> 坍缩
        x_emb = self.embed_text(phoneme_ids, lang_ids, text_mask)
        h_txt = self.backbone(x_emb, mask=text_mask)
        attn_log_prior = build_beta_binomial_log_prior(text_lens, mel_lens, T_max, F_max, scaling=1.0)
        attn_logprob, attn_soft = self.aligner(mel_gt, x_emb, text_mask, attn_log_prior=attn_log_prior)
        loss_align = forward_sum_loss(attn_logprob, text_lens, mel_lens)
        attn_hard, dur_gt = viterbi_monotonic_alignment(attn_logprob, text_lens, mel_lens)

        # 前 25% 步数用 soft attention 让 mel 重建梯度直接训练对齐器；之后直通硬对齐 + 二值化损失，实现训练↔推理零落差
        use_hard = (not self.training) or (step_ratio >= 0.25)
        attn_use = (attn_hard + (attn_soft - attn_soft.detach())) if use_hard else attn_soft
        h_frame = (torch.bmm(attn_use.to(h_txt.dtype), h_txt) + self._frame_phase_embed(dur_gt, F_max, h_txt.dtype)) * mel_mask[:, :, None].to(h_txt.dtype)

        loss_bin = -(attn_hard * attn_logprob * mel_mask[:, :, None]).sum() / mel_mask.sum().clamp(min=1)
        bin_weight = 0.0 if step_ratio < 0.25 else min(1.0, (step_ratio - 0.25) / 0.25)

        log_dur_pred = self.dur_predictor(h_txt)
        loss_dur = (F.l1_loss(log_dur_pred, torch.log(dur_gt.float() + 1.0), reduction="none") * text_mask).sum() / text_mask.sum().clamp(min=1)

        f0_pred = self.f0_predictor(h_frame)
        energy_pred = self.energy_predictor(h_frame)
        loss_f0 = (F.l1_loss(f0_pred, f0_gt, reduction="none") * mel_mask).sum() / mel_mask.sum().clamp(min=1)
        loss_energy = (F.l1_loss(energy_pred, energy_gt, reduction="none") * mel_mask).sum() / mel_mask.sum().clamp(min=1)

        # 验证期固定使用预测 F0/Energy（确定性衡量真实推理质量）；训练后期 50% 概率混合预测 F0/Energy 消除曝光偏差
        use_pred = (not self.training) or (step_ratio >= 0.25 and torch.rand(1).item() < 0.5)
        f0_in = f0_pred.detach() if use_pred else f0_gt
        e_in = energy_pred.detach() if use_pred else energy_gt
        # 右填充帧置零（防 f0_emb/energy_emb 偏置经 frame_decoder[0] 的集中右前瞻泄漏进有效句末帧，保证训练↔推理一致）
        h_dec = (h_frame + self.f0_emb(f0_in.unsqueeze(-1)) + self.energy_emb(e_in.unsqueeze(-1))) * mel_mask[:, :, None].to(h_frame.dtype)
        for blk in self.frame_decoder:
            h_dec = blk(h_dec)
        mel_pred = self.mel_out(h_dec).transpose(1, 2)

        loss_mel = (F.l1_loss(mel_pred, mel_gt, reduction="none") * mel_mask[:, None, :]).sum() / (mel_mask.sum().clamp(min=1) * self.n_mels)
        total_loss = loss_mel + 0.5 * loss_dur + 0.25 * loss_f0 + 0.25 * loss_energy + 0.5 * loss_align + 0.5 * bin_weight * loss_bin
        bos_ratio = (dur_gt[:, 0].float() / mel_lens.float().clamp(min=1)).mean().detach()
        return {
            "loss": total_loss, "mel": loss_mel.detach(), "dur": loss_dur.detach(),
            "f0": loss_f0.detach(), "energy": loss_energy.detach(),
            "align": loss_align.detach(), "bin": loss_bin.detach(), "bos_ratio": bos_ratio,
        }

    @torch.no_grad()
    def infer_offline(self, phoneme_ids: torch.Tensor, lang_ids: torch.Tensor, speed: float = 1.0):
        h_txt = self.encode_text(phoneme_ids, lang_ids)
        dur = torch.clamp(torch.round((torch.exp(self.dur_predictor(h_txt)) - 1.0) / speed), min=1, max=60).long()
        h_frame, _ = self.expand_frames(h_txt, dur)
        f0_pred = self.f0_predictor(h_frame)
        e_pred = self.energy_predictor(h_frame)
        h_dec = h_frame + self.f0_emb(f0_pred.unsqueeze(-1)) + self.energy_emb(e_pred.unsqueeze(-1))
        for blk in self.frame_decoder:
            h_dec = blk(h_dec)
        return self.mel_out(h_dec).transpose(1, 2), dur, f0_pred, e_pred

    def init_stream_state(self, batch_size: int = 1, device=None, dtype=torch.float32):
        assert self.is_streaming
        device = device or next(self.parameters()).device
        return {
            "backbone": self.backbone.init_state(batch_size, device, dtype),
            "dur": self.dur_predictor.init_state(batch_size, device, dtype),
            "f0": self.f0_predictor.init_state(batch_size, device, dtype),
            "energy": self.energy_predictor.init_state(batch_size, device, dtype),
            "frame_dec": [blk.init_state(batch_size, device, dtype) for blk in self.frame_decoder],
        }

    @torch.no_grad()
    def infer_stream_step(self, phoneme_chunk: torch.Tensor, lang_chunk: torch.Tensor, state: dict, is_final: bool = False, speed: float = 1.0):
        x = (self.phoneme_emb(phoneme_chunk) + self.lang_emb(lang_chunk)) if phoneme_chunk.shape[1] > 0 else torch.zeros(phoneme_chunk.shape[0], 0, self.d_model, device=phoneme_chunk.device, dtype=self.phoneme_emb.weight.dtype)
        h_emit, state["backbone"] = self.backbone.forward_step(x, state["backbone"], is_final=is_final)
        if h_emit.shape[1] == 0:
            if not is_final:
                return torch.zeros(x.shape[0], self.n_mels, 0, device=x.device, dtype=x.dtype), state
            h_dec = torch.zeros(x.shape[0], 0, self.d_model, device=x.device, dtype=x.dtype)
        else:
            log_dur, state["dur"] = self.dur_predictor.forward_step(h_emit, state["dur"])
            dur = torch.clamp(torch.round((torch.exp(log_dur) - 1.0) / speed), min=1, max=60).long()
            h_frame, _ = self.expand_frames(h_emit, dur)
            f0_pred, state["f0"] = self.f0_predictor.forward_step(h_frame, state["f0"])
            e_pred, state["energy"] = self.energy_predictor.forward_step(h_frame, state["energy"])
            h_dec = h_frame + self.f0_emb(f0_pred.unsqueeze(-1)) + self.energy_emb(e_pred.unsqueeze(-1))
        # frame_decoder[0] 带集中式帧级右前瞻，须透传 is_final 完成末尾 flush；其余 3 层严格因果
        h_dec, f0_st = self.frame_decoder[0].forward_step(h_dec, state["frame_dec"][0], is_final=is_final)
        new_fdec = [f0_st]
        for blk, fst in zip(self.frame_decoder[1:], state["frame_dec"][1:]):
            h_dec, nfst = blk.forward_step(h_dec, fst)
            new_fdec.append(nfst)
        state["frame_dec"] = new_fdec
        if h_dec.shape[1] == 0:
            return torch.zeros(x.shape[0], self.n_mels, 0, device=x.device, dtype=x.dtype), state
        return self.mel_out(h_dec).transpose(1, 2), state


# ==============================================================================
# 5.4 自研 24 kHz 因果流式声码器 CausalVocosLite (~3.6M) + 零爆音 OLA + 判别器
# ==============================================================================
class StreamingISTFTHead(nn.Module):
    def __init__(self, dim: int, n_fft: int = 1024, hop_length: int = 256):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.out = nn.Linear(dim, n_fft + 2)
        self.register_buffer("window", torch.hann_window(n_fft, periodic=True))

    def _frames_to_time(self, x: torch.Tensor) -> torch.Tensor:
        mag, p = self.out(x).float().chunk(2, dim=-1)
        mag = torch.exp(mag).clamp(max=1e2)
        spec = torch.complex(mag * torch.cos(p), mag * torch.sin(p))
        return torch.fft.irfft(spec, n=self.n_fft, dim=-1) * self.window[None, None, :]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, n_frames, _ = x.shape
        frames = self._frames_to_time(x)
        pad = self.n_fft - self.hop_length
        total_len = n_frames * self.hop_length + pad
        # 用 F.fold 一次性向量化 OLA（消除 n_frames 次 Python 循环，与流式 forward_step 逐样本等价）
        out = F.fold(
            frames.transpose(1, 2),
            output_size=(1, total_len),
            kernel_size=(1, self.n_fft),
            stride=(1, self.hop_length),
        ).reshape(B, total_len)
        w2 = (self.window ** 2).view(1, self.n_fft, 1).expand(1, self.n_fft, n_frames)
        env = F.fold(
            w2,
            output_size=(1, total_len),
            kernel_size=(1, self.n_fft),
            stride=(1, self.hop_length),
        ).reshape(1, total_len)
        # centre=True 约定（与 torch.istft 一致）：OLA 缓冲区下标 n_fft//2 对应采样时刻 0，输出与 mel 帧网格逐样本对齐。
        left_trim = self.n_fft // 2
        return out[:, left_trim : left_trim + n_frames * self.hop_length] / env[:, left_trim : left_trim + n_frames * self.hop_length].clamp(min=1e-6)

    def init_state(self, batch_size: int, device):
        pad = self.n_fft - self.hop_length
        return {
            "ola_out": torch.zeros(batch_size, pad, device=device, dtype=torch.float32),
            "ola_env": torch.zeros(batch_size, pad, device=device, dtype=torch.float32),
            "trim_remaining": self.n_fft // 2,
        }

    @torch.no_grad()
    def forward_step(self, x_chunk: torch.Tensor, state: dict, is_final: bool = False):
        B, C, _ = x_chunk.shape
        pad = self.n_fft - self.hop_length
        left_trim = self.n_fft // 2
        if C == 0 and not is_final:
            return torch.zeros(B, 0, device=x_chunk.device, dtype=torch.float32), state
        out_len = C * self.hop_length + pad
        out = torch.zeros(B, out_len, device=x_chunk.device, dtype=torch.float32)
        env = torch.zeros(B, out_len, device=x_chunk.device, dtype=torch.float32)
        out[:, :pad] = state["ola_out"]
        env[:, :pad] = state["ola_env"]
        if C > 0:
            frames = self._frames_to_time(x_chunk)
            w2 = (self.window ** 2)[None, :]
            for m in range(C):
                st = m * self.hop_length
                out[:, st : st + self.n_fft] += frames[:, m, :]
                env[:, st : st + self.n_fft] += w2
        emit_end = C * self.hop_length
        trim_rem = state["trim_remaining"]
        if is_final:
            tail_end = emit_end + left_trim
            wav_emit = torch.zeros(B, 0, device=x_chunk.device, dtype=torch.float32) if tail_end <= trim_rem else out[:, trim_rem:tail_end] / env[:, trim_rem:tail_end].clamp(min=1e-6)
            return wav_emit, self.init_state(B, x_chunk.device)
        if emit_end <= trim_rem:
            wav_emit = torch.zeros(B, 0, device=x_chunk.device, dtype=torch.float32)
            new_trim = trim_rem - emit_end
        else:
            wav_emit = out[:, trim_rem:emit_end] / env[:, trim_rem:emit_end].clamp(min=1e-6)
            new_trim = 0
        return wav_emit, {"ola_out": out[:, emit_end : emit_end + pad].clone(), "ola_env": env[:, emit_end : emit_end + pad].clone(), "trim_remaining": new_trim}


class CausalVocosLite(nn.Module):
    def __init__(self, n_mels: int = 100, dim: int = 256, n_layers: int = 8, kernel_size: int = 7, n_fft: int = 1024, hop_length: int = 256, lookahead: int = 2):
        super().__init__()
        # 集中式右前瞻：仅 embed 这一层允许看右边 lookahead 帧 mel，其后 8 个 ConvNeXt 块与 iSTFT 全部严格因果。
        # iSTFT 每帧合成窗以当前帧为中心，零前瞻等于让网络凭过去的 mel 预测未来半个窗，瞬态/辅音起始会糊
        self.embed = LookaheadConv1d(n_mels, dim, k_left=kernel_size - 1, lookahead=lookahead)
        self.norm_in = nn.LayerNorm(dim)
        self.blocks = nn.ModuleList([CausalConvNeXtBlock(dim, kernel_size=kernel_size, mult=3) for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(dim)
        self.head = StreamingISTFTHead(dim, n_fft=n_fft, hop_length=hop_length)

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        h = self.norm_in(self.embed(mel).transpose(1, 2))
        for blk in self.blocks:
            h = blk(h)
        return self.head(self.norm_out(h))

    def init_state(self, batch_size: int = 1, device=None, dtype=torch.float32):
        device = device or next(self.parameters()).device
        return {"embed": self.embed.init_state(batch_size, device, dtype), "blocks": [b.init_state(batch_size, device, dtype) for b in self.blocks], "head": self.head.init_state(batch_size, device)}

    @torch.no_grad()
    def forward_step(self, mel_chunk: torch.Tensor, state: dict, is_final: bool = False):
        # 空块不提前返回：embed 带右前瞻时需要攒够未来帧才吐帧，is_final 的空块也必须走到 embed 补零 flush
        h, state["embed"] = self.embed.forward_step(mel_chunk, state["embed"], is_final=is_final)
        h = self.norm_in(h.transpose(1, 2))
        new_blks = []
        for blk, bst in zip(self.blocks, state["blocks"]):
            h, nb = blk.forward_step(h, bst)
            new_blks.append(nb)
        state["blocks"] = new_blks
        wav, state["head"] = self.head.forward_step(self.norm_out(h), state["head"], is_final=is_final)
        return wav, state


def _apply_wn(module: nn.Module) -> nn.Module:
    wn = getattr(nn.utils.parametrizations, "weight_norm", None) or nn.utils.weight_norm
    return wn(module)


class PeriodDisc(nn.Module):
    def __init__(self, period: int):
        super().__init__()
        self.period = period
        self.convs = nn.ModuleList([
            _apply_wn(nn.Conv2d(1, 32, (5, 1), (3, 1), (2, 0))),
            _apply_wn(nn.Conv2d(32, 64, (5, 1), (3, 1), (2, 0))),
            _apply_wn(nn.Conv2d(64, 128, (5, 1), (3, 1), (2, 0))),
            _apply_wn(nn.Conv2d(128, 256, (5, 1), (3, 1), (2, 0))),
            _apply_wn(nn.Conv2d(256, 1, (3, 1), 1, (1, 0))),
        ])

    def forward(self, x: torch.Tensor):
        B, T = x.shape
        if T % self.period != 0:
            x = F.pad(x, (0, self.period - T % self.period), mode="reflect")
        x = x.view(B, 1, -1, self.period)
        fmap = []
        for i, conv in enumerate(self.convs):
            x = conv(x)
            if i < len(self.convs) - 1:
                x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        return x.flatten(1, -1), fmap


class SpecDisc(nn.Module):
    def __init__(self, n_fft: int, hop: int):
        super().__init__()
        self.n_fft, self.hop = n_fft, hop
        self.register_buffer("win", torch.hann_window(n_fft))
        self.convs = nn.ModuleList([
            _apply_wn(nn.Conv2d(1, 32, (3, 9), (1, 1), (1, 4))),
            _apply_wn(nn.Conv2d(32, 64, (3, 9), (1, 2), (1, 4))),
            _apply_wn(nn.Conv2d(64, 128, (3, 9), (1, 2), (1, 4))),
            _apply_wn(nn.Conv2d(128, 128, (3, 3), (1, 1), (1, 1))),
            _apply_wn(nn.Conv2d(128, 1, (3, 3), (1, 1), (1, 1))),
        ])

    def forward(self, x: torch.Tensor):
        h = torch.stft(x.float(), self.n_fft, self.hop, self.n_fft, self.win, return_complex=True).abs().unsqueeze(1)
        fmap = []
        for i, conv in enumerate(self.convs):
            h = conv(h)
            if i < len(self.convs) - 1:
                h = F.leaky_relu(h, 0.1)
            fmap.append(h)
        return h.flatten(1, -1), fmap


class LiteVocoderDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.discs = nn.ModuleList([
            PeriodDisc(2), PeriodDisc(3), PeriodDisc(5), PeriodDisc(7),
            SpecDisc(512, 128), SpecDisc(1024, 256), SpecDisc(2048, 512),
        ])

    def forward(self, y_real: torch.Tensor, y_fake: torch.Tensor):
        ro_l, fo_l, rf_l, ff_l = [], [], [], []
        for d in self.discs:
            ro, rf = d(y_real); fo, ff = d(y_fake)
            ro_l.append(ro); fo_l.append(fo); rf_l.append(rf); ff_l.append(ff)
        return ro_l, fo_l, rf_l, ff_l


def multi_res_stft_loss(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    loss = 0.0
    for n_fft, hop in [(512, 128), (1024, 256), (2048, 512)]:
        win = torch.hann_window(n_fft, device=y_pred.device)
        sp = torch.stft(y_pred.float(), n_fft, hop, n_fft, win, return_complex=True).abs().clamp(min=1e-5)
        st = torch.stft(y_true.float(), n_fft, hop, n_fft, win, return_complex=True).abs().clamp(min=1e-5)
        loss = loss + torch.norm(st - sp, p="fro") / (torch.norm(st, p="fro") + 1e-6) + F.l1_loss(torch.log(sp), torch.log(st))
    return loss / 3.0


def count_params_m(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6


## 6. 训练前硬门禁：参数量检查 & 流式↔离线等价单元测试


In [ ]:
def run_chunkwise_numerical_and_grad_tests():
    """验证门控 DeltaNet (E4) 在多块 (T=75 > 2*SCAN_CHUNK) + 强遗忘条件下的前向等价性与反向梯度有限性（防 0/0 与 inf*0 NaN）。"""
    torch.manual_seed(0)
    B, T, H, D = 2, 75, 4, 16
    q = F.normalize(torch.randn(B, T, H, D), dim=-1).detach().requires_grad_(True)
    kd = F.normalize(torch.randn(B, T, H, D), dim=-1).detach().requires_grad_(True)
    vd = torch.randn(B, T, H, D, requires_grad=True)
    beta = torch.rand(B, T, H).clamp(0.05, 0.95).detach().requires_grad_(True)
    log_a = (-torch.empty(B, T, H).uniform_(5.0, 30.0)).detach().requires_grad_(True)
    Sd0 = torch.randn(B, H, D, D, requires_grad=True)
    yo_chk, Sd_chk = gated_delta_rule_chunkwise(q, kd, vd, beta, log_a, Sd0)
    assert torch.isfinite(yo_chk).all() and torch.isfinite(Sd_chk).all(), "GDN chunkwise non-finite output under strong decay"
    (yo_chk.pow(2).mean() + Sd_chk.pow(2).mean()).backward()
    for name, t in (("q", q), ("k", kd), ("v", vd), ("beta", beta), ("log_alpha", log_a), ("S0", Sd0)):
        assert torch.isfinite(t.grad).all(), f"GDN gated_delta_rule_chunkwise non-finite grad on {name}"
    with torch.no_grad():
        yo_rec, Sd_rec = gated_delta_rule_recurrent(q, kd, vd, beta, log_a, Sd0.clone())
        assert (yo_chk.detach() - yo_rec).abs().max().item() < 1e-5, "GDN chunkwise fwd mismatch under strong decay"
        assert (Sd_chk.detach() - Sd_rec).abs().max().item() < 1e-5, "GDN chunkwise state mismatch under strong decay"
    print(f"  [PASS] {'GDN Log-Space Chunk Scan':26s} | T={T} (>2×{SCAN_CHUNK}) | 强遗忘前向等价 & 反向全有限")


def run_chunk_attn_parallel_and_grad_tests():
    """验证 ChunkLookaheadBackbone 零 chunk 循环并行 forward 与逐块串行 _forward_sequential 的前向与反向梯度等价性。"""
    torch.manual_seed(7)
    bb = ChunkLookaheadBackbone(d_model=64, chunk_size=4, lookahead=8, n_layers=5, n_heads=4, window_size=16)
    T_lens = [35, 19, 3]  # 覆盖滑窗截断 (>W)、非整除长度、以及 T < C 极端短句
    B, T, D = len(T_lens), max(T_lens), 64
    mask = torch.arange(T)[None, :] < torch.tensor(T_lens)[:, None]
    x_base = torch.randn(B, T, D) * mask[:, :, None].float()

    # 1) 变长批前向 + 反向梯度等价
    x_par = x_base.clone().requires_grad_(True)
    y_par = bb.forward(x_par, mask=mask)
    loss_par = (y_par.pow(2) * mask[:, :, None].float()).sum()
    loss_par.backward()
    grads_par = {n: p.grad.clone() for n, p in bb.named_parameters()}
    xgrad_par = x_par.grad.clone()

    bb.zero_grad(set_to_none=True)
    x_seq = x_base.clone().requires_grad_(True)
    y_seq = bb._forward_sequential(x_seq, mask=mask)
    loss_seq = (y_seq.pow(2) * mask[:, :, None].float()).sum()
    loss_seq.backward()

    fwd_diff = ((y_par.detach() - y_seq.detach()) * mask[:, :, None].float()).abs().max().item()
    xgrad_diff = ((xgrad_par - x_seq.grad) * mask[:, :, None].float()).abs().max().item()
    pgrad_diff = max((grads_par[n] - p.grad).abs().max().item() for n, p in bb.named_parameters())
    assert torch.isfinite(y_par).all() and torch.isfinite(xgrad_par).all(), "ChunkAttn parallel non-finite output/grad"
    assert fwd_diff < 1e-5, f"ChunkAttn parallel fwd mismatch: {fwd_diff}"
    assert xgrad_diff < 1e-4 and pgrad_diff < 1e-4, f"ChunkAttn parallel grad mismatch: x={xgrad_diff}, p={pgrad_diff}"

    # 2) 无 mask 单句路径等价
    with torch.no_grad():
        x_single = torch.randn(1, 35, D)
        diff_nomask = (bb.forward(x_single) - bb._forward_sequential(x_single)).abs().max().item()
        assert diff_nomask < 1e-5, f"ChunkAttn no-mask fwd mismatch: {diff_nomask}"
    print(f"  [PASS] {'ChunkAttn Parallel/Grad':26s} | Fwd: {fwd_diff:.2e} | xGrad: {xgrad_diff:.2e} | pGrad: {pgrad_diff:.2e}")


def run_architecture_and_streaming_tests():
    print("=" * 72)
    print("参数量检查 & 流式↔离线等价 + 零前瞻膨胀 + 强遗忘梯度硬门禁 (误差 < 1e-4)")
    print("=" * 72)
    run_chunkwise_numerical_and_grad_tests()
    run_chunk_attn_parallel_and_grad_tests()

    with torch.no_grad():
        T = 75  # > 2 * SCAN_CHUNK (32)，覆盖多块递推、滑窗截断与非整除尾块
        pids = torch.randint(4, FRONTEND.vocab_size, (1, T), dtype=torch.long)
        lids = torch.randint(0, 3, (1, T), dtype=torch.long)

        for exp_name, bb_type, lookahead in EXPERIMENTS:
            m = AetherAcousticModel(
                vocab_size=FRONTEND.vocab_size, backbone_type=bb_type, d_model=CFG["d_model"],
                n_mels=CFG["n_mels"], k_left=CFG["k_left"], lookahead=lookahead,
                swa_window=CFG["swa_window"], chunk_size=CFG["chunk_attn_size"],
                frame_lookahead=CFG["frame_lookahead"],
            ).eval()
            infer_params = sum(p.numel() for n, p in m.named_parameters() if not n.startswith("aligner.")) / 1e6
            mel_off, _, _, _ = m.infer_offline(pids, lids)
            if m.is_streaming:
                def _run_stream(step: int, empty_final: bool = False):
                    st = m.init_stream_state(1, device=pids.device, dtype=torch.float32)
                    chunks = []
                    for i in range(0, T, step):
                        mc, st = m.infer_stream_step(pids[:, i:i+step], lids[:, i:i+step], st, is_final=(i + step >= T and not empty_final))
                        if mc.shape[-1] > 0:
                            chunks.append(mc)
                    if empty_final:
                        mc, st = m.infer_stream_step(pids[:, :0], lids[:, :0], st, is_final=True)
                        if mc.shape[-1] > 0:
                            chunks.append(mc)
                    return torch.cat(chunks, dim=-1)

                mel_str = _run_stream(CFG["stream_text_chunk"], False)
                diff = (mel_off - mel_str).abs().max().item()
                assert mel_str.shape == mel_off.shape and diff < 1e-4, f"{exp_name} streaming mismatch: {diff}"
                # 对分块深前瞻额外测 step=1 + 空块收尾，覆盖非整除 chunk 与 frame_decoder 空块 flush
                if bb_type == "chunk_attn":
                    mel_s1 = _run_stream(1, True)
                    diff = max(diff, (mel_off - mel_s1).abs().max().item())
                    assert mel_s1.shape == mel_off.shape and diff < 1e-4, f"{exp_name} step=1 streaming mismatch: {diff}"

                # 零前瞻膨胀扰动测试：改写超出允许前瞻窗口的未来音素，验证已提交位置隐状态变化严格为 0
                c_probe = 12  # 对齐 C=4 的第 3 块右边界
                cutoff = c_probe + lookahead  # 严格禁止依赖 index >= cutoff 的任何信息
                pids_pert = pids.clone()
                pids_pert[:, cutoff:] = torch.randint(4, FRONTEND.vocab_size, (1, T - cutoff), dtype=torch.long)
                h_ref = m.encode_text(pids, lids)
                h_pert = m.encode_text(pids_pert, lids)
                leak = (h_ref[:, :c_probe] - h_pert[:, :c_probe]).abs().max().item()
                assert leak == 0.0, f"{exp_name} LOOKAHEAD EXPLOSION detected! leak={leak}"

                # 验证批训练右填充等价（B=2 不同长度 vs B=1 无填充，确保右填充不污染有效句末前瞻）
                T_short = 31
                pids_b = torch.zeros(2, T, dtype=torch.long)
                lids_b = torch.zeros(2, T, dtype=torch.long)
                pids_b[0], lids_b[0] = pids[0], lids[0]
                pids_b[1, :T_short], lids_b[1, :T_short] = pids[0, :T_short], lids[0, :T_short]
                tmask_b = torch.arange(T)[None, :] < torch.tensor([T, T_short])[:, None]
                h_batched = m.encode_text(pids_b, lids_b, text_mask=tmask_b)
                h_single = m.encode_text(pids[:, :T_short], lids[:, :T_short])
                pad_diff = (h_batched[1, :T_short] - h_single[0]).abs().max().item()
                assert pad_diff < 1e-4, f"{exp_name} batched padding mismatch: {pad_diff}"

                if bb_type == "chunk_attn":
                    # 校验 Phase 2 流式前端原生接口 forward_committed_with_lookahead
                    x_all = m.embed_text(pids, lids)
                    st_c = m.backbone.init_state(1, x_all.device, x_all.dtype)
                    out_c = []
                    C_sz = CFG["chunk_attn_size"]
                    for s in range(0, T, C_sz):
                        xc = x_all[:, s : min(T, s + C_sz)]
                        xl = x_all[:, min(T, s + C_sz) : min(T, s + C_sz + lookahead)]
                        hc, st_c = m.backbone.forward_committed_with_lookahead(xc, xl, st_c)
                        out_c.append(hc)
                    api_diff = (h_ref - torch.cat(out_c, dim=1)).abs().max().item()
                    assert api_diff < 1e-5, f"{exp_name} forward_committed_with_lookahead mismatch: {api_diff}"

                print(f"  [PASS] {exp_name:26s} | Infer: {infer_params:.2f}M | L={lookahead} fL={CFG['frame_lookahead']} | Stream: {diff:.2e} | Leak: 0.0")
            else:
                print(f"  [PASS] {exp_name:26s} | Infer: {infer_params:.2f}M | (Offline Upper-Bound)")

        # LookaheadConv1d 零前瞻零左上下文边界检查（防 buf[:, :, -0:] 全量切片回归）
        la0 = LookaheadConv1d(8, 8, k_left=0, lookahead=0).eval()
        x0 = torch.randn(1, 8, 7)
        y0_off = la0(x0)
        st0 = la0.init_state(1, x0.device, x0.dtype)
        ys0 = []
        for i in range(7):
            yi, st0 = la0.forward_step(x0[:, :, i:i+1], st0, is_final=(i == 6))
            ys0.append(yi)
        assert (y0_off - torch.cat(ys0, dim=-1)).abs().max().item() < 1e-5, "LookaheadConv1d(0,0) mismatch"

        voc = CausalVocosLite(n_mels=CFG["n_mels"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"], lookahead=CFG["vocoder_lookahead"]).eval()
        mel_dummy = torch.randn(1, CFG["n_mels"], 53)
        wav_off = voc(mel_dummy)

        def _stream_voc_diff(m_step: int, empty_final: bool):
            """逐块流式合成与离线对比。empty_final=True 时逐帧喂入并在末尾多送一个空块 is_final，
            覆盖「块长 <= 前瞻帧数」与「右前瞻攒帧后的收尾 flush」两条边界路径"""
            vst = voc.init_state(1, device=mel_dummy.device, dtype=torch.float32)
            w_chunks = []
            for i in range(0, 53, m_step):
                wc, vst = voc.forward_step(mel_dummy[:, :, i:i+m_step], vst, is_final=(i + m_step >= 53 and not empty_final))
                if wc.shape[-1] > 0:
                    w_chunks.append(wc)
            if empty_final:
                wc, vst = voc.forward_step(mel_dummy[:, :, :0], vst, is_final=True)
                if wc.shape[-1] > 0:
                    w_chunks.append(wc)
            wav_str = torch.cat(w_chunks, dim=-1)
            assert wav_str.shape == wav_off.shape, f"Vocoder stream shape {tuple(wav_str.shape)} != offline {tuple(wav_off.shape)}"
            return (wav_off - wav_str).abs().max().item()

        vdiff = max(_stream_voc_diff(CFG["stream_mel_chunk"], False), _stream_voc_diff(1, True))
        assert vdiff < 1e-4, f"Vocoder OLA mismatch: {vdiff}"
        print(f"  [PASS] {'CausalVocosLite (24kHz)':26s} | Infer: {count_params_m(voc):.2f}M | Lookahead: {CFG['vocoder_lookahead']} 帧 | OLA Diff: {vdiff:.2e}")
    print("=" * 72)


run_architecture_and_streaming_tests()


## 7. DataLoader 构建


In [ ]:
from torch.utils.data import Dataset, DataLoader

STATS = DATASET_META["stats"]
MEL_MEAN = torch.tensor(STATS["mel_mean"], dtype=torch.float32)[:, None]
MEL_STD = torch.tensor(STATS["mel_std"], dtype=torch.float32)[:, None]


def norm_mel(m): return (m - MEL_MEAN.to(m.device)) / MEL_STD.to(m.device)
def denorm_mel(m): return m * MEL_STD.to(m.device) + MEL_MEAN.to(m.device)


class AcousticDataset(Dataset):
    def __init__(self, entries):
        self.entries = entries

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        d = torch.load(self.entries[idx]["feat_path"], map_location="cpu", weights_only=False)
        return {
            "phoneme_ids": d["phoneme_ids"],
            "lang_ids": d["lang_ids"],
            "mel": (d["mel"].float() - MEL_MEAN) / MEL_STD,
            "f0": (d["log_f0"].float() - STATS["f0_mean"]) / STATS["f0_std"],
            "energy": (d["log_energy"].float() - STATS["e_mean"]) / STATS["e_std"],
        }


def acoustic_collate(batch):
    B = len(batch)
    T_max = max(b["phoneme_ids"].shape[0] for b in batch)
    F_max = max(b["mel"].shape[-1] for b in batch)
    pids = torch.zeros(B, T_max, dtype=torch.long)
    lids = torch.zeros(B, T_max, dtype=torch.long)
    t_lens = torch.zeros(B, dtype=torch.long)
    mels = torch.zeros(B, CFG["n_mels"], F_max, dtype=torch.float32)
    f0s = torch.zeros(B, F_max, dtype=torch.float32)
    es = torch.zeros(B, F_max, dtype=torch.float32)
    m_lens = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t, f = b["phoneme_ids"].shape[0], b["mel"].shape[-1]
        pids[i, :t] = b["phoneme_ids"]; lids[i, :t] = b["lang_ids"]; t_lens[i] = t
        mels[i, :, :f] = b["mel"]; f0s[i, :f] = b["f0"]; es[i, :f] = b["energy"]; m_lens[i] = f
    return pids, lids, t_lens, mels, f0s, es, m_lens


class VocoderDataset(Dataset):
    def __init__(self, entries, seg_frames: int = 96):
        self.entries = entries
        self.seg_frames = seg_frames
        self.seg_samples = seg_frames * CFG["hop_length"]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        d = torch.load(self.entries[idx]["feat_path"], map_location="cpu", weights_only=False)
        wav = decode_wav_tensor(d["wav"])
        if wav.shape[0] < self.seg_samples:
            wav = F.pad(wav, (0, self.seg_samples - wav.shape[0]))
            st = 0
        else:
            st = random.randint(0, wav.shape[0] - self.seg_samples)
        wav_seg = wav[st : st + self.seg_samples]
        with torch.no_grad():
            mel_seg = MEL_EXTRACTOR(wav_seg)[:, :self.seg_frames]
        return mel_seg, wav_seg


class BucketBatchSampler:
    """按 mel 帧长加微扰分桶的批采样器：每轮减少 30-40% 右填充，同时加速 chunk_attn 与 CPU Viterbi 对齐。"""

    def __init__(self, entries, batch_size: int, drop_last: bool = True, seed: int = 42, jitter: float = 0.15):
        self.lengths = [int(e.get("n_frames", 1)) for e in entries]
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.seed = seed
        self.jitter = jitter
        self.epoch = 0

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        self.epoch += 1
        noisy = [(l * (1.0 + rng.uniform(-self.jitter, self.jitter)), i) for i, l in enumerate(self.lengths)]
        noisy.sort(key=lambda x: x[0])
        order = [i for _, i in noisy]
        batches = [order[k : k + self.batch_size] for k in range(0, len(order), self.batch_size)]
        if self.drop_last and batches and len(batches[-1]) < self.batch_size:
            batches.pop()
        rng.shuffle(batches)
        yield from batches

    def __len__(self):
        return len(self.lengths) // self.batch_size if self.drop_last else (len(self.lengths) + self.batch_size - 1) // self.batch_size


train_loader = DataLoader(
    AcousticDataset(DATASET_META["train"]),
    batch_sampler=BucketBatchSampler(DATASET_META["train"], batch_size=CFG["batch_size"], drop_last=True, seed=CFG["seed"]),
    num_workers=CFG["num_workers"], pin_memory=torch.cuda.is_available(),
    collate_fn=acoustic_collate,
)
val_loader = DataLoader(
    AcousticDataset(DATASET_META["val"]),
    batch_size=min(16, len(DATASET_META["val"])), shuffle=False,
    num_workers=min(2, CFG["num_workers"]), collate_fn=acoustic_collate,
)
print(f"Train batches/epoch: {len(train_loader)} | Val batches: {len(val_loader)}")


## 8. 训练共享因果声码器 `CausalVocosLite`（支持每 1500 步 / 8 分钟 + 中断紧急精确续训）


In [ ]:
VOC_TAG = f"causal_vocos_lite_{CFG['run_suffix']}_L{CFG['vocoder_lookahead']}"
VOC_STORAGE_DIR = STORAGE_ROOT / "vocoder" / VOC_TAG
VOC_LOCAL_DIR = LOCAL_WORK / "vocoder" / VOC_TAG
VOC_STORAGE_DIR.mkdir(parents=True, exist_ok=True)
VOC_LOCAL_DIR.mkdir(parents=True, exist_ok=True)


def train_or_load_causal_vocoder():
    voc = CausalVocosLite(n_mels=CFG["n_mels"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"], lookahead=CFG["vocoder_lookahead"]).to(DEVICE)
    disc = LiteVocoderDiscriminator().to(DEVICE)
    opt_g = torch.optim.AdamW(voc.parameters(), lr=CFG["lr_vocoder"], betas=(0.8, 0.99))
    opt_d = torch.optim.AdamW(disc.parameters(), lr=CFG["lr_vocoder"], betas=(0.8, 0.99))

    s1_steps = CFG["vocoder_stage1_steps"]
    s2_steps = CFG["vocoder_stage2_steps"]
    total_steps = s1_steps + s2_steps

    voc_warmup = min(500, max(100, total_steps // 20))
    def _voc_lr_lambda(s):
        if s < voc_warmup:
            return float(s + 1) / float(voc_warmup)
        prog = float(s - voc_warmup) / float(max(1, total_steps - voc_warmup))
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

    def _disc_lr_lambda(s):
        prog = float(s) / float(max(1, s2_steps))
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

    sched_g = torch.optim.lr_scheduler.LambdaLR(opt_g, _voc_lr_lambda)
    sched_d = torch.optim.lr_scheduler.LambdaLR(opt_d, _disc_lr_lambda)

    final_storage = VOC_STORAGE_DIR / "final.pt"
    latest_storage = VOC_STORAGE_DIR / "latest.pt"
    done_storage = VOC_STORAGE_DIR / ".done"
    latest_local = VOC_LOCAL_DIR / "latest.pt"
    final_local = VOC_LOCAL_DIR / "final.pt"
    done_local = VOC_LOCAL_DIR / ".done"

    if SKIP_IF_DONE and done_storage.exists() and final_storage.exists():
        print(f"[Resume Done] 直接加载已完结因果声码器: {final_storage}")
        voc.load_state_dict(torch.load(final_storage, map_location=DEVICE, weights_only=True))
        save_progress_status("vocoder", {"status": "done_cached", "step": total_steps, "total_steps": total_steps})
        return voc.eval()

    step = 0
    if latest_storage.exists():
        print(f"[Mid-Run Resume] 检测到未完结声码器断点，正在恢复: {latest_storage}")
        ckpt = torch.load(latest_storage, map_location=DEVICE, weights_only=False)
        voc.load_state_dict(ckpt["voc"])
        opt_g.load_state_dict(ckpt["opt_g"])
        try:
            disc.load_state_dict(ckpt["disc"])
            opt_d.load_state_dict(ckpt["opt_d"])
        except Exception as e:
            print(f"  [Warn] 判别器结构已升级（weight_norm），重新初始化判别器并保留生成器权重: {e}")
        step = int(ckpt["step"])
        if "sched_g" in ckpt:
            sched_g.load_state_dict(ckpt["sched_g"])
        else:
            sched_g.last_epoch = step
        if "sched_d" in ckpt:
            sched_d.load_state_dict(ckpt["sched_d"])
        else:
            sched_d.last_epoch = max(0, step - s1_steps)
        print(f"  -> 从 step={step}/{total_steps} 无缝继续训练！")

    voc_loader = DataLoader(
        VocoderDataset(DATASET_META["train"], seg_frames=96),
        batch_size=min(16, CFG["batch_size"]), shuffle=True,
        num_workers=CFG["num_workers"], pin_memory=torch.cuda.is_available(), drop_last=True,
    )
    mel_gpu = VocosCompatibleMel(CFG["sample_rate"], CFG["n_fft"], CFG["hop_length"], CFG["n_mels"]).to(DEVICE)

    t0 = time.time()
    last_backup_t = time.time()
    pbar = tqdm(total=total_steps, initial=step, desc="Train CausalVocosLite (Stage1->GAN)")

    def _backup_voc(cur_step: int, is_final: bool = False):
        nonlocal last_backup_t
        payload = {
            "step": cur_step, "voc": voc.state_dict(), "disc": disc.state_dict(),
            "opt_g": opt_g.state_dict(), "opt_d": opt_d.state_dict(),
            "sched_g": sched_g.state_dict(), "sched_d": sched_d.state_dict(),
        }
        atomic_torch_save(payload, latest_local)
        sync_file_to_storage(latest_local, latest_storage)
        if is_final:
            atomic_torch_save(voc.state_dict(), final_local)
            sync_file_to_storage(final_local, final_storage)
            atomic_write_text(f"completed_step={cur_step}\n", done_local)
            sync_file_to_storage(done_local, done_storage)
        save_progress_status("vocoder", {"status": "done" if is_final else "running", "step": cur_step, "total_steps": total_steps})
        last_backup_t = time.time()

    try:
        while step < total_steps:
            for mel_seg, wav_gt in voc_loader:
                if step >= total_steps:
                    break
                mel_seg = mel_seg.to(DEVICE, non_blocking=True)
                wav_gt = wav_gt.to(DEVICE, non_blocking=True)
                in_gan = step >= s1_steps

                wav_pred = voc(mel_seg)
                L_min = min(wav_pred.shape[-1], wav_gt.shape[-1])
                wav_pred, wav_gt = wav_pred[:, :L_min], wav_gt[:, :L_min]

                if in_gan:
                    opt_d.zero_grad(set_to_none=True)
                    ro, fo, _, _ = disc(wav_gt, wav_pred.detach())
                    # 按 HiFi-GAN / Vocos 标准跨判别器求和，避免被除小 7x/35x 后被频谱损失淹没
                    loss_d = sum(torch.mean((r - 1.0) ** 2) + torch.mean(f ** 2) for r, f in zip(ro, fo))
                    loss_d.backward()
                    torch.nn.utils.clip_grad_norm_(disc.parameters(), 1.0)
                    opt_d.step()
                    sched_d.step()

                opt_g.zero_grad(set_to_none=True)
                loss_stft = multi_res_stft_loss(wav_pred, wav_gt)
                loss_mel = F.l1_loss(mel_gpu(wav_pred), mel_gpu(wav_gt))
                if in_gan:
                    _, fo, rfmaps, ffmaps = disc(wav_gt, wav_pred)
                    loss_adv = sum(torch.mean((f - 1.0) ** 2) for f in fo)
                    loss_fm = sum(
                        F.l1_loss(ff, rf.detach())
                        for rf_l, ff_l in zip(rfmaps, ffmaps)
                        for rf, ff in zip(rf_l[:-1], ff_l[:-1])
                    )
                    loss_g = 2.0 * loss_stft + 30.0 * loss_mel + loss_adv + 2.0 * loss_fm
                else:
                    loss_g = 15.0 * (loss_stft + loss_mel)
                loss_g.backward()
                torch.nn.utils.clip_grad_norm_(voc.parameters(), 1.0)
                opt_g.step()
                sched_g.step()

                step += 1
                pbar.update(1)
                if step % 200 == 0 or step == total_steps:
                    pbar.set_postfix(stage="GAN" if in_gan else "SPEC", stft=f"{loss_stft.item():.3f}", mel=f"{loss_mel.item():.3f}")

                if _STOP_REQUESTED:
                    _backup_voc(step, is_final=False)
                    print(f"\n[Emergency Backup Saved] CausalVocosLite 已安全保存至 step={step} -> {latest_storage}")
                    raise SystemExit(0)

                need_backup = (step % VOCODER_BACKUP_EVERY_STEPS == 0) or ((time.time() - last_backup_t) >= BACKUP_EVERY_MINUTES * 60.0)
                if need_backup and step < total_steps:
                    _backup_voc(step, is_final=False)
    except KeyboardInterrupt:
        _backup_voc(step, is_final=False)
        print(f"\n[KeyboardInterrupt Backup] CausalVocosLite 已紧急备份至 step={step} -> {latest_storage}")
        raise
    finally:
        pbar.close()

    _backup_voc(step, is_final=True)
    print(f"[Done] CausalVocosLite 训练完成（本次耗时 {(time.time()-t0)/60:.1f} min），已保存至: {VOC_STORAGE_DIR}")
    return voc.eval()


CAUSAL_VOCODER = None
PRETRAINED_VOCOS = None

if CFG["vocoder_mode"] in ("both", "causal_vocos_lite"):
    CAUSAL_VOCODER = train_or_load_causal_vocoder()

if CFG["vocoder_mode"] in ("both", "pretrained_vocos"):
    try:
        from vocos import Vocos
        print("[Load] 加载预训练 charactr/vocos-mel-24khz 作为高质量声码器对照锚点...")
        PRETRAINED_VOCOS = Vocos.from_pretrained("charactr/vocos-mel-24khz").to(DEVICE).eval()
    except Exception as e:
        print(f"[Warn] 预训练 Vocos 加载失败 ({e})，将仅使用自研 CausalVocosLite。")


## 9. 批量训练声学主干（聚焦流式输入 6 槽矩阵，高频断点备份 + 里程碑快照 + 中断紧急保存 + 精确续训）


In [ ]:
TRAIN_HISTORIES = {}


def train_one_model(exp_name: str, backbone_type: str, lookahead: int):
    tag = f"{exp_name}_{CFG['acoustic_suffix']}"
    run_local = LOCAL_WORK / "runs" / tag
    run_storage = STORAGE_ROOT / "runs" / tag
    run_local.mkdir(parents=True, exist_ok=True)
    run_storage.mkdir(parents=True, exist_ok=True)
    (run_local / "previews").mkdir(parents=True, exist_ok=True)

    storage_best = run_storage / "best.pt"
    storage_latest = run_storage / "latest.pt"
    storage_hist = run_storage / "history.json"
    storage_done = run_storage / ".done"

    local_best = run_local / "best.pt"
    local_latest = run_local / "latest.pt"
    local_hist = run_local / "history.json"
    local_done = run_local / ".done"

    model = AetherAcousticModel(
        vocab_size=FRONTEND.vocab_size, backbone_type=backbone_type, d_model=CFG["d_model"],
        n_mels=CFG["n_mels"], k_left=CFG["k_left"], lookahead=lookahead,
        swa_window=CFG["swa_window"], chunk_size=CFG["chunk_attn_size"],
        frame_lookahead=CFG["frame_lookahead"],
    ).to(DEVICE)

    if SKIP_IF_DONE and storage_done.exists() and (storage_best.exists() or storage_latest.exists()):
        src_ckpt = storage_best if storage_best.exists() else storage_latest
        print(f"\n[Resume Done] {exp_name} 已存在完结标记 .done，直接加载最优权重: {src_ckpt}")
        ckpt = torch.load(src_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        TRAIN_HISTORIES[exp_name] = json.loads(storage_hist.read_text(encoding="utf-8")) if storage_hist.exists() else {"steps": [], "val_mel": []}
        save_progress_status(exp_name, {"status": "done_cached", "best_val_mel": ckpt.get("best_val_mel")})
        return model.eval()

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr_acoustic"], betas=(0.9, 0.98), weight_decay=1e-2)
    total_steps = CFG["acoustic_steps"]
    warmup = max(200, int(total_steps * 0.05))

    def lr_lambda(s):
        if s < warmup:
            return float(s + 1) / float(warmup)
        prog = float(s - warmup) / float(max(1, total_steps - warmup))
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * prog))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))

    history = {"steps": [], "train_loss": [], "val_mel": [], "val_dur": [], "elapsed_min": 0.0}
    best_val_mel = float("inf")
    step = 0
    prior_elapsed_min = 0.0

    # 中途精确断点续训：如果存在 latest.pt（但无 .done），无缝恢复完整优化器状态继续跑！
    if storage_latest.exists():
        print(f"\n[Mid-Run Resume] 发现 {exp_name} 未完结断点，正在从 {storage_latest} 精确续训...")
        ckpt = torch.load(storage_latest, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        if "scaler" in ckpt:
            scaler.load_state_dict(ckpt["scaler"])
        step = int(ckpt["step"])
        best_val_mel = float(ckpt.get("best_val_mel", float("inf")))
        history = ckpt.get("history", history)
        prior_elapsed_min = float(history.get("elapsed_min", 0.0))
        if storage_best.exists() and not _same_path(local_best, storage_best):
            shutil.copy2(storage_best, local_best)
        print(f"  -> 已恢复至 step={step}/{total_steps}, best_val_mel={best_val_mel:.4f}")
    else:
        print(f"\n{'='*72}")
        print(f"开始训练: {exp_name} (backbone={backbone_type}, L={lookahead}, params={count_params_m(model):.2f}M)")
        print(f"{'='*72}")

    t0 = time.time()
    last_backup_t = time.time()
    eval_every = max(300, total_steps // 10)
    preview_text = "你好，请问今天北京的天气怎么样？" if CFG["dataset_source"] != "ljspeech" else "Hello, how is the weather today?"
    pbar = tqdm(total=total_steps, initial=step, desc=f"Train {exp_name}")

    def _save_resumable_latest(cur_step: int):
        nonlocal last_backup_t
        history["elapsed_min"] = round(prior_elapsed_min + (time.time() - t0) / 60.0, 2)
        ckpt_obj = {
            "exp_name": exp_name, "backbone_type": backbone_type, "lookahead": lookahead,
            "step": cur_step, "best_val_mel": best_val_mel, "history": history,
            "cfg": CFG, "stats": STATS, "vocab": FRONTEND.vocab,
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
        }
        atomic_torch_save(ckpt_obj, local_latest)
        sync_file_to_storage(local_latest, storage_latest)
        atomic_write_text(json.dumps(history, indent=2), local_hist)
        sync_file_to_storage(local_hist, storage_hist)
        save_progress_status(exp_name, {
            "status": "running", "step": cur_step, "total_steps": total_steps,
            "best_val_mel": None if math.isinf(best_val_mel) else round(best_val_mel, 4),
            "elapsed_min": history["elapsed_min"],
        })
        last_backup_t = time.time()

    def _save_preview_wav(cur_step: int):
        voc = PRETRAINED_VOCOS if PRETRAINED_VOCOS is not None else CAUSAL_VOCODER
        if voc is None:
            return
        try:
            pids_p, lids_p = FRONTEND.text_to_ids(preview_text)
            with torch.no_grad():
                mel_n, _, _, _ = model.infer_offline(
                    torch.tensor([pids_p], dtype=torch.long, device=DEVICE),
                    torch.tensor([lids_p], dtype=torch.long, device=DEVICE),
                )
                mel_raw = denorm_mel(mel_n)
                w = voc.decode(mel_raw) if (voc is PRETRAINED_VOCOS) else voc(mel_raw)
                wav_np = np.clip(w.squeeze(0).cpu().float().numpy(), -1.0, 1.0)
            p_loc = run_local / "previews" / f"step_{cur_step:06d}.wav"
            sf.write(p_loc, wav_np, CFG["sample_rate"])
            sync_file_to_storage(p_loc, run_storage / "previews" / p_loc.name)
        except Exception:
            pass

    try:
        while step < total_steps:
            model.train()
            for pids, lids, t_lens, mels, f0s, es, m_lens in train_loader:
                if step >= total_steps:
                    break
                pids = pids.to(DEVICE, non_blocking=True)
                lids = lids.to(DEVICE, non_blocking=True)
                t_lens = t_lens.to(DEVICE, non_blocking=True)
                mels = mels.to(DEVICE, non_blocking=True)
                f0s = f0s.to(DEVICE, non_blocking=True)
                es = es.to(DEVICE, non_blocking=True)
                m_lens = m_lens.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=(DEVICE.type == "cuda")):
                    out = model.forward_train(pids, lids, t_lens, mels, f0s, es, m_lens, step_ratio=step / max(total_steps, 1))
                    loss = out["loss"]

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

                step += 1
                pbar.update(1)
                if step % 100 == 0:
                    pbar.set_postfix(
                        loss=f"{loss.item():.3f}", mel=f"{out['mel'].item():.3f}",
                        dur=f"{out['dur'].item():.3f}", aln=f"{out['align'].item():.3f}",
                        bin=f"{out['bin'].item():.3f}", bos=f"{out['bos_ratio'].item():.2f}",
                    )

                if _STOP_REQUESTED:
                    _save_resumable_latest(step)
                    print(f"\n[Emergency Backup Saved] {exp_name} 已安全保存至 step={step} -> {storage_latest}")
                    raise SystemExit(0)

                # 定期验证 + 保存 best.pt + 导出训练中试听预览
                if step % eval_every == 0 or step == total_steps:
                    model.eval()
                    v_mel, v_dur, v_bos, v_cnt = 0.0, 0.0, 0.0, 0
                    with torch.no_grad():
                        for vp, vl, vtl, vm, vf0, ve, vml in val_loader:
                            vo = model.forward_train(
                                vp.to(DEVICE), vl.to(DEVICE), vtl.to(DEVICE),
                                vm.to(DEVICE), vf0.to(DEVICE), ve.to(DEVICE), vml.to(DEVICE),
                                step_ratio=1.0,
                            )
                            v_mel += vo["mel"].item(); v_dur += vo["dur"].item(); v_bos += vo["bos_ratio"].item(); v_cnt += 1
                    v_mel /= max(v_cnt, 1); v_dur /= max(v_cnt, 1); v_bos /= max(v_cnt, 1)
                    history["steps"].append(step)
                    history["train_loss"].append(round(loss.item(), 4))
                    history["val_mel"].append(round(v_mel, 4))
                    history["val_dur"].append(round(v_dur, 4))
                    history.setdefault("val_bos_ratio", []).append(round(v_bos, 3))

                    if not math.isnan(v_mel) and v_mel < best_val_mel:
                        best_val_mel = v_mel
                        best_payload = {
                            "exp_name": exp_name, "backbone_type": backbone_type, "lookahead": lookahead,
                            "step": step, "best_val_mel": best_val_mel, "cfg": CFG, "stats": STATS,
                            "vocab": FRONTEND.vocab, "model": model.state_dict(),
                        }
                        atomic_torch_save(best_payload, local_best)
                        sync_file_to_storage(local_best, storage_best)
                    _save_preview_wav(step)
                    _save_resumable_latest(step)
                    model.train()

                # 额外高频备份触发（按步数或按时间）
                elif (step % BACKUP_EVERY_STEPS == 0) or ((time.time() - last_backup_t) >= BACKUP_EVERY_MINUTES * 60.0):
                    _save_resumable_latest(step)

                # 不可覆盖里程碑权重快照
                if SAVE_MILESTONE_EVERY > 0 and (step % SAVE_MILESTONE_EVERY == 0) and step < total_steps:
                    ms_local = run_local / f"milestone_step_{step:06d}.pt"
                    atomic_torch_save({"step": step, "model": model.state_dict()}, ms_local)
                    sync_file_to_storage(ms_local, run_storage / ms_local.name)
    except KeyboardInterrupt:
        _save_resumable_latest(step)
        print(f"\n[KeyboardInterrupt Backup] {exp_name} 已紧急备份至 step={step} -> {storage_latest}")
        raise
    finally:
        pbar.close()

    # 完结后加载 best.pt（若不存在则回退 latest.pt）并导出动态 INT8 包，原子写入 .done 标记
    best_src = next((p for p in (local_best, storage_best, local_latest, storage_latest) if p.exists()), None)
    if best_src is not None:
        ckpt = torch.load(best_src, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model"])
        if not storage_best.exists():
            best_payload = {
                "exp_name": exp_name, "backbone_type": backbone_type, "lookahead": lookahead,
                "step": step, "best_val_mel": best_val_mel, "cfg": CFG, "stats": STATS,
                "vocab": FRONTEND.vocab, "model": model.state_dict(),
            }
            atomic_torch_save(best_payload, local_best)
            sync_file_to_storage(local_best, storage_best)

    model_cpu = model.to("cpu").eval()
    try:
        q_model = torch.ao.quantization.quantize_dynamic(model_cpu, {nn.Linear, nn.GRU}, dtype=torch.qint8)
        local_int8 = run_local / "model_int8.pt"
        atomic_torch_save(q_model.state_dict(), local_int8)
        sync_file_to_storage(local_int8, run_storage / "model_int8.pt")
    except Exception as e:
        print(f"[Warn] INT8 dynamic export skipped for {exp_name}: {e}")

    atomic_write_text(f"completed_step={step}\nbest_val_mel={best_val_mel:.4f}\n", local_done)
    sync_file_to_storage(local_done, storage_done)
    save_progress_status(exp_name, {
        "status": "done", "step": step, "total_steps": total_steps,
        "best_val_mel": None if math.isinf(best_val_mel) else round(best_val_mel, 4),
        "elapsed_min": history["elapsed_min"],
    })
    model = model.to(DEVICE).eval()
    TRAIN_HISTORIES[exp_name] = history
    print(f"[Done] {exp_name} | Best Val Mel L1: {best_val_mel:.4f} | Total Time: {history['elapsed_min']} min | 保存位置: {run_storage}")
    return model


TRAINED_MODELS = {}
for exp_name, bb_type, la in EXPERIMENTS:
    TRAINED_MODELS[exp_name] = train_one_model(exp_name, bb_type, la)


## 10. 批量生成测试音频（离线 + 真流式逐块合成）并归档保存


In [ ]:
GOLDEN_ZH = [
    ("zh_qa_01", "你好，请问今天北京的天气怎么样？"),
    ("zh_qa_02", "好的，已经帮你把明早七点的闹钟设置好了。"),
    ("zh_poly_01", "中国人民银行行长今天上午发表了重要讲话。"),
    ("zh_poly_02", "这件事情非常重，我们不要重复同样的错误。"),
    ("zh_sandhi_01", "你好，我想去展览馆买把好雨伞。"),
    ("zh_tn_01", "今年是2026年，同比增长了18.5%。"),
    ("zh_short_01", "好的，没问题。"),
    ("zh_short_02", "收到，马上处理。"),
    ("zh_long_01", "在嵌入式设备上实现端侧语音合成时，需要在自然度、首包延迟和纯中央处理器算力占用之间找到一个非常精准的平衡点。"),
    ("zh_cs_01", "帮我把 Wi-Fi 切换到 5G，顺便查一下今天的 API latency。"),
]
GOLDEN_EN = [
    ("en_qa_01", "Hello, how is the weather in San Francisco today?"),
    ("en_qa_02", "Sure, I have set your alarm for seven thirty tomorrow morning."),
    ("en_short_01", "Got it, no problem."),
    ("en_short_02", "Sure thing."),
    ("en_tech_01", "Streaming text to speech requires low latency and consistent speaker identity."),
    ("en_long_01", "Running neural speech synthesis directly on an embedded Linux processor eliminates cloud round trips while protecting user privacy."),
]


def build_eval_suite():
    suite = []
    for e in DATASET_META["val"][: CFG["eval_max_utts"]]:
        suite.append({"utt_id": f"val_{e['utt_id']}", "text": e["text"], "feat_path": e["feat_path"], "is_val": True})
    extras = GOLDEN_ZH if CFG["dataset_source"] == "bznsyp" else (GOLDEN_EN if CFG["dataset_source"] == "ljspeech" else GOLDEN_ZH + GOLDEN_EN)
    for uid, txt in extras:
        suite.append({"utt_id": uid, "text": txt, "feat_path": None, "is_val": False})
    return suite


EVAL_SUITE = build_eval_suite()
print(f"评测集构建完成：共 {len(EVAL_SUITE)} 句（含真声参考 Val: {sum(x['is_val'] for x in EVAL_SUITE)} 句）")


@torch.no_grad()
def synthesize_offline(model: AetherAcousticModel, text: str, vocoder, use_pretrained_vocos: bool = False):
    pids, lids = FRONTEND.text_to_ids(text)
    mel_norm, _, _, _ = model.infer_offline(
        torch.tensor([pids], dtype=torch.long, device=DEVICE),
        torch.tensor([lids], dtype=torch.long, device=DEVICE),
    )
    mel = denorm_mel(mel_norm)
    wav = PRETRAINED_VOCOS.decode(mel) if (use_pretrained_vocos and PRETRAINED_VOCOS is not None) else vocoder(mel)
    return wav.squeeze(0).cpu().float().numpy(), mel.squeeze(0).cpu().float().numpy()


@torch.no_grad()
def synthesize_streaming(model: AetherAcousticModel, text: str, vocoder: CausalVocosLite):
    pids, lids = FRONTEND.text_to_ids(text)
    pt = torch.tensor([pids], dtype=torch.long, device=DEVICE)
    lt = torch.tensor([lids], dtype=torch.long, device=DEVICE)
    T = pt.shape[1]
    ast_state = model.init_stream_state(1, device=DEVICE, dtype=torch.float32)
    voc_state = vocoder.init_state(1, device=DEVICE, dtype=torch.float32)
    mel_buf = torch.zeros(1, CFG["n_mels"], 0, device=DEVICE, dtype=torch.float32)
    wav_chunks, mel_chunks, seam_positions = [], [], []
    cur_samples = 0
    t_step, m_step = CFG["stream_text_chunk"], CFG["stream_mel_chunk"]

    for i in range(0, T, t_step):
        is_last = (i + t_step >= T)
        mc, ast_state = model.infer_stream_step(pt[:, i:i+t_step], lt[:, i:i+t_step], ast_state, is_final=is_last)
        if mc.shape[-1] > 0:
            mc_d = denorm_mel(mc)
            mel_chunks.append(mc_d)
            mel_buf = torch.cat([mel_buf, mc_d], dim=-1)
        while mel_buf.shape[-1] >= m_step:
            wc, voc_state = vocoder.forward_step(mel_buf[:, :, :m_step], voc_state, is_final=False)
            mel_buf = mel_buf[:, :, m_step:]
            if wc.shape[-1] > 0:
                if cur_samples > 0:
                    seam_positions.append(cur_samples)
                wav_chunks.append(wc.squeeze(0).cpu().float().numpy())
                cur_samples += wc.shape[-1]
        if is_last:
            wc, voc_state = vocoder.forward_step(mel_buf, voc_state, is_final=True)
            if wc.shape[-1] > 0:
                if cur_samples > 0:
                    seam_positions.append(cur_samples)
                wav_chunks.append(wc.squeeze(0).cpu().float().numpy())
                cur_samples += wc.shape[-1]

    wav = np.concatenate(wav_chunks) if wav_chunks else np.zeros(2400, dtype=np.float32)
    mel_all = torch.cat(mel_chunks, dim=-1).squeeze(0).cpu().float().numpy() if mel_chunks else np.zeros((CFG["n_mels"], 1))
    return wav, mel_all, seam_positions


EVAL_LOCAL_DIR = LOCAL_WORK / "eval_samples" / f"{CFG['eval_suffix']}"
EVAL_STORAGE_DIR = STORAGE_ROOT / "eval_samples" / f"{CFG['eval_suffix']}"
EVAL_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

gt_dir = EVAL_LOCAL_DIR / "00_ground_truth"
gt_dir.mkdir(parents=True, exist_ok=True)
for item in EVAL_SUITE:
    if item["feat_path"] is not None:
        d = torch.load(item["feat_path"], map_location="cpu", weights_only=False)
        sf.write(gt_dir / f"{item['utt_id']}.wav", decode_wav_tensor(d["wav"]).numpy(), CFG["sample_rate"])
sync_dir_to_storage(gt_dir, EVAL_STORAGE_DIR / "00_ground_truth")

GENERATED_RECORDS = {}
use_pre_for_offline = (CAUSAL_VOCODER is None)

for exp_name, model in TRAINED_MODELS.items():
    exp_out_off = EVAL_LOCAL_DIR / exp_name / "offline"
    exp_out_str = EVAL_LOCAL_DIR / exp_name / "streaming"
    exp_out_pre = EVAL_LOCAL_DIR / exp_name / "pretrained_vocos"
    exp_out_off.mkdir(parents=True, exist_ok=True)
    if model.is_streaming and CAUSAL_VOCODER is not None:
        exp_out_str.mkdir(parents=True, exist_ok=True)
    if PRETRAINED_VOCOS is not None:
        exp_out_pre.mkdir(parents=True, exist_ok=True)

    recs = []
    for item in tqdm(EVAL_SUITE, desc=f"Synthesize {exp_name}"):
        wav_off, _ = synthesize_offline(
            model, item["text"],
            vocoder=CAUSAL_VOCODER if not use_pre_for_offline else PRETRAINED_VOCOS,
            use_pretrained_vocos=use_pre_for_offline,
        )
        p_off = exp_out_off / f"{item['utt_id']}.wav"
        sf.write(p_off, np.clip(wav_off, -1.0, 1.0), CFG["sample_rate"])

        p_pre = None
        if PRETRAINED_VOCOS is not None and not use_pre_for_offline:
            wav_pre, _ = synthesize_offline(model, item["text"], vocoder=None, use_pretrained_vocos=True)
            p_pre = exp_out_pre / f"{item['utt_id']}.wav"
            sf.write(p_pre, np.clip(wav_pre, -1.0, 1.0), CFG["sample_rate"])

        p_str, seams, stream_diff = None, [], 0.0
        if model.is_streaming and CAUSAL_VOCODER is not None:
            wav_str, _, seams = synthesize_streaming(model, item["text"], CAUSAL_VOCODER)
            p_str = exp_out_str / f"{item['utt_id']}.wav"
            sf.write(p_str, np.clip(wav_str, -1.0, 1.0), CFG["sample_rate"])
            L_min = min(len(wav_off), len(wav_str))
            stream_diff = float(np.max(np.abs(wav_off[:L_min] - wav_str[:L_min]))) if L_min > 0 else 0.0

        recs.append({
            "utt_id": item["utt_id"], "text": item["text"], "is_val": item["is_val"],
            "gt_wav": str(gt_dir / f"{item['utt_id']}.wav") if item["is_val"] else None,
            "off_wav": str(p_off), "str_wav": str(p_str) if p_str else None,
            "pre_wav": str(p_pre) if p_pre else None,
            "seams": seams, "stream_wav_max_diff": stream_diff,
        })
    GENERATED_RECORDS[exp_name] = recs
    # 每合成完一个模型立即同步，避免中途断连前功尽弃
    sync_dir_to_storage(EVAL_LOCAL_DIR / exp_name, EVAL_STORAGE_DIR / exp_name)

save_progress_status("synthesis", {"status": "done", "models": list(GENERATED_RECORDS.keys()), "utts_per_model": len(EVAL_SUITE)})
print(f"[Saved] 全部生成音频已归档至: {EVAL_STORAGE_DIR}")


## 11. 多维自动化质量打分 & 嵌入式纯 CPU（2 线程）RTF / 首包延迟基准


In [ ]:
import pandas as pd
from jiwer import wer as calc_wer

UTMOS_MODEL = None
try:
    print("[Eval] 加载 UTMOS (utmos22_strong)...")
    UTMOS_MODEL = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(DEVICE).eval()
except Exception as e:
    print(f"[Warn] UTMOS unavailable: {e}")

SQUIM_MODEL = None
try:
    print("[Eval] 加载 TorchAudio SQUIM_OBJECTIVE...")
    SQUIM_MODEL = torchaudio.pipelines.SQUIM_OBJECTIVE.get_model().to(DEVICE).eval()
except Exception as e:
    print(f"[Warn] SQUIM unavailable: {e}")

SPK_ENCODER = None
try:
    from resemblyzer import VoiceEncoder, preprocess_wav
    print("[Eval] 加载 Resemblyzer 声纹编码器...")
    SPK_ENCODER = VoiceEncoder(device=str(DEVICE))
except Exception as e:
    print(f"[Warn] Resemblyzer unavailable: {e}")

WHISPER_MODEL = None
WHISPER_MODEL_NAME = "small" if CFG["profile"] == "quick_smoke" else "large-v3-turbo"
WHISPER_ON_CUDA = False
try:
    from faster_whisper import WhisperModel
    if torch.cuda.is_available():
        try:
            print(f"[Eval] 加载 Faster-Whisper ({WHISPER_MODEL_NAME}, CUDA float16)...")
            _wm = WhisperModel(WHISPER_MODEL_NAME, device="cuda", compute_type="float16")
            # 实测 0.2s 静音前向，提前触发 ctranslate2 的 libcublas.so.12 / cudnn 延迟加载校验
            list(_wm.transcribe(np.zeros(3200, dtype=np.float32), language="zh", beam_size=1)[0])
            WHISPER_MODEL = _wm
            WHISPER_ON_CUDA = True
        except Exception as e_cuda:
            print(f"[Warn] Faster-Whisper CUDA 不可用 ({e_cuda})，自动回退 CPU int8（避免 MER 全 NaN）...")
            WHISPER_MODEL = WhisperModel(WHISPER_MODEL_NAME, device="cpu", compute_type="int8")
            WHISPER_ON_CUDA = False
    else:
        print(f"[Eval] 加载 Faster-Whisper ({WHISPER_MODEL_NAME}, CPU int8)...")
        WHISPER_MODEL = WhisperModel(WHISPER_MODEL_NAME, device="cpu", compute_type="int8")
except Exception as e:
    print(f"[Warn] Faster-Whisper unavailable: {e}")


def compute_mixed_error_rate(ref: str, hyp: str) -> float:
    def tokenize(s):
        s = unicodedata.normalize("NFKC", s).lower()
        return re.findall(r"[\u4e00-\u9fff]|[a-z0-9]+", s)
    r_toks, h_toks = tokenize(ref), tokenize(hyp)
    return 0.0 if not r_toks else float(calc_wer(" ".join(r_toks), " ".join(h_toks)))


@torch.no_grad()
def score_utmos(wav_np: np.ndarray, sr: int = 24000) -> float:
    if UTMOS_MODEL is None:
        return float("nan")
    w16 = librosa.resample(wav_np, orig_sr=sr, target_sr=16000)
    return float(UTMOS_MODEL(torch.from_numpy(w16).unsqueeze(0).to(DEVICE), 16000).item())


@torch.no_grad()
def score_squim(wav_np: np.ndarray, sr: int = 24000):
    if SQUIM_MODEL is None:
        return float("nan"), float("nan"), float("nan")
    w16 = librosa.resample(wav_np, orig_sr=sr, target_sr=16000)
    stoi, pesq, sisdr = SQUIM_MODEL(torch.from_numpy(w16).unsqueeze(0).to(DEVICE))
    return float(pesq.item()), float(sisdr.item()), float(stoi.item())


def hf_2_8k_ratio(wav_np: np.ndarray, sr: int = 24000) -> float:
    spec = np.abs(np.fft.rfft(wav_np)) ** 2
    freqs = np.fft.rfftfreq(len(wav_np), 1.0 / sr)
    band = (freqs >= 2000.0) & (freqs <= 8000.0)
    return float(spec[band].sum() / (spec.sum() + 1e-8))


def detect_seam_click(wav_np: np.ndarray, seams: list, sr: int = 24000) -> float:
    if not seams:
        return 0.0
    diff = np.abs(np.diff(wav_np))
    win = int(0.005 * sr)
    global_rms = float(np.sqrt(np.mean(diff ** 2) + 1e-10))
    clicks = 0
    for pos in seams:
        lo, hi = max(0, pos - win), min(len(diff), pos + win)
        if hi <= lo:
            continue
        local_max = float(np.max(diff[lo:hi]))
        ref_lo, ref_hi = max(0, pos - 4 * win), min(len(diff), pos + 4 * win)
        local_rms = float(np.sqrt(np.mean(diff[ref_lo:ref_hi] ** 2) + 1e-10))
        if local_max > max(12.0 * local_rms, 15.0 * global_rms):
            clicks += 1
    return float(clicks) / len(seams)


def compute_asr_error(wav_or_path, ref_text: str, sr: int = 24000) -> float:
    global WHISPER_MODEL, WHISPER_ON_CUDA
    if WHISPER_MODEL is None:
        return float("nan")
    try:
        if isinstance(wav_or_path, (str, Path)):
            wav_np, sr = sf.read(str(wav_or_path), dtype="float32")
        else:
            wav_np = np.asarray(wav_or_path, dtype=np.float32)
        if wav_np.ndim > 1:
            wav_np = wav_np.mean(axis=-1)
        if len(wav_np) == 0:
            return 1.0
        # 直接传入 16 kHz float32 ndarray，彻底绕开 faster-whisper 内部 PyAV av.open(..., metadata_errors="ignore") 在新版 av/Python 3.13 上的 TypeError
        w16 = librosa.resample(wav_np, orig_sr=sr, target_sr=16000) if sr != 16000 else wav_np
        w16 = np.ascontiguousarray(w16, dtype=np.float32)
        lang = "zh" if re.search(r"[\u4e00-\u9fff]", ref_text) else "en"
        try:
            segs, _ = WHISPER_MODEL.transcribe(w16, language=lang, beam_size=1)
            hyp = "".join(s.text for s in segs)
        except Exception as e_run:
            if WHISPER_ON_CUDA:
                print(f"[Warn] Faster-Whisper CUDA 推理失败 ({e_run})，切换 CPU int8 重试...")
                from faster_whisper import WhisperModel
                WHISPER_MODEL = WhisperModel(WHISPER_MODEL_NAME, device="cpu", compute_type="int8")
                WHISPER_ON_CUDA = False
                segs, _ = WHISPER_MODEL.transcribe(w16, language=lang, beam_size=1)
                hyp = "".join(s.text for s in segs)
            else:
                raise
        return compute_mixed_error_rate(ref_text, hyp)
    except Exception as e:
        print(f"[Warn] ASR scoring failed ({e}), returning NaN.")
        return float("nan")


def _safe_nanmean(vals) -> float:
    arr = np.asarray(vals, dtype=np.float64)
    valid = arr[~np.isnan(arr)]
    return float(valid.mean()) if valid.size > 0 else float("nan")


GT_REF_EMBS, GT_HF_RATIOS, GT_UTMOS_LIST = [], [], []
for item in EVAL_SUITE:
    if item["is_val"]:
        p = EVAL_LOCAL_DIR / "00_ground_truth" / f"{item['utt_id']}.wav"
        w, _ = sf.read(str(p), dtype="float32")
        GT_HF_RATIOS.append(hf_2_8k_ratio(w))
        GT_UTMOS_LIST.append(score_utmos(w))
        if SPK_ENCODER is not None:
            GT_REF_EMBS.append(SPK_ENCODER.embed_utterance(preprocess_wav(w, source_sr=CFG["sample_rate"])))

GT_SPK_CENTROID = np.mean(GT_REF_EMBS, axis=0) if GT_REF_EMBS else None
if GT_SPK_CENTROID is not None:
    GT_SPK_CENTROID /= (np.linalg.norm(GT_SPK_CENTROID) + 1e-8)
GT_HF_MEAN = float(np.mean(GT_HF_RATIOS)) if GT_HF_RATIOS else 0.15
GT_UTMOS_MEAN = _safe_nanmean(GT_UTMOS_LIST)
print(f"[Gold Anchor] Ground-Truth UTMOS: {GT_UTMOS_MEAN:.3f} | 2-8kHz Ratio: {GT_HF_MEAN:.4f}")


@torch.no_grad()
def benchmark_cpu_2thread(model: AetherAcousticModel, vocoder: CausalVocosLite, sample_text: str):
    prev_threads = torch.get_num_threads()
    torch.set_num_threads(2)
    m_cpu = model.to("cpu").eval()
    v_cpu = vocoder.to("cpu").eval() if vocoder is not None else None
    pids, lids = FRONTEND.text_to_ids(sample_text)
    pt, lt = torch.tensor([pids], dtype=torch.long), torch.tensor([lids], dtype=torch.long)
    T = pt.shape[1]

    def _run_once():
        if m_cpu.is_streaming and v_cpu is not None:
            t_start, ttfa_ms, total_samples = time.perf_counter(), None, 0
            ast_st = m_cpu.init_stream_state(1, device=torch.device("cpu"), dtype=torch.float32)
            voc_st = v_cpu.init_state(1, device=torch.device("cpu"), dtype=torch.float32)
            mel_buf = torch.zeros(1, CFG["n_mels"], 0)
            t_step, m_step = CFG["stream_text_chunk"], CFG["stream_mel_chunk"]
            for i in range(0, T, t_step):
                is_last = (i + t_step >= T)
                mc, ast_st = m_cpu.infer_stream_step(pt[:, i:i+t_step], lt[:, i:i+t_step], ast_st, is_final=is_last)
                if mc.shape[-1] > 0:
                    mel_buf = torch.cat([mel_buf, denorm_mel(mc)], dim=-1)
                while mel_buf.shape[-1] >= m_step:
                    wc, voc_st = v_cpu.forward_step(mel_buf[:, :, :m_step], voc_st, is_final=False)
                    mel_buf = mel_buf[:, :, m_step:]
                    total_samples += wc.shape[-1]
                    if ttfa_ms is None and wc.shape[-1] > 0:
                        ttfa_ms = (time.perf_counter() - t_start) * 1000.0
                if is_last:
                    wc, voc_st = v_cpu.forward_step(mel_buf, voc_st, is_final=True)
                    total_samples += wc.shape[-1]
                    if ttfa_ms is None and wc.shape[-1] > 0:
                        ttfa_ms = (time.perf_counter() - t_start) * 1000.0
            elapsed = time.perf_counter() - t_start
            return elapsed / max(total_samples / CFG["sample_rate"], 1e-3), (ttfa_ms or 0.0)
        t_start = time.perf_counter()
        mel = denorm_mel(m_cpu.infer_offline(pt, lt)[0])
        audio_sec = max((v_cpu(mel).shape[-1] if v_cpu is not None else mel.shape[-1] * CFG["hop_length"]) / CFG["sample_rate"], 1e-3)
        elapsed = time.perf_counter() - t_start
        return elapsed / audio_sec, elapsed * 1000.0

    # 先预热 1 遍真实推理路径，再跑 3 遍取中位数，消除云端 vCPU 瞬时抖动
    _run_once()
    runs = [_run_once() for _ in range(3)]
    rtf = float(np.median([x[0] for x in runs]))
    ttfa_ms = float(np.median([x[1] for x in runs]))

    model.to(DEVICE)
    if vocoder is not None:
        vocoder.to(DEVICE)
    torch.set_num_threads(prev_threads)
    return round(rtf, 4), round(ttfa_ms, 1)


all_sample_rows, summary_rows = [], []
bench_text = GOLDEN_ZH[-2][1] if CFG["dataset_source"] != "ljspeech" else GOLDEN_EN[-1][1]

for exp_name, model in TRAINED_MODELS.items():
    recs = GENERATED_RECORDS[exp_name]
    utmos_s, pesq_s, sisdr_s, stoi_s, mer_s, sim_s, hf_diff_s, click_s, sdiff_s, utmos_pre_s = [], [], [], [], [], [], [], [], [], []
    for r in tqdm(recs, desc=f"Scoring {exp_name}"):
        eval_wav_path = r["str_wav"] if r["str_wav"] else r["off_wav"]
        wav, wav_sr = sf.read(eval_wav_path, dtype="float32")
        u = score_utmos(wav, sr=wav_sr)
        pesq, sisdr, stoi = score_squim(wav, sr=wav_sr)
        mer = compute_asr_error(wav, r["text"], sr=wav_sr)
        hfd = abs(hf_2_8k_ratio(wav, sr=wav_sr) - GT_HF_MEAN)
        clk = detect_seam_click(wav, r["seams"], sr=wav_sr) if r["str_wav"] else 0.0
        sim = float("nan")
        if SPK_ENCODER is not None and GT_SPK_CENTROID is not None:
            emb = SPK_ENCODER.embed_utterance(preprocess_wav(wav, source_sr=wav_sr))
            sim = float(np.dot(emb / (np.linalg.norm(emb) + 1e-8), GT_SPK_CENTROID))
        u_pre = float("nan")
        if r["pre_wav"]:
            w_pre, w_pre_sr = sf.read(r["pre_wav"], dtype="float32")
            u_pre = score_utmos(w_pre, sr=w_pre_sr)
            utmos_pre_s.append(u_pre)
        utmos_s.append(u); pesq_s.append(pesq); sisdr_s.append(sisdr); stoi_s.append(stoi)
        mer_s.append(mer); sim_s.append(sim); hf_diff_s.append(hfd); click_s.append(clk); sdiff_s.append(r["stream_wav_max_diff"])
        all_sample_rows.append({"model": exp_name, "utt_id": r["utt_id"], "text": r["text"], "UTMOS": u, "UTMOS_PreVocos": u_pre, "PESQ": pesq, "SI_SDR": sisdr, "MER": mer, "SIM": sim, "HF_2_8k_Diff": hfd, "Seam_Click": clk})

    rtf_2t, ttfa_ms = benchmark_cpu_2thread(model, CAUSAL_VOCODER, bench_text)
    infer_params_m = sum(p.numel() for n, p in model.named_parameters() if not n.startswith("aligner.")) / 1e6
    int8_path = STORAGE_ROOT / "runs" / f"{exp_name}_{CFG['acoustic_suffix']}" / "model_int8.pt"
    size_mb = round(int8_path.stat().st_size / 1e6, 2) if int8_path.exists() else round(infer_params_m * 4.0, 2)

    mean_utmos = _safe_nanmean(utmos_s)
    mean_utmos_pre = _safe_nanmean(utmos_pre_s)
    mean_mer, mean_sim, mean_pesq, mean_click = _safe_nanmean(mer_s), _safe_nanmean(sim_s), _safe_nanmean(pesq_s), float(np.mean(click_s))

    s_nat = np.clip((mean_utmos - 1.5) / 3.0, 0.0, 1.0) * 35.0 if not np.isnan(mean_utmos) else 20.0
    s_int = np.clip(1.0 - mean_mer * 3.0, 0.0, 1.0) * 30.0 if not np.isnan(mean_mer) else 20.0
    s_sim = np.clip((mean_sim - 0.5) / 0.45, 0.0, 1.0) * 15.0 if not np.isnan(mean_sim) else 10.0
    s_lat = np.clip(1.0 - ttfa_ms / 500.0, 0.0, 1.0) * 10.0 + np.clip(1.0 - rtf_2t / 0.5, 0.0, 1.0) * 10.0
    composite = round(float(s_nat + s_int + s_sim + s_lat - 20.0 * mean_click), 2)

    summary_rows.append({
        "Model": exp_name, "Streaming": model.is_streaming, "Lookahead": model.lookahead,
        "Params_M": round(infer_params_m, 2), "Size_MB": size_mb,
        "UTMOS↑": round(mean_utmos, 3), "UTMOS_PreVocos↑": round(mean_utmos_pre, 3) if not np.isnan(mean_utmos_pre) else None,
        "MER/CER↓": round(mean_mer, 4), "SIM↑": round(mean_sim, 3), "SQUIM_PESQ↑": round(mean_pesq, 3),
        "HF_2_8k_Diff↓": round(float(np.mean(hf_diff_s)), 4), "Seam_Click↓": round(mean_click, 4),
        "Stream_MaxDiff": f"{max(sdiff_s):.1e}", "CPU2T_RTF↓": rtf_2t, "CPU2T_TTFA_ms↓": ttfa_ms, "Composite_Score↑": composite,
    })

LEADERBOARD = pd.DataFrame(summary_rows).sort_values("Composite_Score↑", ascending=False).reset_index(drop=True)
PER_SAMPLE_DF = pd.DataFrame(all_sample_rows)

report_local = LOCAL_WORK / "reports" / f"{CFG['eval_suffix']}"
report_storage = STORAGE_ROOT / "reports" / f"{CFG['eval_suffix']}"
report_local.mkdir(parents=True, exist_ok=True)
report_storage.mkdir(parents=True, exist_ok=True)

LEADERBOARD.to_csv(report_local / "leaderboard.csv", index=False)
LEADERBOARD.to_json(report_local / "leaderboard.json", orient="records", force_ascii=False, indent=2)
PER_SAMPLE_DF.to_csv(report_local / "per_sample_scores.csv", index=False)
sync_dir_to_storage(report_local, report_storage)

print("\n================ AetherTTS Phase 1 综合评测排行榜 ================")
_in_nb = False
try:
    from IPython import get_ipython
    from IPython.display import display
    if get_ipython() is not None:
        display(LEADERBOARD)
        _in_nb = True
except Exception:
    pass
if not _in_nb:
    print(LEADERBOARD.to_string(index=False))
save_progress_status("evaluation", {"status": "done", "top_model": str(LEADERBOARD.iloc[0]["Model"]) if len(LEADERBOARD) else None})
print(f"\n[Saved] 评测报告已保存至: {report_storage}")


## 12. 可视化对比图、盲听打分表 & 可选 Colab 自动释放


In [ ]:
import matplotlib
if not IS_COLAB and os.environ.get("DISPLAY", "") == "":
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
for exp_name, hist in TRAIN_HISTORIES.items():
    if hist.get("steps"):
        ax.plot(hist["steps"], hist["val_mel"], marker="o", label=exp_name)
ax.set_title("Validation Mel L1 Convergence (Lower is Better)")
ax.set_xlabel("Steps"); ax.set_ylabel("Val Mel L1"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

ax = axes[1]
x = np.arange(len(LEADERBOARD))
ax.bar(x - 0.18, LEADERBOARD["UTMOS↑"].fillna(0), width=0.36, label="UTMOS (CausalVocosLite)")
if LEADERBOARD["UTMOS_PreVocos↑"].notna().any():
    ax.bar(x + 0.18, LEADERBOARD["UTMOS_PreVocos↑"].fillna(0), width=0.36, label="UTMOS (Pretrained Vocos)")
if not np.isnan(GT_UTMOS_MEAN):
    ax.axhline(GT_UTMOS_MEAN, color="red", linestyle="--", label=f"Gold GT ({GT_UTMOS_MEAN:.2f})")
ax.set_xticks(x); ax.set_xticklabels(LEADERBOARD["Model"], rotation=25, ha="right", fontsize=8)
ax.set_title("Naturalness (UTMOS)"); ax.set_ylim(0, 5.0); ax.grid(True, axis="y", alpha=0.3); ax.legend(fontsize=8)

ax = axes[2]
ax.bar(x, LEADERBOARD["CPU2T_TTFA_ms↓"], color="#4C78A8", alpha=0.85, label="TTFA (ms)")
ax.axhline(150.0, color="green", linestyle="--", label="Target TTFA (150ms)")
ax.set_xticks(x); ax.set_xticklabels(LEADERBOARD["Model"], rotation=25, ha="right", fontsize=8)
ax.set_title("CPU 2-Thread First-Audio Latency (ms)"); ax.grid(True, axis="y", alpha=0.3); ax.legend(fontsize=8)

plt.tight_layout()
chart_local = report_local / "comparison_charts.png"
plt.savefig(chart_local, dpi=150)
sync_file_to_storage(chart_local, report_storage / "comparison_charts.png")
if IS_COLAB or os.environ.get("DISPLAY", ""):
    plt.show()
else:
    plt.close(fig)

# 盲听打分表（在 Notebook 界面下渲染试听播放器；自建无界面服务器直接导出 CSV）
blind_rows = []
rng_b = random.Random(CFG["seed"] + 99)
model_names = list(TRAINED_MODELS.keys())
preview_utts = [u for u in EVAL_SUITE if not u["is_val"]][:5] or EVAL_SUITE[:5]

interactive_nb = False
try:
    from IPython import get_ipython
    from IPython.display import Audio, display, HTML
    interactive_nb = get_ipython() is not None
except Exception:
    pass

for u in preview_utts:
    if interactive_nb:
        display(HTML(f"<h4>[{u['utt_id']}] {u['text']}</h4>"))
    shuffled = list(model_names)
    rng_b.shuffle(shuffled)
    for label_idx, m_name in enumerate(shuffled):
        rec = next(r for r in GENERATED_RECORDS[m_name] if r["utt_id"] == u["utt_id"])
        wav_p = rec["str_wav"] if rec["str_wav"] else rec["off_wav"]
        tag = f"Candidate_{chr(65 + label_idx)}"
        blind_rows.append({"utt_id": u["utt_id"], "text": u["text"], "blind_tag": tag, "true_model": m_name, "wav_path": wav_p, "CMOS_score": "", "defect_tags": ""})
        if interactive_nb:
            print(f"  {tag}:")
            display(Audio(filename=wav_p, rate=CFG["sample_rate"]))

blind_df = pd.DataFrame(blind_rows)
blind_local = report_local / "blind_ab_listening_sheet.csv"
blind_df.to_csv(blind_local, index=False)
sync_file_to_storage(blind_local, report_storage / "blind_ab_listening_sheet.csv")
save_progress_status("all_complete", {"status": "done", "report_dir": str(report_storage)})
print(f"[Saved] 图表与盲听打分表已保存至: {report_storage}")

if IS_COLAB and AUTO_UNASSIGN_COLAB:
    print("[Colab] 全部任务已完成并同步至 Google Drive，10 秒后自动释放 GPU 运行时...")
    time.sleep(10)
    from google.colab import runtime
    runtime.unassign()


## 13. （可选扩展）Apache 2.0 教师离线造数模板（Phase 2 单音色中英混读黄金集）


In [ ]:
RUN_TEACHER_SYNTH = False  # Phase 2 造数时改为 True

if RUN_TEACHER_SYNTH:
    OUT_CUSTOM_DIR = STORAGE_ROOT / "datasets" / "custom_bilingual"
    (OUT_CUSTOM_DIR / "wavs").mkdir(parents=True, exist_ok=True)
    print(f"教师造数输出目录已就绪: {OUT_CUSTOM_DIR}")
    print("参考 docs/03_teacher_distillation_pipeline.md：放入合规干声 ref.wav -> 用 Qwen3-TTS / CosyVoice 批量合成 20-25h -> ASR+SIM+UTMOS 四重清洗 -> 设置 DATASET_SOURCE='custom_bilingual' 开训。")
else:
    print("当前跳过教师离线造数。")
